# Vincent Sampling ABC — v16 fast-auto hybrid Vincent + global exploration

This version fixes v14 initialization so global CaDsd attempts cannot run silently for a very long time.


## Pythonizing Vincent!

In [45]:
import numpy as np
from typing import Union, Optional


# ------------------------------------------------------------
# 1) Ppi: build the orthogonal matrix V from first-order Pis
# ------------------------------------------------------------

def Ppi(Pi: Union[np.ndarray, list]) -> np.ndarray:
    """
    Python/Numpy port of the R function Ppi(Pi).

    Parameters
    ----------
    Pi : array-like of shape (N,)
        First-order inclusion probabilities, 0 < Pi_i < 1,
        sum(Pi) must be (approximately) an integer.

    Returns
    -------
    V : ndarray of shape (N, n)
        Orthogonal matrix associated to the DSD construction.
    """
    Pi = np.asarray(Pi, dtype=float).ravel()
    N = Pi.size

    # --- Error checks ---
    if N < 2:
        raise ValueError(
            "The sampling designs should be defined on a set of more than "
            "one element. (length(Pi) > 1)"
        )

    if np.any(Pi <= 0) or np.any(Pi >= 1):
        raise ValueError("Pi is not a vector of probabilities (0 < p < 1).")

    sum_pi_rounded = round(Pi.sum(), 9)
    n_int = int(sum_pi_rounded)
    if int(round(sum_pi_rounded, 9)) - sum_pi_rounded != 0:
        raise ValueError(
            "The sum of the first order inclusion probabilities "
            "should be an integer (up to rounding)."
        )

    # --- Main algorithm ---
    s_vals = np.zeros(N, dtype=float)
    c_vals = np.zeros(N, dtype=float)
    alpha = np.zeros(N, dtype=float)

    # kr will store the indices (0-based) where cumulative sum crosses integers
    if n_int <= 0:
        raise ValueError("Sum of Pi must be at least 1.")
    kr = [None] * n_int

    cum_sum = 0.0
    r = 1          # current integer threshold
    r_prev = 0     # last integer that was crossed

    for k in range(N):
        prev_sum = cum_sum
        cum_sum += Pi[k]

        if cum_sum >= r:  # crossed integer r
            if r <= n_int:
                alpha[k] = r - prev_sum
                kr[r - 1] = k   # store 0-based index
                val = np.sqrt((1.0 - Pi[k]) / (1.0 - alpha[k]))
                s_vals[k] = np.round(val, 8)
                r_prev = r
                r += 1
        else:
            denom = (r_prev + 1 - prev_sum)
            val = np.sqrt(Pi[k] / denom)
            s_vals[k] = np.round(val, 15)

        c_vals[k] = np.sqrt(1.0 - s_vals[k] ** 2)

    # Patch: ensure last crossing index corresponds to last unit (like R hack)
    # If some entries of kr are still None, set the last one to N-1.
    if any(x is None for x in kr):
        kr[-1] = N - 1
    # For safety, also replace any remaining None by the last index
    last_index = kr[-1]
    kr = [last_index if x is None else x for x in kr]

    # Use sample size n_int as number of columns (simplified vs. R hack)
    r_prev = n_int

    # --- Build V ---
    V = np.zeros((N, r_prev), dtype=float)
    V[0, 0] = 1.0

    # In R: V[kr[r] + 1, r + 1] = 1 for r in 1:(r_prev-1)
    # Here r_idx corresponds to r-1 in R, and we use 0-based indices.
    if r_prev - 1 != 0:
        for r_idx in range(1, r_prev):
            kpos = kr[r_idx - 1]  # 0-based index for row
            V[kpos + 1, r_idx] = 1.0

    # Apply Givens-like rotations
    for k in range(N - 1):
        L = V[k, :].copy()
        M = V[k + 1, :].copy()
        V[k, :] = s_vals[k] * L - c_vals[k] * M
        V[k + 1, :] = c_vals[k] * L + s_vals[k] * M

    return V


# ------------------------------------------------------------
# 2) DSD sampling (real and complex versions)
# ------------------------------------------------------------

def Drawing_Dsd(
    v: Union[np.ndarray, list],
    s: int = 1,
    B: bool = False,
    seed: Optional[int] = None,
):
    """
    Python/Numpy port of the R function Drawing_Dsd(v, s=1, B=FALSE, seed=NULL).

    Parameters
    ----------
    v : array-like, shape (N, n)
        Matrix of (real or complex) vectors used in DSD.
    s : int, default 1
        Number of samples (replicates).
    B : bool, default False
        If True: return 0/1 indicator vector(s).
        If False: return indices of selected units (1-based, to match R).
    seed : int or None
        Seed for reproducibility.

    Returns
    -------
    If s == 1:
        1D array of length N (if B=True) or selected indices (if B=False).
    If s > 1:
        2D array of shape (N, s) (if B=True) or (n, s) with indices per sample.
    """
    v = np.asarray(v)
    rng = np.random.default_rng(seed)

    if np.iscomplexobj(v):
        return _dsd_sampling_mult_complex(v, s, B, rng)
    else:
        return _dsd_sampling_mult(v, s, B, rng)


# ---------------------- helpers: real case ---------------------- #

def _dsd_sampling_mult(
    v: np.ndarray,
    s: int,
    B: bool,
    rng: np.random.Generator,
):
    v = np.asarray(v, dtype=float)
    if v.ndim == 1:
        v = v[:, None]  # treat as (N,1)

    if s == 1:
        return _dsd_sampling_01_B_C(v, B, rng)
    else:
        samples = [_dsd_sampling_01_B_C(v, B, rng) for _ in range(s)]
        # stack as columns like R's replicate (N x s)
        return np.column_stack(samples)


def _dsd_sampling_01_B_C(
    v: np.ndarray,
    B: bool = True,
    rng: Optional[np.random.Generator] = None,
):
    """
    Real-valued version of .dsd_sampling_01_B_C in R.
    """
    v = np.asarray(v, dtype=float)
    if v.ndim == 1:
        v = v[:, None]

    N, n = v.shape
    echant = np.zeros(N, dtype=int)

    if rng is None:
        rng = np.random.default_rng()
    ref = rng.random(n)

    # Step 1: first element
    w = v.copy()

    pi1 = np.einsum("ij,ij->i", v, v)  # diag(v %*% t(v))
    total = 0.0
    i = -1

    while total < ref[0]:
        i += 1
        if i >= N:
            raise RuntimeError("Sampling failed in first step (real case).")
        total += pi1[i] / n
    echant[i] = 1

    l = v[i, :]
    norm_l = np.sqrt(np.dot(l, l))
    if norm_l == 0:
        raise ValueError("Encountered zero-norm vector in real DSD.")
    e1 = l / norm_l

    # Step 2: remaining n-1 elements
    for j in range(n - 1):
        r = n - (j + 1)
        inter = v @ e1  # (N,)
        pi1 = pi1 - inter * inter
        pi2 = pi1 / r

        total = 0.0
        i = -1
        while total < ref[j + 1]:
            i += 1
            if i >= N:
                raise RuntimeError("Sampling failed in step 2 (real case).")
            total += pi2[i]
        echant[i] = 1

        # Update w and e1 (Gram-Schmidt-like update)
        proj = w @ e1                      # shape (N,)
        w = w - np.outer(proj, e1)         # (N,n)
        L = w[i, :]
        norm_L = np.sqrt(np.dot(L, L))
        if norm_L == 0:
            raise ValueError("Encountered zero-norm vector in real DSD.")
        e1 = L / norm_L

    if B:
        # 0/1 vector -> same as R "echant"
        return echant
    else:
        # indices (1-based, like in R: (1:N)[echant==1])
        return np.nonzero(echant == 1)[0] + 1


# -------------------- helpers: complex case --------------------- #

def _dsd_sampling_mult_complex(
    v: np.ndarray,
    s: int,
    B: bool,
    rng: np.random.Generator,
):
    v = np.asarray(v, dtype=np.complex128)
    if v.ndim == 1:
        v = v[:, None]

    if s == 1:
        return _dsd_sampling_01_B_C_complex(v, B, rng)
    else:
        samples = [_dsd_sampling_01_B_C_complex(v, B, rng) for _ in range(s)]
        return np.column_stack(samples)


def _dsd_sampling_01_B_C_complex(
    v: np.ndarray,
    B: bool = True,
    rng: Optional[np.random.Generator] = None,
):
    """
    Complex-valued version of .dsd_sampling_01_B_C_complex in R.
    """
    v = np.asarray(v, dtype=np.complex128)
    if v.ndim == 1:
        v = v[:, None]

    N, n = v.shape
    echant = np.zeros(N, dtype=int)

    if rng is None:
        rng = np.random.default_rng()
    ref = rng.random(n)

    w = v.copy()

    # pi1 = Re( diag( v %*% t(Conj(v)) ) )
    pi1 = np.real(np.einsum("ij,ij->i", v, np.conjugate(v)))

    if np.any(pi1 < 0) or np.any(pi1 >= 1):
        raise ValueError(
            "The matrix v given as input doesn't suit the expected input "
            "(cf. pgd / periodic_dsd)."
        )

    # Step 1: first element
    total = 0.0
    i = -1

    while total < ref[0]:
        i += 1
        if i >= N:
            raise RuntimeError("Sampling failed in first step (complex case).")
        total += pi1[i] / n
    echant[i] = 1

    M = v[i, :]
    norm_M = np.sqrt(np.real(np.vdot(M, M)))
    if norm_M == 0:
        raise ValueError("Encountered zero-norm vector in complex DSD.")
    e1 = M / norm_M

    # Step 2: remaining n-1 elements
    for j in range(n - 1):
        r = n - (j + 1)

        # inter <- v %*% Conj(e1)
        inter = v @ np.conjugate(e1)           # (N,)
        pi1 = pi1 - np.real(inter * np.conjugate(inter))
        pi2 = np.real(pi1 / r)

        total = 0.0
        i = -1
        while total < ref[j + 1]:
            i += 1
            if i >= N:
                raise RuntimeError("Sampling failed in step 2 (complex case).")
            total += pi2[i]
        echant[i] = 1

        # w <- w - (projection on e1)
        proj = w @ np.conjugate(e1)           # (N,)
        w = w - np.outer(proj, e1)            # (N, n)

        L = w[i, :]
        norm_L = np.sqrt(np.real(np.vdot(L, L)))
        if norm_L == 0:
            raise ValueError("Encountered zero-norm vector in complex DSD.")
        e1 = L / norm_L

    if B:
        return echant
    else:
        # Return 1-based indices, to match the R function
        return np.nonzero(echant == 1)[0] + 1

In [46]:
import numpy as np
from typing import Optional, Union


def spec(omega: np.ndarray, M: int, pi: Union[np.ndarray, list], spectre=100) -> np.ndarray:
    """
    Faster Python/Numpy port of the R function spec().

    The output is the same object as before: a matrix of shape (M, N).
    Speed-up comes from cumulative sums instead of thousands of repeated
    small numpy sum calls inside Python loops.
    """
    pi = np.asarray(pi, dtype=float).ravel()
    N = pi.size
    omega = np.asarray(omega, dtype=float)
    if omega.shape != (M, N):
        raise ValueError("omega must have shape (M, N)")

    mat_spectre = np.zeros((M, N), dtype=float)
    pi_down = np.sort(pi)[::-1]
    pi_prefix = np.concatenate(([0.0], np.cumsum(pi_down)))
    mu = float(pi.sum())

    if (np.isscalar(spectre) and spectre == 100) or (
        not np.isscalar(spectre)
        and len(np.atleast_1d(spectre)) == 1
        and np.atleast_1d(spectre)[0] == 100
    ):
        spectre_vec = np.zeros(M, dtype=float)
        cumsum_1 = 0.0
        lam = 0.0

        for j in range(M - 1):
            jR = j + 1
            A = max(lam, mu - cumsum_1 - (M - jR))
            B = 1.0

            for iR in range(1, M - jR + 1):
                t_idx = M - jR - iR + 1
                s_down = pi_prefix[t_idx]
                val = (mu - cumsum_1 - s_down) / iR
                if val < B:
                    B = val

            val = (mu - cumsum_1) / (M - jR + 1)
            if val < B:
                B = val

            spectre_vec[j] = A + omega[j, N - 1] * (B - A)
            lam = spectre_vec[j]
            cumsum_1 += spectre_vec[j]

        spectre_vec[M - 1] = mu - spectre_vec[: M - 1].sum()
    else:
        spectre_arr = np.asarray(spectre, dtype=float).ravel()
        if spectre_arr.size != M:
            raise ValueError("spectre must have length M")
        spectre_vec = spectre_arr

    mat_spectre[:, N - 1] = spectre_vec

    for kR in range(N - 1, 0, -1):
        startR = max(1, M - kR + 1)
        lambda1 = mat_spectre[:, kR].copy()
        lambda2 = mat_spectre[:, kR - 1].copy()
        prefix_l1 = np.concatenate(([0.0], np.cumsum(lambda1)))
        cum_l2_before_j = 0.0

        # rows before startR are zero by construction, so cum_l2_before_j starts at 0.
        for jR in range(startR, M + 1):
            lam_prev = lambda1[jR - 2] if jR >= 2 else 0.0
            sum_l1 = prefix_l1[jR]
            A = max(0.0, lam_prev, sum_l1 - cum_l2_before_j - pi_down[kR])

            B_inner = float("inf")
            for iR in range(jR, M + 1):
                start_idx0 = M - iR
                if start_idx0 < kR:
                    prem = pi_prefix[kR] - pi_prefix[start_idx0]
                else:
                    prem = 0.0

                if jR <= (iR - 1):
                    deux = prefix_l1[iR - 1] - prefix_l1[jR - 1]
                else:
                    deux = 0.0

                B_val = prem - deux - cum_l2_before_j
                if B_val < B_inner:
                    B_inner = B_val

            B = min(lambda1[jR - 1], B_inner)
            new_val = A + omega[jR - 1, kR - 1] * (B - A)
            mat_spectre[jR - 1, kR - 1] = new_val
            lambda2[jR - 1] = new_val
            cum_l2_before_j += new_val

    return mat_spectre


def CaDsd(
    pi: Union[np.ndarray, list],
    M: Optional[int] = None,
    omega: Optional[np.ndarray] = None,
    rho: Optional[np.ndarray] = None,
    spectre=100,
    U: Optional[np.ndarray] = None,
    option: bool = True,
):
    """
    Faster CaDsd implementation with the same interface and output keys.

    Main speed-ups:
    - uses the faster spec() above;
    - preallocates phi instead of repeated hstack;
    - replaces permutation matrices by direct column indexing;
    - replaces multiplication by a diagonal phase matrix with column scaling.
    """
    pi = np.asarray(pi, dtype=float).ravel()
    N = pi.size

    if M is None:
        M = int(round(pi.sum(), 7))
    if int(M) != M:
        raise ValueError("M should be an integer")
    M = int(M)

    if omega is None:
        omega = 0.5 * np.ones((M, N), dtype=float)
    else:
        omega = np.asarray(omega, dtype=float)
        if omega.shape != (M, N):
            raise ValueError("omega must have shape (M, N)")

    if rho is None:
        rho = 0.5 * np.ones((M, N - 1), dtype=float)
    else:
        rho = np.asarray(rho, dtype=float)
        if rho.shape != (M, N - 1):
            raise ValueError("rho must have shape (M, N-1)")

    rho_angles = np.round(rho * 2 * np.pi, 7)
    mat_spectre = np.round(spec(omega, M, pi, spectre), 7)
    pi_down = np.sort(pi)[::-1]

    if U is None:
        U_work = np.eye(M, dtype=complex)
    else:
        U_work = np.asarray(U, dtype=complex).copy()
        if U_work.shape != (M, M):
            raise ValueError("U must have shape (M, M)")

    phi = np.zeros((M, N), dtype=complex)
    phi[:, 0] = np.round(np.sqrt(pi_down[0]) * U_work[:, 0], 7)
    ens = np.arange(1, M + 1, dtype=int)

    eye_index = np.arange(M)

    for kR in range(2, N + 1):
        phases = np.exp(1j * rho_angles[:, kR - 2])
        lambda1 = mat_spectre[:, kR - 1].copy()
        lambda2 = mat_spectre[:, kR - 2].copy()

        E1 = ens.tolist()
        E2 = ens.tolist()

        # Keep exact equality because mat_spectre is rounded to 7 digits as in the earlier code.
        for jR in ens:
            if not E1:
                break
            val = lambda2[jR - 1]
            lam1_E1 = lambda1[np.array(E1) - 1]
            matches = np.where(lam1_E1 == val)[0]
            if matches.size > 0:
                E2 = [x for x in E2 if x != jR]
                del E1[matches[0]]

        E1_arr = np.array(E1, dtype=int)
        E2_arr = np.array(E2, dtype=int)
        E1_rev = M + 1 - E1_arr
        E2_rev = M + 1 - E2_arr
        r = len(E1_rev)

        if r == 0:
            continue

        if r != M:
            mask1 = np.ones(M, dtype=bool)
            mask2 = np.ones(M, dtype=bool)
            mask1[E1_rev - 1] = False
            mask2[E2_rev - 1] = False
            E1_perm = np.concatenate([np.sort(E1_rev), ens[mask1]])
            E2_perm = np.concatenate([np.sort(E2_rev), ens[mask2]])
            perm1 = E1_perm - 1
            perm2 = E2_perm - 1
        else:
            perm1 = eye_index
            perm2 = eye_index

        lambda2_E2 = lambda2[E2_arr - 1]
        lambda1_E1 = lambda1[E1_arr - 1]
        R = np.column_stack([lambda2_E2, lambda1_E1]).astype(complex)[::-1, :]

        v = np.zeros(r, dtype=complex)
        w = np.zeros(r, dtype=complex)

        for i in range(r):
            v1 = R[i, 0] - R[:, 1]
            v2 = R[i, 0] - R[:, 0]
            v2[i] = 1.0 + 0j
            order = np.argsort(np.abs(v1))
            v[i] = np.round(np.sqrt(-np.prod(v1[order] / v2[order])), 7)

            w1 = R[i, 1] - R[:, 0]
            w2 = R[i, 1] - R[:, 1]
            w2[i] = 1.0 + 0j
            order_w = np.argsort(np.abs(w1))
            w[i] = np.round(np.sqrt(np.prod(w1[order_w] / w2[order_w])), 7)

        col_vals = R[:, 1]
        row_vals = R[:, 0]
        denom = col_vals[np.newaxis, :] - row_vals[:, np.newaxis]
        W = (v[:, None] * w[None, :]) / denom

        # U @ diag(phases) is just column scaling.
        UV = U_work * phases[np.newaxis, :]

        # Equivalent to U %*% V %*% t(sigma2) %*% vect.
        UV_sigma2 = UV[:, perm2]
        phi[:, kR - 1] = UV_sigma2[:, :r] @ v

        # Equivalent to U %*% V %*% t(sigma2) %*% block %*% sigma1.
        U_block = UV_sigma2.copy()
        U_block[:, :r] = UV_sigma2[:, :r] @ W
        U_new = np.empty_like(U_block)
        U_new[:, perm1] = U_block
        U_work = U_new

    d = np.sqrt(mat_spectre[:, N - 1])
    if np.any(d == 0):
        raise ValueError("Zero on diagonal of spectrum, cannot invert sqrt.")

    # Avoid forming an explicit inverse diagonal matrix.
    EigenBasis = (phi.conj().T @ U_work) / d[np.newaxis, :]
    K = np.round(phi.conj().T @ phi, 7)

    return {
        "K": K,
        "spectrum": mat_spectre,
        "EigenBasis": EigenBasis,
    }


print("Fast spec() and CaDsd() loaded.")


Fast spec() and CaDsd() loaded.


In [47]:
import numpy as np

# --------------------------------------------------------
# 1) inclusionprobabilities() – Python version
#    (size measure p -> inclusion probs π with sum π = n)
# --------------------------------------------------------

def inclusionprobabilities(p, n):
    """
    Rough equivalent of sampling::inclusionprobabilities(p, n) in R.

    p : array-like, strictly positive size measures
    n : desired fixed sample size (integer)

    Returns
    -------
    pik : ndarray of length N
        First-order inclusion probabilities, 0 < pik_i <= 1, sum pik_i = n.
    """
    p = np.asarray(p, dtype=float)
    N = p.size
    if n <= 0 or n > N:
        raise ValueError("n must be in 1..N")

    pik = np.zeros(N, dtype=float)
    mask_fixed = np.zeros(N, dtype=bool)  # True where pik is already fixed to 1
    n_rem = float(n)
    p_work = p.copy()

    # Iterative rescaling: set any prob >= 1 to 1, rescale remaining
    while True:
        idx = ~mask_fixed
        if not np.any(idx):
            break

        p_sub = p_work[idx]
        # if all remaining p_sub are zero but n_rem > 0, it's impossible
        if p_sub.sum() <= 0 and n_rem > 1e-12:
            raise RuntimeError("Cannot construct inclusion probabilities with given p and n.")

        temp = n_rem * p_sub / p_sub.sum()
        over = temp >= (1.0 - 1e-12)  # tolerance

        # If no temp >= 1, we're done
        if not np.any(over):
            pik[idx] = temp
            break

        # Fix those >=1 to exactly 1
        idx_global = np.where(idx)[0]
        over_global = idx_global[over]

        pik[over_global] = 1.0
        mask_fixed[over_global] = True
        n_rem -= over.sum()
        p_work[over_global] = 0.0

        if n_rem <= 1e-12:
            # All remaining inclusion probabilities must be zero
            break

    return pik




In [48]:
def to_r_vector(arr, name="vec"):
    arr = np.asarray(arr).ravel()
    values = ", ".join([f"{x:.10f}" for x in arr])
    return f"{name} <- c({values})"

def to_r_matrix_complex(mat, name="mat"):
    mat = np.asarray(mat)
    N, M = mat.shape
    
    r_code = f"{name} <- matrix(c(\n"
    elements = []
    
    for j in range(M):  # R fills by column
        for i in range(N):
            val = mat[i, j]
            real_part = np.real(val)
            imag_part = np.imag(val)
            
            if abs(imag_part) < 1e-10:
                elements.append(f"  {real_part:.10f}")
            else:
                sign = "+" if imag_part >= 0 else ""
                elements.append(f"  complex(real={real_part:.10f}, imaginary={imag_part:.10f})")
    
    r_code += ",\n".join(elements)
    r_code += f"\n), nrow={N}, ncol={M}, byrow=FALSE)"
    return r_code

def to_r_matrix_real(mat, name="mat"):
    mat = np.asarray(mat)
    N, M = mat.shape
    
    r_code = f"{name} <- matrix(c(\n"
    elements = []
    
    for j in range(M):  # R fills by column
        for i in range(N):
            elements.append(f"  {mat[i, j]:.10f}")
    
    r_code += ",\n".join(elements)
    r_code += f"\n), nrow={N}, ncol={M}, byrow=FALSE)"
    return r_code


## Update notes for corrected version

- The reference design in the final comparison is now the **Vincent empirical ordered** $P^{\pi}$ design, using the supplied $z/\pi$ order.
- The CaDsd/ABC candidates still use the internal descending-$\pi$ order required by the constructor.
- The notebook keeps both references:
  - `PpiVincent`: true Vincent empirical $z/\pi$-ordered reference.
  - `PpiInternal`: old/current internal descending-$\pi$ reference.
- A debug/precheck section is inserted before the main run. It verifies that all methods use the same inclusion probabilities and that the Vincent reference is computed correctly.
- Progress and final summaries report **positive ratios** such as `ABC_z/PpiVincent_z`, not negative efficiency percentages.


## ABC Algorithm (improved)


In [49]:
import numpy as np
import time
from typing import Tuple, List, Optional, Dict, Any


class ABCAlgorithm:
    """
    Faster Artificial Bee Colony optimizer for the CaDsd parameterization.

    Speed changes that keep the method intact:
    - variance is computed with precomputed y/pi and z/pi vectors, no dense Dpi multiplications;
    - eigenvalue checks are optional/occasional because CaDsd should return a valid DSD kernel by construction;
    - early stopping is available but conservative;
    - onlooker intensity can be tuned without changing the algorithm.

    Use validation_mode="strict" if you want every candidate checked with eigvalsh.
    Use validation_mode="fast" for the normal Monte Carlo runs.
    """

    def __init__(
        self,
        y_sorted,
        z_sorted,
        pik_sorted,
        var_srs_y,
        var_srs_z,
        M,
        n,
        case_name="",
        objective: str = "eff_z",
        enforce_cadsd_order: bool = True,
        random_state: Optional[int] = None,
        validation_mode: str = "fast",
        eigen_check_interval: int = 0,
        initial_strict_checks: int = 2,
    ):
        self.rng = np.random.default_rng(random_state)

        self.input_y = np.asarray(y_sorted, dtype=float).ravel()
        self.input_z = np.asarray(z_sorted, dtype=float).ravel()
        self.input_pi = np.asarray(pik_sorted, dtype=float).ravel()

        if not (len(self.input_y) == len(self.input_z) == len(self.input_pi)):
            raise ValueError("y_sorted, z_sorted, and pik_sorted must have the same length.")
        if np.any(self.input_pi <= 0):
            raise ValueError("All inclusion probabilities must be positive.")

        self.enforce_cadsd_order = enforce_cadsd_order
        if enforce_cadsd_order:
            self.order = np.argsort(self.input_pi)[::-1]
        else:
            self.order = np.arange(len(self.input_pi))

        self.y_sorted = self.input_y[self.order]
        self.z_sorted = self.input_z[self.order]
        self.pik_sorted = self.input_pi[self.order]

        self.var_srs_y = float(var_srs_y)
        self.var_srs_z = float(var_srs_z)
        self.M = int(M)
        self.n = int(n)
        self.N = len(self.pik_sorted)
        self.case_name = case_name
        self.objective = objective

        if validation_mode not in {"fast", "strict"}:
            raise ValueError("validation_mode must be 'fast' or 'strict'.")
        self.validation_mode = validation_mode
        self.eigen_check_interval = int(eigen_check_interval) if eigen_check_interval else 0
        self.initial_strict_checks = int(initial_strict_checks)

        # Kept for compatibility with downstream sensitivity code.
        self.I_N = np.eye(self.N)
        self.Dpi_inv = np.diag(1.0 / self.pik_sorted)

        # Faster variance vectors: HT variance is (y/pi)' A (y/pi).
        self.y_over_pi = self.y_sorted / self.pik_sorted
        self.z_over_pi = self.z_sorted / self.pik_sorted

        # CaDsd returns K in descending-pi order.
        self.pik_sorted_desc = np.sort(self.input_pi)[::-1]

        self._calculate_optimal()

        # Best solution tracking
        self.global_best_eff_z = 0.0
        self.global_best_eff_y = 0.0
        self.global_best_score = -np.inf
        self.global_best_omega = None
        self.global_best_rho = None

        # History and counters
        self.history = []
        self.history_records = []
        self.scout_history = []
        self.eval_count = 0
        self.valid_count = 0
        self.best_update_count = 0
        self.strict_check_count = 0

    def _variance_pair_from_kernel(self, Kmat: np.ndarray, diag_K: Optional[np.ndarray] = None) -> Tuple[float, float]:
        """
        Fast HT variance calculation for a DSD kernel.

        A = (I-K) * K entrywise, so off-diagonal A_ij = -|K_ij|^2 and
        diagonal A_ii = pi_i(1-pi_i). This avoids two dense diagonal matrix
        multiplications at every candidate evaluation.
        """
        if diag_K is None:
            diag_K = np.real(np.diag(Kmat))

        A = -np.abs(Kmat) ** 2
        np.fill_diagonal(A, diag_K * (1.0 - diag_K))

        var_y = float(np.real(self.y_over_pi @ (A @ self.y_over_pi)))
        var_z = float(np.real(self.z_over_pi @ (A @ self.z_over_pi)))
        return var_y, var_z

    def _calculate_optimal(self):
        """Compute the reference P_pi design efficiency."""
        Base_opt = Ppi(self.pik_sorted)
        Ppi_mat = Base_opt @ Base_opt.T
        diag_K = np.real(np.diag(Ppi_mat))
        var_opt_y, var_opt_z = self._variance_pair_from_kernel(Ppi_mat, diag_K=diag_K)

        self.var_opt_y = var_opt_y
        self.var_opt_z = var_opt_z
        self.eff_y_optimal = self.var_srs_y / var_opt_y if var_opt_y > 0 else np.inf
        self.eff_z_optimal = self.var_srs_z / var_opt_z if var_opt_z > 0 else np.inf

    def _score(self, eff_z: float, eff_y: float) -> float:
        """Objective used by ABC. Default is z-efficiency."""
        if self.objective == "eff_z":
            return eff_z
        if self.objective == "eff_y":
            return eff_y
        if self.objective == "harmonic":
            if eff_z <= 0 or eff_y <= 0:
                return 0.0
            return 2.0 * eff_z * eff_y / (eff_z + eff_y)
        if self.objective == "mean":
            return 0.5 * (eff_z + eff_y)
        raise ValueError("objective must be one of: 'eff_z', 'eff_y', 'harmonic', 'mean'.")

    def _needs_strict_kernel_check(self) -> bool:
        if self.validation_mode == "strict":
            return True
        if self.eval_count <= self.initial_strict_checks:
            return True
        if self.eigen_check_interval and self.eval_count % self.eigen_check_interval == 0:
            return True
        return False

    def _kernel_passes_validation(self, Kmat: np.ndarray, diag_K: np.ndarray) -> bool:
        if not np.all(np.isfinite(diag_K)):
            return False
        if not np.allclose(diag_K, self.pik_sorted_desc, atol=1e-3):
            return False

        # In fast mode, CaDsd's construction is trusted after the diagonal check.
        # Occasional strict checks can still be requested.
        if not self._needs_strict_kernel_check():
            return True

        self.strict_check_count += 1
        try:
            evals = np.linalg.eigvalsh(Kmat)
        except Exception:
            return False

        if not (np.all(evals >= -1e-3) and np.all(evals <= 1.0 + 1e-3)):
            return False
        if not np.isclose(evals.sum(), self.n, atol=1e-3):
            return False
        return True

    def evaluate(self, omega: np.ndarray, rho: np.ndarray) -> Tuple[float, float, bool]:
        """Evaluate a solution and return eff_z, eff_y, valid."""
        self.eval_count += 1

        try:
            K_dict = CaDsd(pi=self.pik_sorted, M=self.M, omega=omega, rho=rho)
            Kmat = K_dict["K"].astype(np.complex128, copy=False)
        except Exception:
            return (0.0, 0.0, False)

        diag_K = np.real(np.diag(Kmat))
        if not self._kernel_passes_validation(Kmat, diag_K):
            return (0.0, 0.0, False)

        var_y, var_z = self._variance_pair_from_kernel(Kmat, diag_K=diag_K)
        if (not np.isfinite(var_y)) or (not np.isfinite(var_z)) or var_y <= 0 or var_z <= 0:
            return (0.0, 0.0, False)

        eff_y = self.var_srs_y / var_y
        eff_z = self.var_srs_z / var_z
        if not (np.isfinite(eff_y) and np.isfinite(eff_z)):
            return (0.0, 0.0, False)

        self.valid_count += 1
        return (eff_z, eff_y, True)

    def _food_from_arrays(self, omega: np.ndarray, rho: np.ndarray, trial: int = 0) -> Optional[dict]:
        eff_z, eff_y, valid = self.evaluate(omega, rho)
        if not valid:
            return None
        score = self._score(eff_z, eff_y)
        food = {
            "omega": omega,
            "rho": rho,
            "eff_z": eff_z,
            "eff_y": eff_y,
            "score": score,
            "trial": trial,
        }
        self._update_global_best(food)
        return food

    def _update_global_best(self, food: dict) -> bool:
        if food["score"] > self.global_best_score:
            self.global_best_score = food["score"]
            self.global_best_eff_z = food["eff_z"]
            self.global_best_eff_y = food["eff_y"]
            self.global_best_omega = food["omega"].copy()
            self.global_best_rho = food["rho"].copy()
            self.best_update_count += 1
            return True
        return False

    def _random_candidate(self) -> Tuple[np.ndarray, np.ndarray]:
        omega = self.rng.random((self.M, self.N))
        rho = self.rng.random((self.M, self.N - 1))
        return omega, rho

    def _candidate_near_best(self, scale: float = 0.05) -> Tuple[np.ndarray, np.ndarray]:
        if self.global_best_omega is None or self.global_best_rho is None:
            return self._random_candidate()
        omega = self.global_best_omega + self.rng.normal(0.0, scale, self.global_best_omega.shape)
        rho = self.global_best_rho + self.rng.normal(0.0, scale, self.global_best_rho.shape)
        return np.clip(omega, 0.0, 1.0), np.clip(rho, 0.0, 1.0)

    def initialize_population(self, colony_size: int, verbose: bool = True) -> List[dict]:
        """Initialize food sources."""
        if verbose:
            print(f" Initializing {colony_size} food sources...")

        population = []

        center_omega = 0.5 * np.ones((self.M, self.N))
        center_rho = 0.5 * np.ones((self.M, self.N - 1))
        food = self._food_from_arrays(center_omega, center_rho)
        if food is not None:
            population.append(food)

        max_attempts = max(colony_size * 20, 120)
        count = 0

        while len(population) < colony_size and count < max_attempts:
            omega, rho = self._random_candidate()
            food = self._food_from_arrays(omega, rho)
            if food is not None:
                population.append(food)

            count += 1
            if verbose and count % 100 == 0:
                print(f"   evaluated {count}, valid sources {len(population)}/{colony_size}", end="\r")

        if verbose:
            print(f"\n    Initialized {len(population)} food sources (requested {colony_size})")
            if len(population) > 0:
                print(f"    Best initial: eff_z={self.global_best_eff_z:.4f}, eff_y={self.global_best_eff_y:.4f}")
            else:
                print("    WARNING: no valid solution found.")

        return population

    def _mutate_food(
        self,
        food: dict,
        partner: dict,
        progress: float,
        mode: str = "abc",
    ) -> Tuple[np.ndarray, np.ndarray]:
        """Create a new candidate with adaptive exploration."""
        adaptive_scale = max(0.12, 1.0 - 0.88 * progress)

        if mode == "local":
            return self._candidate_near_best(scale=0.015 + 0.07 * (1.0 - progress))

        phi_omega = self.rng.uniform(-adaptive_scale, adaptive_scale, food["omega"].shape)
        phi_rho = self.rng.uniform(-adaptive_scale, adaptive_scale, food["rho"].shape)

        new_omega = food["omega"] + phi_omega * (food["omega"] - partner["omega"])
        new_rho = food["rho"] + phi_rho * (food["rho"] - partner["rho"])

        if self.global_best_omega is not None and progress > 0.10:
            pull = self.rng.uniform(0.0, 0.30 * progress)
            new_omega = new_omega + pull * (self.global_best_omega - new_omega)
            new_rho = new_rho + pull * (self.global_best_rho - new_rho)

        jitter = 0.006 * adaptive_scale
        new_omega = new_omega + self.rng.normal(0.0, jitter, new_omega.shape)
        new_rho = new_rho + self.rng.normal(0.0, jitter, new_rho.shape)

        return np.clip(new_omega, 0.0, 1.0), np.clip(new_rho, 0.0, 1.0)

    def _greedy_replace(self, old_food: dict, omega: np.ndarray, rho: np.ndarray) -> dict:
        new_food = self._food_from_arrays(omega, rho)
        if new_food is not None and new_food["score"] > old_food["score"]:
            return new_food
        old_copy = old_food.copy()
        old_copy["trial"] = old_food["trial"] + 1
        return old_copy

    def employed_bee_phase(self, population: List[dict], progress: float) -> List[dict]:
        if len(population) == 0:
            return population

        new_population = []
        for i, food in enumerate(population):
            if len(population) > 1:
                k = int(self.rng.integers(0, len(population) - 1))
                if k >= i:
                    k += 1
            else:
                k = i

            new_omega, new_rho = self._mutate_food(food, population[k], progress, mode="abc")
            new_population.append(self._greedy_replace(food, new_omega, new_rho))

        return new_population

    def onlooker_bee_phase(
        self,
        population: List[dict],
        progress: float,
        onlooker_factor: float = 0.60,
    ) -> List[dict]:
        """
        Onlooker phase with tunable intensity.

        onlooker_factor=1.0 is the standard full onlooker phase.
        onlooker_factor=0.5 to 0.7 is much faster and usually enough for the
        large Monte Carlo grid.
        """
        if len(population) == 0 or onlooker_factor <= 0:
            return population

        scores = np.array([food["score"] for food in population], dtype=float)
        scores = scores - np.min(scores) + 1e-12
        if not np.isfinite(scores).all() or scores.sum() <= 0:
            probs = np.ones(len(population)) / len(population)
        else:
            probs = scores / scores.sum()

        new_population = [food.copy() for food in population]
        n_onlookers = max(1, int(round(onlooker_factor * len(population))))

        for _ in range(n_onlookers):
            i = int(self.rng.choice(len(population), p=probs))
            if len(population) > 1:
                k = int(self.rng.integers(0, len(population) - 1))
                if k >= i:
                    k += 1
            else:
                k = i

            mode = "local" if self.rng.random() < 0.10 else "abc"
            new_omega, new_rho = self._mutate_food(new_population[i], population[k], progress, mode=mode)
            new_population[i] = self._greedy_replace(new_population[i], new_omega, new_rho)

        return new_population

    def scout_bee_phase(self, population: List[dict], limit: int, progress: float) -> Tuple[List[dict], int]:
        if len(population) == 0:
            return population, 0

        new_population = []
        n_abandoned = 0

        for food in population:
            if food["trial"] >= limit:
                n_abandoned += 1

                best_new = None
                attempts = 0
                max_attempts = 25

                while attempts < max_attempts:
                    if self.rng.random() < 0.60:
                        omega, rho = self._candidate_near_best(scale=0.03 + 0.08 * (1.0 - progress))
                    else:
                        omega, rho = self._random_candidate()

                    candidate = self._food_from_arrays(omega, rho)
                    if candidate is not None and (best_new is None or candidate["score"] > best_new["score"]):
                        best_new = candidate
                    attempts += 1

                if best_new is not None:
                    best_new["trial"] = 0
                    new_population.append(best_new)
                else:
                    food_copy = food.copy()
                    food_copy["trial"] = 0
                    new_population.append(food_copy)
            else:
                new_population.append(food)

        return new_population, n_abandoned

    def local_search_phase(self, population: List[dict], progress: float, attempts: int = 2) -> List[dict]:
        """Small memetic improvement around the current global best."""
        if len(population) == 0 or self.global_best_omega is None or attempts <= 0:
            return population

        best_candidate = None
        scale = max(0.004, 0.03 * (1.0 - progress))

        for _ in range(attempts):
            omega, rho = self._candidate_near_best(scale=scale)
            candidate = self._food_from_arrays(omega, rho)
            if candidate is not None and (best_candidate is None or candidate["score"] > best_candidate["score"]):
                best_candidate = candidate

        if best_candidate is not None:
            worst_idx = int(np.argmin([food["score"] for food in population]))
            if best_candidate["score"] > population[worst_idx]["score"]:
                population[worst_idx] = best_candidate

        return population

    def _inject_elite(self, population: List[dict]) -> List[dict]:
        """Ensure the current global best is not lost."""
        if len(population) == 0 or self.global_best_omega is None:
            return population

        scores = np.array([food["score"] for food in population])
        if np.max(scores) + 1e-15 < self.global_best_score:
            worst_idx = int(np.argmin(scores))
            population[worst_idx] = {
                "omega": self.global_best_omega.copy(),
                "rho": self.global_best_rho.copy(),
                "eff_z": self.global_best_eff_z,
                "eff_y": self.global_best_eff_y,
                "score": self.global_best_score,
                "trial": 0,
            }

        return population

    def optimize(
        self,
        colony_size: int = 40,
        max_iterations: int = 100,
        limit: int = 25,
        verbose: bool = True,
        progress_interval: Optional[int] = None,
        local_search_interval: int = 25,
        local_search_attempts: int = 2,
        onlooker_factor: float = 0.60,
        early_stopping: bool = True,
        patience: Optional[int] = None,
        min_iterations: Optional[int] = None,
        rel_tol: float = 1e-5,
        time_limit_seconds: Optional[float] = None,
        random_searcher: Optional["RandomSearchAlgorithm"] = None,
        random_evals_per_iteration: Optional[int] = None,
        random_start_evals: Optional[int] = None,
    ) -> dict:
        """Run the ABC algorithm, optionally with a side-by-side random-search baseline."""
        start_time = time.time()

        if progress_interval is None:
            progress_interval = max(1, max_iterations // 20)
        if patience is None:
            patience = max(25, max_iterations // 4)
        if min_iterations is None:
            min_iterations = max(20, max_iterations // 3)

        use_random = random_searcher is not None
        if use_random:
            if random_evals_per_iteration is None:
                random_evals_per_iteration = max(1, int(round(colony_size * (1.0 + onlooker_factor))))
            if random_start_evals is None:
                random_start_evals = colony_size

        if verbose:
            print("=" * 80)
            print(f" ABC ALGORITHM - {self.case_name}")
            print("=" * 80)
            print(" Configuration:")
            print(f"   colony_size          = {colony_size}")
            print(f"   max_iterations       = {max_iterations}")
            print(f"   abandonment limit    = {limit}")
            print(f"   objective            = {self.objective}")
            print(f"   validation mode      = {self.validation_mode}")
            print(f"   onlooker factor      = {onlooker_factor}")
            print(f"   early stopping       = {early_stopping}, patience={patience}")
            print(f"   time limit           = {time_limit_seconds}")
            if use_random:
                print(f"   random search        = ON")
                print(f"   random start evals   = {random_start_evals}")
                print(f"   random evals/iter    = {random_evals_per_iteration}")
            else:
                print(f"   random search        = OFF")
            print(f"   progress interval    = every {progress_interval} iterations")
            print(f"   local search interval= every {local_search_interval} iterations")
            print(" Reference P_pi:")
            print(f"   eff_z = {self.eff_z_optimal:.4f}")
            print(f"   eff_y = {self.eff_y_optimal:.4f}\n")

        population = self.initialize_population(colony_size, verbose)

        if len(population) == 0:
            if verbose:
                print("\nFAILED: could not initialize any valid solution.")
            return self._prepare_results(time.time() - start_time, colony_size, 0)

        if use_random and random_start_evals and random_start_evals > 0:
            random_searcher.step(random_start_evals, include_center_once=True)

        total_abandoned = 0
        start_best = self.global_best_eff_z
        last_improvement_iter = 0
        best_score_for_stopping = self.global_best_score
        stopped_early = False
        stopped_by_time = False
        completed_iterations = 0

        if verbose:
            if use_random:
                header = (
                    f"{'iter':>7} | {'ABC_z':>10} | {'Rand_z':>10} | "
                    f"{'ABC-Pπ':>9} | {'Rand-Pπ':>9} | {'scouts':>6} | "
                    f"{'ABC valid/eval':>14} | {'Rand valid/eval':>15} | {'elapsed':>8}"
                )
            else:
                header = (
                    f"{'iter':>9} | {'best_z':>10} | {'best_y':>10} | "
                    f"{'Δstart':>9} | {'vs Pπ':>9} | {'scouts':>6} | {'valid/eval':>11} | {'elapsed':>8}"
                )
            print(header)
            print("-" * len(header))

        for iteration in range(max_iterations):
            progress = (iteration + 1) / max_iterations

            old_best = self.global_best_score

            population = self.employed_bee_phase(population, progress)
            population = self.onlooker_bee_phase(population, progress, onlooker_factor=onlooker_factor)
            population, n_abandoned = self.scout_bee_phase(population, limit, progress)
            total_abandoned += n_abandoned

            if local_search_interval and (iteration + 1) % local_search_interval == 0:
                population = self.local_search_phase(
                    population,
                    progress,
                    attempts=local_search_attempts,
                )

            population = self._inject_elite(population)

            if use_random and random_evals_per_iteration and random_evals_per_iteration > 0:
                random_searcher.step(random_evals_per_iteration, include_center_once=False)

            completed_iterations = iteration + 1

            if self.global_best_score > old_best * (1.0 + rel_tol):
                last_improvement_iter = iteration + 1
                best_score_for_stopping = self.global_best_score

            self.history.append(self.global_best_eff_z)
            self.scout_history.append(n_abandoned)
            record = {
                "iteration": iteration + 1,
                "best_eff_z": self.global_best_eff_z,
                "best_eff_y": self.global_best_eff_y,
                "best_score": self.global_best_score,
                "n_abandoned": n_abandoned,
                "eval_count": self.eval_count,
                "valid_count": self.valid_count,
                "strict_check_count": self.strict_check_count,
                "elapsed": time.time() - start_time,
            }
            if use_random:
                record.update({
                    "random_best_eff_z": random_searcher.global_best_eff_z,
                    "random_best_eff_y": random_searcher.global_best_eff_y,
                    "random_best_score": random_searcher.global_best_score,
                    "random_eval_count": random_searcher.eval_count,
                    "random_valid_count": random_searcher.valid_count,
                })
            self.history_records.append(record)

            should_print = (
                verbose
                and (
                    iteration == 0
                    or (iteration + 1) % progress_interval == 0
                    or (iteration + 1) == max_iterations
                )
            )

            if should_print:
                delta_start = (
                    100.0 * (self.global_best_eff_z / start_best - 1.0)
                    if start_best > 0 else 0.0
                )
                vs_ppi = (
                    100.0 * (self.global_best_eff_z / self.eff_z_optimal - 1.0)
                    if self.eff_z_optimal > 0 else np.nan
                )
                valid_ratio = f"{self.valid_count}/{self.eval_count}"
                elapsed = time.time() - start_time

                if use_random:
                    rand_vs_ppi = (
                        100.0 * (random_searcher.global_best_eff_z / self.eff_z_optimal - 1.0)
                        if self.eff_z_optimal > 0 and random_searcher.global_best_eff_z > 0 else np.nan
                    )
                    rand_valid_ratio = f"{random_searcher.valid_count}/{random_searcher.eval_count}"
                    print(
                        f"{iteration+1:7d} | {self.global_best_eff_z:10.4f} | "
                        f"{random_searcher.global_best_eff_z:10.4f} | "
                        f"{vs_ppi:+8.2f}% | {rand_vs_ppi:+8.2f}% | {n_abandoned:6d} | "
                        f"{valid_ratio:>14} | {rand_valid_ratio:>15} | {elapsed:7.1f}s"
                    )
                else:
                    print(
                        f"{iteration+1:9d} | {self.global_best_eff_z:10.4f} | "
                        f"{self.global_best_eff_y:10.4f} | {delta_start:+8.2f}% | "
                        f"{vs_ppi:+8.2f}% | {n_abandoned:6d} | {valid_ratio:>11} | "
                        f"{elapsed:7.1f}s"
                    )

            if (
                early_stopping
                and (iteration + 1) >= min_iterations
                and (iteration + 1 - last_improvement_iter) >= patience
            ):
                stopped_early = True
                if verbose:
                    print(
                        f"    Early stop at iter {iteration+1}: "
                        f"no relative improvement > {rel_tol:g} for {patience} iterations."
                    )
                break

            if (
                time_limit_seconds is not None
                and (iteration + 1) >= min_iterations
                and (time.time() - start_time) >= float(time_limit_seconds)
            ):
                stopped_by_time = True
                if verbose:
                    print(f"    Time stop at iter {iteration+1}: reached {time_limit_seconds}s budget.")
                break

        total_time = time.time() - start_time
        results = self._prepare_results(total_time, colony_size, total_abandoned)
        if use_random:
            random_relative_to_ppi = (
                100.0 * (random_searcher.global_best_eff_z / self.eff_z_optimal - 1.0)
                if self.eff_z_optimal > 0 and random_searcher.global_best_eff_z > 0 else -np.inf
            )
            results.update({
                "random_best_eff_z": random_searcher.global_best_eff_z,
                "random_best_eff_y": random_searcher.global_best_eff_y,
                "random_best_score": random_searcher.global_best_score,
                "random_relative_to_ppi_percent": random_relative_to_ppi,
                "random_eval_count": random_searcher.eval_count,
                "random_valid_count": random_searcher.valid_count,
                "random_history": random_searcher.history.copy(),
                "random_history_records": getattr(random_searcher, "random_history_records", []).copy(),
                "random_search_enabled": True,
                "random_evals_per_iteration": random_evals_per_iteration,
                "random_start_evals": random_start_evals,
            })
        else:
            results["random_search_enabled"] = False
        results["stopped_early"] = stopped_early
        results["stopped_by_time"] = stopped_by_time
        results["completed_iterations"] = completed_iterations
        results["strict_check_count"] = self.strict_check_count
        return results

    def _prepare_results(self, total_time: float, colony_size: int, total_abandoned: int) -> dict:
        relative_to_ppi = (
            100.0 * (self.global_best_eff_z / self.eff_z_optimal - 1.0)
            if self.eff_z_optimal > 0 and self.global_best_eff_z > 0 else -np.inf
        )

        gap_percent = (
            100.0 * (self.eff_z_optimal / self.global_best_eff_z - 1.0)
            if self.eff_z_optimal > 0 and self.global_best_eff_z > 0 else np.inf
        )

        success = self.global_best_eff_z > self.eff_z_optimal

        return {
            "case": self.case_name,
            "best_eff_z": self.global_best_eff_z,
            "best_eff_y": self.global_best_eff_y,
            "best_score": self.global_best_score,
            "best_omega": self.global_best_omega,
            "best_rho": self.global_best_rho,
            "optimal_eff_z": self.eff_z_optimal,
            "optimal_eff_y": self.eff_y_optimal,
            "success": success,
            "gap_percent": gap_percent,
            "relative_to_ppi_percent": relative_to_ppi,
            "improvement_percent": max(relative_to_ppi, 0.0),
            "total_time": total_time,
            "colony_size": colony_size,
            "total_abandoned": total_abandoned,
            "eval_count": self.eval_count,
            "valid_count": self.valid_count,
            "history": self.history.copy(),
            "history_records": self.history_records.copy(),
            "scout_history": self.scout_history.copy(),
            "order": self.order.copy(),
            "enforce_cadsd_order": self.enforce_cadsd_order,
            "validation_mode": self.validation_mode,
            "strict_check_count": self.strict_check_count,
        }

    def print_results(self, results: dict):
        """Print formatted final results."""
        print("=" * 80)
        print(" FINAL RESULTS (ABC)")
        print("=" * 80)
        print(f" Case: {results['case']}")
        print("\n Best solution:")
        print(f"   eff_z = {results['best_eff_z']:.6f}")
        print(f"   eff_y = {results['best_eff_y']:.6f}")
        print("\n Reference P_pi:")
        print(f"   eff_z = {results['optimal_eff_z']:.6f}")
        print(f"   eff_y = {results['optimal_eff_y']:.6f}")
        print("\n Comparison:")
        print(f"   ABC vs P_pi = {results['relative_to_ppi_percent']:+.2f}%")
        print("\n Performance:")
        print(f"   time              = {results['total_time']:.2f}s")
        print(f"   colony size       = {results['colony_size']}")
        print(f"   total scouts      = {results['total_abandoned']}")
        print(f"   valid/eval        = {results['valid_count']}/{results['eval_count']}")
        print(f"   strict checks     = {results.get('strict_check_count', 0)}")
        print(f"   stopped early     = {results.get('stopped_early', False)}")
        print(f"   stopped by time   = {results.get('stopped_by_time', False)}")

        if len(results["history"]) > 0 and results["history"][0] > 0:
            conv = 100.0 * (results["history"][-1] / results["history"][0] - 1.0)
            print("\n Convergence:")
            print(f"   start eff_z = {results['history'][0]:.4f}")
            print(f"   final eff_z = {results['history'][-1]:.4f}")
            print(f"   gain        = {conv:+.2f}%")
        print("=" * 80)




class RandomSearchAlgorithm(ABCAlgorithm):
    """
    Pure random-search baseline using exactly the same CaDsd parameterization
    and the same efficiency evaluation as ABC.

    This is not intended to replace ABC. It is a reference curve: if ABC is
    genuinely learning, ABC_z should improve faster than Rand_z under a similar
    evaluation budget.
    """

    def __init__(self, *args, **kwargs):
        super().__init__(*args, **kwargs)
        self.center_evaluated = False
        self.random_history_records = []

    def step(self, n_evals: int = 1, include_center_once: bool = False) -> dict:
        """Run n random candidate evaluations and update the random-search best."""
        start_time = time.time()
        n_evals = int(max(0, n_evals))
        done = 0

        if include_center_once and (not self.center_evaluated) and n_evals > 0:
            center_omega = 0.5 * np.ones((self.M, self.N))
            center_rho = 0.5 * np.ones((self.M, self.N - 1))
            self._food_from_arrays(center_omega, center_rho)
            self.center_evaluated = True
            done += 1

        while done < n_evals:
            omega, rho = self._random_candidate()
            self._food_from_arrays(omega, rho)
            done += 1

        self.history.append(self.global_best_eff_z)
        record = {
            "step_evals": n_evals,
            "best_eff_z": self.global_best_eff_z,
            "best_eff_y": self.global_best_eff_y,
            "best_score": self.global_best_score,
            "eval_count": self.eval_count,
            "valid_count": self.valid_count,
            "strict_check_count": self.strict_check_count,
            "elapsed_step": time.time() - start_time,
        }
        self.random_history_records.append(record)
        return record

    def optimize(
        self,
        max_evaluations: int = 1000,
        verbose: bool = True,
        progress_interval: Optional[int] = None,
        include_center_once: bool = True,
    ) -> dict:
        """Standalone random search, useful for quick debugging."""
        start_time = time.time()
        if progress_interval is None:
            progress_interval = max(1, max_evaluations // 20)

        if verbose:
            print("=" * 80)
            print(f" RANDOM SEARCH - {self.case_name}")
            print("=" * 80)
            print(f" Reference P_pi eff_z = {self.eff_z_optimal:.4f}")
            print(f"{'eval':>9} | {'Rand_z':>10} | {'Rand_y':>10} | {'Rand-Pπ':>9} | {'valid/eval':>11} | {'elapsed':>8}")
            print("-" * 76)

        done = 0
        while done < max_evaluations:
            batch = min(progress_interval, max_evaluations - done)
            self.step(batch, include_center_once=(include_center_once and done == 0))
            done += batch
            if verbose:
                rand_vs_ppi = (
                    100.0 * (self.global_best_eff_z / self.eff_z_optimal - 1.0)
                    if self.eff_z_optimal > 0 and self.global_best_eff_z > 0 else np.nan
                )
                print(
                    f"{done:9d} | {self.global_best_eff_z:10.4f} | "
                    f"{self.global_best_eff_y:10.4f} | {rand_vs_ppi:+8.2f}% | "
                    f"{self.valid_count}/{self.eval_count:>5} | {time.time() - start_time:7.1f}s"
                )

        results = self._prepare_results(time.time() - start_time, colony_size=0, total_abandoned=0)
        results["search_type"] = "random"
        results["max_evaluations"] = max_evaluations
        return results


print("Faster ABCAlgorithm and RandomSearchAlgorithm class loaded. Use validation_mode='strict' only for checking/debugging.")


Faster ABCAlgorithm and RandomSearchAlgorithm class loaded. Use validation_mode='strict' only for checking/debugging.


In [50]:
# ============================================================
# Terminal progress patch: positive ratios only
# ============================================================

def _valid_percent(valid, total):
    return 100.0 * valid / total if total and total > 0 else 0.0


def _ratio_to_ref(value, ref):
    """
    Positive efficiency ratio.
    value/ref = 1.00 means equal to Pπ.
    value/ref = 0.95 means 95% of Pπ.
    value/ref = 1.05 means 5% better than Pπ.
    """
    return value / ref if ref and ref > 0 and value > 0 else np.nan


def _abc_optimize_progress_terminal(
    self,
    colony_size=40,
    max_iterations=1000,
    limit=25,
    verbose=True,
    progress_interval=None,
    local_search_interval=25,
    local_search_attempts=2,
    onlooker_factor=0.60,
    early_stopping=True,
    patience=None,
    min_iterations=None,
    rel_tol=1e-5,
    time_limit_seconds=None,
    random_searcher=None,
    random_evals_per_iteration=None,
    random_start_evals=None,
):
    start_time = time.time()

    if progress_interval is None:
        progress_interval = max(1, max_iterations // 20)
    if patience is None:
        patience = max(25, max_iterations // 4)
    if min_iterations is None:
        min_iterations = max(20, max_iterations // 3)

    use_random = random_searcher is not None

    if use_random:
        if random_evals_per_iteration is None:
            random_evals_per_iteration = max(1, int(round(colony_size * (1.0 + onlooker_factor))))
        if random_start_evals is None:
            random_start_evals = colony_size

    if verbose:
        print("=" * 80, flush=True)
        print(f" ABC ALGORITHM - {self.case_name}", flush=True)
        print("=" * 80, flush=True)
        print(" Configuration:", flush=True)
        print(f"   colony_size          = {colony_size}", flush=True)
        print(f"   max_iterations       = {max_iterations}", flush=True)
        print(f"   abandonment limit    = {limit}", flush=True)
        print(f"   objective            = {self.objective}", flush=True)
        print(f"   validation mode      = {getattr(self, 'validation_mode', 'unknown')}", flush=True)
        print(f"   onlooker factor      = {onlooker_factor}", flush=True)
        print(f"   early stopping       = {early_stopping}, patience={patience}", flush=True)
        print(f"   time limit           = {time_limit_seconds}", flush=True)
        print(f"   random search        = {'ON' if use_random else 'OFF'}", flush=True)

        if use_random:
            print(f"   random start evals   = {random_start_evals}", flush=True)
            print(f"   random evals/iter    = {random_evals_per_iteration}", flush=True)

        print(f"   progress interval    = every {progress_interval} iterations", flush=True)
        print(f"   local search interval= every {local_search_interval} iterations", flush=True)
        print(" Reference Pπ efficiency versus SRS:", flush=True)
        print(f"   Pπ_z = {self.eff_z_optimal:.2f}", flush=True)
        print(f"   Pπ_y = {self.eff_y_optimal:.2f}\n", flush=True)

    population = self.initialize_population(colony_size, verbose)

    if len(population) == 0:
        if verbose:
            print("\nFAILED: could not initialize any valid solution.", flush=True)
        return self._prepare_results(time.time() - start_time, colony_size, 0)

    if use_random and random_start_evals and random_start_evals > 0:
        random_searcher.step(random_start_evals, include_center_once=True)

    total_abandoned = 0
    last_improvement_iter = 0
    stopped_early = False
    stopped_by_time = False
    completed_iterations = 0

    if verbose:
        if use_random:
            header = (
                f"{'iter':>5} | "
                f"{'ABC_z/Pπ_z':>14} | {'ABC_y/Pπ_y':>14} | "
                f"{'Rand_z/Pπ_z':>14} | {'Rand_y/Pπ_y':>14} | "
                f"{'scout':>5} | {'ABC val%':>8} | {'Rnd val%':>8} | {'elapsed':>8}"
            )
        else:
            header = (
                f"{'iter':>5} | "
                f"{'ABC_z/Pπ_z':>14} | {'ABC_y/Pπ_y':>14} | "
                f"{'scout':>5} | {'ABC val%':>8} | {'elapsed':>8}"
            )

        print(header, flush=True)
        print("-" * len(header), flush=True)

    for iteration in range(max_iterations):
        progress = (iteration + 1) / max_iterations
        old_best = self.global_best_score

        population = self.employed_bee_phase(population, progress)
        population = self.onlooker_bee_phase(
            population,
            progress,
            onlooker_factor=onlooker_factor,
        )
        population, n_abandoned = self.scout_bee_phase(population, limit, progress)
        total_abandoned += n_abandoned

        if local_search_interval and (iteration + 1) % local_search_interval == 0:
            population = self.local_search_phase(
                population,
                progress,
                attempts=local_search_attempts,
            )

        population = self._inject_elite(population)

        if use_random and random_evals_per_iteration and random_evals_per_iteration > 0:
            random_searcher.step(random_evals_per_iteration, include_center_once=False)

        completed_iterations = iteration + 1

        if self.global_best_score > old_best * (1.0 + rel_tol):
            last_improvement_iter = iteration + 1

        record = {
            "iteration": iteration + 1,
            "best_eff_z": self.global_best_eff_z,
            "best_eff_y": self.global_best_eff_y,
            "best_score": self.global_best_score,
            "abc_z_over_ppi_z": _ratio_to_ref(self.global_best_eff_z, self.eff_z_optimal),
            "abc_y_over_ppi_y": _ratio_to_ref(self.global_best_eff_y, self.eff_y_optimal),
            "n_abandoned": n_abandoned,
            "eval_count": self.eval_count,
            "valid_count": self.valid_count,
            "abc_valid_percent": _valid_percent(self.valid_count, self.eval_count),
            "strict_check_count": getattr(self, "strict_check_count", 0),
            "elapsed": time.time() - start_time,
        }

        if use_random:
            record.update({
                "random_best_eff_z": random_searcher.global_best_eff_z,
                "random_best_eff_y": random_searcher.global_best_eff_y,
                "random_best_score": random_searcher.global_best_score,
                "random_z_over_ppi_z": _ratio_to_ref(random_searcher.global_best_eff_z, self.eff_z_optimal),
                "random_y_over_ppi_y": _ratio_to_ref(random_searcher.global_best_eff_y, self.eff_y_optimal),
                "random_eval_count": random_searcher.eval_count,
                "random_valid_count": random_searcher.valid_count,
                "random_valid_percent": _valid_percent(random_searcher.valid_count, random_searcher.eval_count),
            })

        self.history.append(self.global_best_eff_z)
        self.history_records.append(record)
        self.scout_history.append(n_abandoned)

        should_print = (
            verbose
            and (
                iteration == 0
                or (iteration + 1) % progress_interval == 0
                or (iteration + 1) == max_iterations
            )
        )

        if should_print:
            abc_z_ratio = _ratio_to_ref(self.global_best_eff_z, self.eff_z_optimal)
            abc_y_ratio = _ratio_to_ref(self.global_best_eff_y, self.eff_y_optimal)
            abc_valid = _valid_percent(self.valid_count, self.eval_count)
            elapsed = time.time() - start_time

            if use_random:
                rand_z_ratio = _ratio_to_ref(random_searcher.global_best_eff_z, self.eff_z_optimal)
                rand_y_ratio = _ratio_to_ref(random_searcher.global_best_eff_y, self.eff_y_optimal)
                rand_valid = _valid_percent(random_searcher.valid_count, random_searcher.eval_count)

                print(
                    f"{iteration+1:5d} | "
                    f"{abc_z_ratio:14.8f} | {abc_y_ratio:14.8f} | "
                    f"{rand_z_ratio:14.8f} | {rand_y_ratio:14.8f} | "
                    f"{n_abandoned:5d} | {abc_valid:7.2f}% | {rand_valid:7.2f}% | "
                    f"{elapsed:7.2f}s",
                    flush=True,
                )
            else:
                print(
                    f"{iteration+1:5d} | "
                    f"{abc_z_ratio:14.8f} | {abc_y_ratio:14.8f} | "
                    f"{n_abandoned:5d} | {abc_valid:7.2f}% | {elapsed:7.2f}s",
                    flush=True,
                )

        if (
            early_stopping
            and (iteration + 1) >= min_iterations
            and (iteration + 1 - last_improvement_iter) >= patience
        ):
            stopped_early = True
            if verbose:
                print(
                    f"    Early stop at iter {iteration+1}: "
                    f"no relative improvement > {rel_tol:g} for {patience} iterations.",
                    flush=True,
                )
            break

        if (
            time_limit_seconds is not None
            and (iteration + 1) >= min_iterations
            and (time.time() - start_time) >= float(time_limit_seconds)
        ):
            stopped_by_time = True
            if verbose:
                print(f"    Time stop at iter {iteration+1}: reached {time_limit_seconds}s budget.", flush=True)
            break

    total_time = time.time() - start_time
    results = self._prepare_results(total_time, colony_size, total_abandoned)

    results["abc_z_over_Ppi_z"] = _ratio_to_ref(self.global_best_eff_z, self.eff_z_optimal)
    results["abc_y_over_Ppi_y"] = _ratio_to_ref(self.global_best_eff_y, self.eff_y_optimal)
    results["abc_valid_percent"] = _valid_percent(self.valid_count, self.eval_count)

    if use_random:
        results.update({
            "random_best_eff_z": random_searcher.global_best_eff_z,
            "random_best_eff_y": random_searcher.global_best_eff_y,
            "random_best_score": random_searcher.global_best_score,
            "random_z_over_Ppi_z": _ratio_to_ref(random_searcher.global_best_eff_z, self.eff_z_optimal),
            "random_y_over_Ppi_y": _ratio_to_ref(random_searcher.global_best_eff_y, self.eff_y_optimal),
            "random_eval_count": random_searcher.eval_count,
            "random_valid_count": random_searcher.valid_count,
            "random_valid_percent": _valid_percent(random_searcher.valid_count, random_searcher.eval_count),
            "random_history": random_searcher.history.copy(),
            "random_history_records": getattr(random_searcher, "random_history_records", []).copy(),
            "random_search_enabled": True,
            "random_evals_per_iteration": random_evals_per_iteration,
            "random_start_evals": random_start_evals,
        })
    else:
        results["random_search_enabled"] = False

    results["stopped_early"] = stopped_early
    results["stopped_by_time"] = stopped_by_time
    results["completed_iterations"] = completed_iterations
    results["strict_check_count"] = getattr(self, "strict_check_count", 0)

    return results


ABCAlgorithm.optimize = _abc_optimize_progress_terminal

## Run ABC on MU284: corrected Vincent reference

### main parts

In [51]:
import pandas as pd
import numpy as np
from pathlib import Path

# Try the normal repository path first, then local filenames.
MU284_PATH_CANDIDATES = [
    Path("/home/bardia/projects/graphical-sampling/simulations_abc/populations/real/MU284.csv"),
    Path("MU284.csv"),
    Path("MU284_filtered.csv"),
    Path("/mnt/data/MU284.csv"),
    Path("/mnt/data/MU284_filtered.csv"),
]

MU284_PATH = None
for candidate in MU284_PATH_CANDIDATES:
    if candidate.exists():
        MU284_PATH = candidate
        break

if MU284_PATH is None:
    raise FileNotFoundError(
        "Could not find MU284 data. Put MU284.csv or MU284_filtered.csv in the notebook folder, "
        "or edit MU284_PATH_CANDIDATES."
    )

df_full = pd.read_csv(MU284_PATH)

df = df_full.sample(n=30, random_state=2312).reset_index(drop=True)
# df = df_full.copy()
df["CONST"] = 1.0
# df = df_full.copy()
print("=" * 80)
print("MU284 DATA LOADED")
print("=" * 80)
print(f"path = {MU284_PATH}")
print(f"N = {len(df)}")
print(f"columns = {list(df.columns)}")

y_var = "P85"

print("\nCorrelations with P85:")
for col in ['P85', 'P75', 'RMT85', 'CS82', 'SS82', 'S82', 'ME84', 'REV84', 'REG',]:
    if col in df.columns and y_var in df.columns:
        corr = np.corrcoef(df[y_var].to_numpy(dtype=float), df[col].to_numpy(dtype=float))[0, 1]
        print(f"Corr({y_var}, {col}) = {corr:.3f}")

MU284 DATA LOADED
path = /home/bardia/projects/graphical-sampling/simulations_abc/populations/real/MU284.csv
N = 30
columns = ['LABEL', 'P85', 'P75', 'RMT85', 'CS82', 'SS82', 'S82', 'ME84', 'REV84', 'REG', 'CL', 'CONST']

Correlations with P85:
Corr(P85, P85) = 1.000
Corr(P85, P75) = 0.988
Corr(P85, RMT85) = 0.978
Corr(P85, CS82) = 0.665
Corr(P85, SS82) = 0.403
Corr(P85, S82) = 0.807
Corr(P85, ME84) = 0.986
Corr(P85, REV84) = 0.746
Corr(P85, REG) = -0.226


### Setting the parameters

In [52]:
import numpy as np
import pandas as pd

# ============================================================
# USER SETTINGS: edit only this block for the main run
# ============================================================

# Recommended first test:
# use unequal inclusion probabilities, because the inverse omega/rho
# representation is much more stable than for constant pi.
test_cases = [
    # ("ME84",    "REV84", "High Corr"),
    # ("S82",     "REV84", "Medium Corr"),
    # ("CS82",    "REV84", "Low Corr"),

    # Equal pi cases are now handled by the hybrid global-jump part.
    # Uncomment these when you want the CONST tests.
    ("ME84",    "CONST", "High Corr"),
    ("S82",     "CONST", "Medium Corr"),
    ("CS82",    "CONST", "Low Corr"),
]

sample_sizes = [5]
y_var = "P85"

ABC_RANDOM_SEED = 12345
RANDOM_SEARCH_SEED = 54321
OBJECTIVE = "eff_z"      # options: "eff_z", "eff_y", "harmonic", "mean"

SIM_PARAMS = {
    "colony_size": 30,
    "max_iterations": 1000,
    "limit": 25,
    "onlooker_factor": 1.00,
    "local_search_attempts": 4,
    "local_search_interval": 10,
    "progress_interval": 50,
    "validation_mode": "fast",
    "time_limit_seconds": None,

    "early_stopping": False,
    "patience": 50,
    "min_iterations": 20,
    "rel_tol": 1e-5,

    "random_start_evals": 10,
    "random_evals_per_iteration": 20,
}

OUTPUT_CSV = "abc_random_mu284_hybrid_v16_fast_auto.csv"

print("USER SETTINGS LOADED")
print(f"test_cases = {test_cases}")
print(f"sample_sizes = {sample_sizes}")
print(f"y_var = {y_var}")
print(f"OBJECTIVE = {OBJECTIVE}")
print(f"OUTPUT_CSV = {OUTPUT_CSV}")


USER SETTINGS LOADED
test_cases = [('ME84', 'CONST', 'High Corr'), ('S82', 'CONST', 'Medium Corr'), ('CS82', 'CONST', 'Low Corr')]
sample_sizes = [5]
y_var = P85
OBJECTIVE = eff_z
OUTPUT_CSV = abc_random_mu284_hybrid_v16_fast_auto.csv


### other parts including debugging

In [53]:
# ============================================================
# PATCH: Use Vincent z/pi ordered Pπ as the true reference
# ============================================================

import numpy as np
import pandas as pd


def _ppi_efficiency_for_order(y_ordered, z_ordered, pi_ordered, var_srs_y, var_srs_z):
    """
    Compute exact Pπ variance/efficiency for a given population order.

    This is the Vincent empirical Pπ when the order is z/pi.
    """
    y_ordered = np.asarray(y_ordered, dtype=float)
    z_ordered = np.asarray(z_ordered, dtype=float)
    pi_ordered = np.asarray(pi_ordered, dtype=float)

    V = Ppi(pi_ordered)
    K = V @ V.T

    diag_error = float(np.max(np.abs(np.diag(K) - pi_ordered)))

    A = -np.abs(K) ** 2
    np.fill_diagonal(A, pi_ordered * (1.0 - pi_ordered))

    y_over_pi = y_ordered / pi_ordered
    z_over_pi = z_ordered / pi_ordered

    var_y = float(np.real(y_over_pi @ (A @ y_over_pi)))
    var_z = float(np.real(z_over_pi @ (A @ z_over_pi)))

    eff_y = var_srs_y / var_y if var_y > 0 else np.nan
    eff_z = var_srs_z / var_z if var_z > 0 else np.nan

    return {
        "var_y": var_y,
        "var_z": var_z,
        "eff_y": eff_y,
        "eff_z": eff_z,
        "diag_error": diag_error,
    }


def _calculate_optimal_vincent_reference(self):
    """
    Corrected reference calculation.

    self.input_y, self.input_z, self.input_pi are the order supplied to ABCAlgorithm.
    In our main run, that order is Vincent's z/pi order.

    self.y_sorted, self.z_sorted, self.pik_sorted are the internal descending-pi order
    needed by CaDsd.
    """

    # True Vincent empirical Pπ baseline: input order = z/pi order.
    vin = _ppi_efficiency_for_order(
        self.input_y,
        self.input_z,
        self.input_pi,
        self.var_srs_y,
        self.var_srs_z,
    )

    # Old/internal baseline: descending-pi order used internally by CaDsd.
    internal = _ppi_efficiency_for_order(
        self.y_sorted,
        self.z_sorted,
        self.pik_sorted,
        self.var_srs_y,
        self.var_srs_z,
    )

    # IMPORTANT:
    # From now on, "optimal_eff_z" and "optimal_eff_y" mean Vincent ordered Pπ.
    self.var_opt_y = vin["var_y"]
    self.var_opt_z = vin["var_z"]
    self.eff_y_optimal = vin["eff_y"]
    self.eff_z_optimal = vin["eff_z"]

    self.reference_ppi_order = "Vincent input order, usually z/pi"

    # Store the old/current internal Pπ too, for diagnostics.
    self.var_internal_ppi_y = internal["var_y"]
    self.var_internal_ppi_z = internal["var_z"]
    self.eff_y_internal_ppi = internal["eff_y"]
    self.eff_z_internal_ppi = internal["eff_z"]

    self.ppi_vincent_diag_error = vin["diag_error"]
    self.ppi_internal_diag_error = internal["diag_error"]

    # Check that no inclusion probabilities were changed, only reordered.
    self.same_pi_multiset_check = np.allclose(
        np.sort(self.input_pi),
        np.sort(self.pik_sorted),
        atol=1e-12,
        rtol=1e-12,
    )


# Save original prepare_results only once.
if not hasattr(ABCAlgorithm, "_prepare_results_before_vincent_patch"):
    ABCAlgorithm._prepare_results_before_vincent_patch = ABCAlgorithm._prepare_results


def _prepare_results_with_vincent_reference(self, total_time, colony_size, total_abandoned):
    results = ABCAlgorithm._prepare_results_before_vincent_patch(
        self,
        total_time,
        colony_size,
        total_abandoned,
    )

    results.update({
        "reference_ppi_order": getattr(self, "reference_ppi_order", None),

        "PpiVincent_eff_z": getattr(self, "eff_z_optimal", np.nan),
        "PpiVincent_eff_y": getattr(self, "eff_y_optimal", np.nan),
        "PpiVincent_var_z": getattr(self, "var_opt_z", np.nan),
        "PpiVincent_var_y": getattr(self, "var_opt_y", np.nan),

        "PpiInternal_eff_z": getattr(self, "eff_z_internal_ppi", np.nan),
        "PpiInternal_eff_y": getattr(self, "eff_y_internal_ppi", np.nan),
        "PpiInternal_var_z": getattr(self, "var_internal_ppi_z", np.nan),
        "PpiInternal_var_y": getattr(self, "var_internal_ppi_y", np.nan),

        "vincent_ppi_diag_error": getattr(self, "ppi_vincent_diag_error", np.nan),
        "internal_ppi_diag_error": getattr(self, "ppi_internal_diag_error", np.nan),
        "same_pi_multiset_check": getattr(self, "same_pi_multiset_check", None),

        "ABC_z_over_PpiVincent_z": _ratio_to_ref(self.global_best_eff_z, self.eff_z_optimal),
        "ABC_y_over_PpiVincent_y": _ratio_to_ref(self.global_best_eff_y, self.eff_y_optimal),

        "ABC_z_over_PpiInternal_z": _ratio_to_ref(
            self.global_best_eff_z,
            getattr(self, "eff_z_internal_ppi", np.nan),
        ),
        "ABC_y_over_PpiInternal_y": _ratio_to_ref(
            self.global_best_eff_y,
            getattr(self, "eff_y_internal_ppi", np.nan),
        ),
    })

    return results


ABCAlgorithm._calculate_optimal = _calculate_optimal_vincent_reference
ABCAlgorithm._prepare_results = _prepare_results_with_vincent_reference

# RandomSearchAlgorithm inherits ABCAlgorithm, so it uses the same corrected reference.
print("PATCH APPLIED: Pπ reference is now Vincent ordered Pπ, based on the input z/pi order.")
print("CaDsd/ABC still uses internal descending-π order for candidate kernels.")

PATCH APPLIED: Pπ reference is now Vincent ordered Pπ, based on the input z/pi order.
CaDsd/ABC still uses internal descending-π order for candidate kernels.


In [54]:
# ============================================================
# STRICT PATCH: use the ORIGINAL R Reciprocal_CaDsd to seed ABC
# ============================================================
# Goal:
#   Vincent ordered Pπ kernel -> R Reciprocal_CaDsd -> omega0, rho0
#   Then ABC starts from omega0/rho0 and the other initial nodes are
#   small perturbations around it.
#
# Important:
#   There is NO fake/non-mutable fallback here. If the R reciprocal
#   seed cannot be verified, the run stops.
# ============================================================

import os
import json
import time
import urllib.request
from pathlib import Path
import numpy as np
import pandas as pd

# -----------------------------
# Switches
# -----------------------------
USE_R_RECIPROCAL_VINCENT_SEED = True
REQUIRE_MUTABLE_VINCENT_SEED = True

# If these are too strict on your machine, loosen only a little.
VINCENT_SEED_ROUNDTRIP_TOL = 1e-3
VINCENT_SEED_DIAG_TOL = 1e-3
VINCENT_SEED_RATIO_FLOOR = 0.999

# Make the first population close to Vincent.
VINCENT_INIT_LOCAL_SCALE_OMEGA = 0.003
VINCENT_INIT_LOCAL_SCALE_RHO = 0.003

# Random-search baseline also starts locally around Vincent.
RANDOM_SEARCH_LOCAL_AROUND_VINCENT = True
RANDOM_SEARCH_LOCAL_SCALE_OMEGA = 0.010
RANDOM_SEARCH_LOCAL_SCALE_RHO = 0.010

# Debugging of node efficiencies.
DETAILED_NODE_OUTPUT = True
PRINT_EACH_NODE = False       # True prints every evaluated node; can be very noisy.

# Download/source the original R code.
# We patch ONLY numerical fragility:
#   1) remove the example at the bottom of CaDsd, which prints a matrix when sourced;
#   2) replace solve(X'X) by MASS::ginv(X'X) inside Reciprocal_CaDsd, because
#      exact projection kernels can give singular Gram matrices in the reverse step.
R_DSD_DIR = Path("r_dsd_tools")
R_DSD_PATCHED_DIR = Path("r_dsd_tools_patched")
DOWNLOAD_R_DSD_FILES = True
R_DSD_RAW_URLS = {
    "CaDsd": "https://raw.githubusercontent.com/InseeFrLab/Determinantal-Sampling-Designs/main/CaDsd",
    "Reciprocal_CaDsd": "https://raw.githubusercontent.com/InseeFrLab/Determinantal-Sampling-Designs/main/Reciprocal_CaDsd",
}

R_RECIPROCAL_USE_PSEUDOINVERSE = True
R_RECIPROCAL_RETRIES = 8


def _strip_r_examples(code: str) -> str:
    """Remove example code that is executed when the raw R file is sourced."""
    marker = "#Exemples"
    if marker in code:
        return code.split(marker)[0].rstrip() + "\n"
    return code


def _patch_reciprocal_r_code(code: str) -> str:
    """Make Reciprocal_CaDsd robust to singular Gram matrices."""
    if not R_RECIPROCAL_USE_PSEUDOINVERSE:
        return code
    # The official code uses solve(t(Conj(X))%*%X). For exact projection kernels
    # this matrix can be singular. MASS::ginv gives a stable orthogonal projection.
    code = code.replace(
        "solve(t(Conj(X))%*%X)",
        "MASS::ginv(t(Conj(X))%*%X)"
    )
    return code


def _ensure_r_dsd_files():
    """Download raw R files and create patched source files used by the notebook."""
    R_DSD_DIR.mkdir(exist_ok=True)
    R_DSD_PATCHED_DIR.mkdir(exist_ok=True)

    out_paths = {}
    for name, url in R_DSD_RAW_URLS.items():
        raw_path = R_DSD_DIR / name
        patched_path = R_DSD_PATCHED_DIR / name

        if not (raw_path.exists() and raw_path.stat().st_size > 100):
            if not DOWNLOAD_R_DSD_FILES:
                raise FileNotFoundError(
                    f"Missing {raw_path}. Put the R file there or set DOWNLOAD_R_DSD_FILES=True."
                )
            print(f"Downloading R file: {name}")
            urllib.request.urlretrieve(url, raw_path)

        code = raw_path.read_text(encoding="utf-8")
        if name == "CaDsd":
            code = _strip_r_examples(code)
        if name == "Reciprocal_CaDsd":
            code = _patch_reciprocal_r_code(code)

        patched_path.write_text(code, encoding="utf-8")
        out_paths[name] = patched_path

    return out_paths


def _load_r_reciprocal_backend():
    """Load rpy2 and source the official R functions, with numerical patching."""
    try:
        import rpy2.robjects as ro
        from rpy2.robjects import numpy2ri
        from rpy2.robjects.conversion import localconverter
    except Exception as e:
        raise ImportError(
            "rpy2 is required for the R Reciprocal_CaDsd bridge.\n"
            "Install it in your environment, for example:\n"
            "    uv add rpy2\n"
            "or:\n"
            "    pip install rpy2\n"
            "Also make sure R is installed on Ubuntu."
        ) from e

    paths = _ensure_r_dsd_files()

    # MASS is a recommended R package and is normally installed with R.
    # We use MASS::ginv in the patched reciprocal function.
    ro.r("suppressPackageStartupMessages(library(MASS))")

    for name in ["CaDsd", "Reciprocal_CaDsd"]:
        ro.r(f"source({json.dumps(str(paths[name]))})")

    ro.r("set.seed(12345)")
    print("R backend loaded: patched CaDsd and robust Reciprocal_CaDsd are available.")
    return ro, numpy2ri, localconverter


_R_BACKEND = None


def _get_r_backend():
    global _R_BACKEND
    if _R_BACKEND is None:
        _R_BACKEND = _load_r_reciprocal_backend()
    return _R_BACKEND


def _np_to_r_vector(x):
    """Convert a 1D numpy array to an R numeric vector.

    Important: do NOT send pi as an R matrix. CaDsd uses length(pi);
    sending a strange array object can make R think the population size is wrong.
    """
    ro, numpy2ri, localconverter = _get_r_backend()
    x = np.asarray(x, dtype=float).ravel()
    return ro.FloatVector(x)


def _np_to_r_matrix(x):
    """Convert a 2D numpy array to an R matrix with explicit dimensions.

    This avoids rpy2/numpy dimension loss. R is column-major, so we flatten
    using order='F'. This is essential for omega, rho, and complex kernels.
    """
    ro, numpy2ri, localconverter = _get_r_backend()
    x = np.asarray(x)
    if x.ndim != 2:
        raise ValueError(f"_np_to_r_matrix expects a 2D array, got shape {x.shape}")

    nrow, ncol = x.shape
    flat = np.asarray(x, order="F").ravel(order="F")

    if np.iscomplexobj(x):
        r_vec = ro.ComplexVector([complex(v) for v in flat])
    else:
        r_vec = ro.FloatVector([float(v) for v in flat])

    return ro.r["matrix"](r_vec, nrow=int(nrow), ncol=int(ncol))


def _r_matrix_to_np(r_obj, dtype=np.complex128):
    """Convert an R matrix to a numpy 2D array, preserving R dimensions.

    Without this, rpy2 can return a flat vector; that caused the previous
    shape error: (25,) versus (900,).
    """
    arr = np.asarray(r_obj, dtype=dtype)

    dims = None
    try:
        dims = tuple(int(v) for v in list(r_obj.do_slot("dim")))
    except Exception:
        try:
            dims = tuple(int(v) for v in list(r_obj.dim))
        except Exception:
            dims = None

    if dims is not None and len(dims) == 2:
        nrow, ncol = dims
        if arr.size != nrow * ncol:
            raise ValueError(
                f"R matrix dimension mismatch: dim={dims}, but numpy size={arr.size}"
            )
        return np.asarray(arr, dtype=dtype).reshape((nrow, ncol), order="F")

    # Fallback: if rpy2 already returned a matrix, keep it.
    if arr.ndim == 2:
        return arr

    return arr


def r_reciprocal_cadsd(K, pi=None, M=None):
    """Call robust R Reciprocal_CaDsd(K) and return numpy arrays.

    If pi and M are supplied, also reconstruct the kernel with the official R CaDsd
    for an R-side roundtrip diagnostic.
    """
    ro, numpy2ri, localconverter = _get_r_backend()
    K_np = np.asarray(K, dtype=np.complex128)
    rK = _np_to_r_matrix(K_np)

    last_error = None
    for attempt in range(R_RECIPROCAL_RETRIES):
        try:
            ro.r(f"set.seed({12345 + attempt})")
            r_res = ro.globalenv["Reciprocal_CaDsd"](rK)

            Ksort = _r_matrix_to_np(r_res.rx2("KSort"), dtype=np.complex128)
            omega = _r_matrix_to_np(r_res.rx2("omega"), dtype=float)
            rho = _r_matrix_to_np(r_res.rx2("rho"), dtype=float)
            spectre = _r_matrix_to_np(r_res.rx2("spectre"), dtype=float)

            out = {
                "KSort": Ksort,
                "omega": omega,
                "rho": rho,
                "spectre": spectre,
                "attempt": attempt + 1,
                "R_error": None,
            }

            if pi is not None and M is not None:
                # Explicit vector/matrix conversion avoids the bug where R reconstructed
                # a 5 x 5 kernel instead of the expected N x N kernel.
                r_pi = _np_to_r_vector(np.asarray(pi, dtype=float).ravel())
                r_omega = _np_to_r_matrix(omega)
                r_rho = _np_to_r_matrix(rho)

                r_recon = ro.globalenv["CaDsd"](
                    omega=r_omega,
                    rho=r_rho,
                    M=int(M),
                    pi=r_pi,
                )
                out["K_recon_R"] = _r_matrix_to_np(r_recon.rx2("K"), dtype=np.complex128)

            return out

        except Exception as e:
            last_error = e

    raise RuntimeError(
        "R Reciprocal_CaDsd failed after "
        f"{R_RECIPROCAL_RETRIES} attempts. Last error: {last_error}"
    )



def _ratio_to_ref(value, ref):
    if ref is None or not np.isfinite(ref) or abs(ref) < 1e-14:
        return np.nan
    return value / ref


def _vincent_kernel_in_input_order(self):
    """Vincent ordered Pπ kernel in the input order, which is z/pi order."""
    V = Ppi(self.input_pi)
    return V @ V.T


def _vincent_kernel_in_internal_order(self):
    """Vincent Pπ kernel permuted into the same descending-pi order used by CaDsd."""
    K_input = _vincent_kernel_in_input_order(self)
    return K_input[np.ix_(self.order, self.order)]


def _record_node(self, source, omega, rho, eff_z, eff_y, valid):
    if not DETAILED_NODE_OUTPUT:
        return
    if not hasattr(self, "node_records"):
        self.node_records = []

    omega0 = getattr(self, "vincent_seed_omega", None)
    rho0 = getattr(self, "vincent_seed_rho", None)

    if omega0 is not None and omega is not None:
        d_omega = float(np.linalg.norm(omega - omega0) / np.sqrt(omega.size))
    else:
        d_omega = np.nan
    if rho0 is not None and rho is not None:
        d_rho = float(np.linalg.norm(rho - rho0) / np.sqrt(rho.size))
    else:
        d_rho = np.nan

    rec = {
        "case_name": getattr(self, "case_name", None),
        "eval_count": getattr(self, "eval_count", np.nan),
        "source": source,
        "valid": bool(valid),
        "eff_z": eff_z,
        "eff_y": eff_y,
        "z_over_vincent": _ratio_to_ref(eff_z, getattr(self, "eff_z_optimal", np.nan)),
        "y_over_vincent": _ratio_to_ref(eff_y, getattr(self, "eff_y_optimal", np.nan)),
        "d_omega_from_vincent": d_omega,
        "d_rho_from_vincent": d_rho,
    }
    self.node_records.append(rec)

    if PRINT_EACH_NODE:
        print(
            f"    NODE {source:24s} valid={valid!s:5s} "
            f"z/V={rec['z_over_vincent']:.6f} y/V={rec['y_over_vincent']:.6f} "
            f"dω={d_omega:.3e} dρ={d_rho:.3e}"
        )


def _food_from_arrays_with_source(self, omega, rho, trial=0):
    source = getattr(self, "_current_candidate_source", "ABC_candidate")
    eff_z, eff_y, valid = self.evaluate(omega, rho)
    _record_node(self, source, omega, rho, eff_z, eff_y, valid)
    if not valid:
        return None
    food = {
        "omega": omega,
        "rho": rho,
        "eff_z": eff_z,
        "eff_y": eff_y,
        "score": self._score(eff_z, eff_y),
        "trial": trial,
        "source": source,
    }
    self._update_global_best(food)
    return food


def _update_global_best_with_source(self, food):
    if food["score"] > self.global_best_score:
        self.global_best_score = food["score"]
        self.global_best_eff_z = food["eff_z"]
        self.global_best_eff_y = food["eff_y"]
        self.global_best_omega = food["omega"].copy()
        self.global_best_rho = food["rho"].copy()
        self.global_best_source = food.get("source", "candidate")
        self.best_update_count += 1
        return True
    return False


def _make_local_candidate_around_vincent(self, scale_omega, scale_rho):
    if getattr(self, "vincent_seed_omega", None) is None or getattr(self, "vincent_seed_rho", None) is None:
        raise RuntimeError("Vincent omega/rho seed is not available.")
    omega = self.vincent_seed_omega + self.rng.normal(0.0, scale_omega, self.vincent_seed_omega.shape)
    rho = self.vincent_seed_rho + self.rng.normal(0.0, scale_rho, self.vincent_seed_rho.shape)
    return np.clip(omega, 0.0, 1.0), np.mod(rho, 1.0)


def _prepare_r_vincent_seed(self, verbose=True):
    """Recover omega/rho from Vincent Pπ using official R Reciprocal_CaDsd and verify it."""
    if getattr(self, "r_vincent_seed_ready", False):
        return self.vincent_seed_omega, self.vincent_seed_rho

    if not USE_R_RECIPROCAL_VINCENT_SEED:
        raise RuntimeError("USE_R_RECIPROCAL_VINCENT_SEED=False.")

    K_vincent_internal = _vincent_kernel_in_internal_order(self)
    diag_target = self.pik_sorted

    rec = r_reciprocal_cadsd(K_vincent_internal, pi=self.pik_sorted, M=self.M)
    omega0 = np.asarray(rec["omega"], dtype=float)
    rho0 = np.asarray(rec["rho"], dtype=float)

    if omega0.shape != (self.M, self.N):
        raise RuntimeError(f"R omega has shape {omega0.shape}, expected {(self.M, self.N)}.")
    if rho0.shape != (self.M, self.N - 1):
        raise RuntimeError(f"R rho has shape {rho0.shape}, expected {(self.M, self.N - 1)}.")

    # Verify using BOTH:
    #   - R CaDsd: does the official reciprocal invert the official CaDsd?
    #   - Python CaDsd: does the seed work in the optimizer used below?
    K_round_R = rec.get("K_recon_R", None)
    if K_round_R is not None and np.shape(K_round_R) == np.shape(rec["KSort"]):
        r_roundtrip_error = float(np.max(np.abs(K_round_R - rec["KSort"])))
    else:
        r_roundtrip_error = np.nan
        if verbose:
            print(
                "    WARNING: R roundtrip shape mismatch; "
                f"K_recon_R shape={np.shape(K_round_R)}, KSort shape={np.shape(rec['KSort'])}. "
                "Continuing to Python roundtrip check."
            )

    K_round = CaDsd(pi=self.pik_sorted, M=self.M, omega=omega0, rho=rho0)["K"].astype(np.complex128)
    diag_error = float(np.max(np.abs(np.real(np.diag(K_round)) - diag_target)))
    roundtrip_error = float(np.max(np.abs(K_round - K_vincent_internal)))

    eff_z, eff_y, valid = self.evaluate(omega0, rho0)
    z_ratio = _ratio_to_ref(eff_z, self.eff_z_optimal)
    y_ratio = _ratio_to_ref(eff_y, self.eff_y_optimal)

    self.vincent_seed_omega = omega0
    self.vincent_seed_rho = rho0
    self.vincent_seed_diag_error = diag_error
    self.vincent_seed_roundtrip_error = roundtrip_error
    self.vincent_seed_R_roundtrip_error = r_roundtrip_error
    self.vincent_seed_eff_z = eff_z
    self.vincent_seed_eff_y = eff_y
    self.vincent_seed_z_ratio = z_ratio
    self.vincent_seed_y_ratio = y_ratio

    ok = (
        valid
        and diag_error <= VINCENT_SEED_DIAG_TOL
        and roundtrip_error <= VINCENT_SEED_ROUNDTRIP_TOL
        and z_ratio >= VINCENT_SEED_RATIO_FLOOR
    )

    if verbose:
        print("    R Reciprocal_CaDsd Vincent seed check:")
        print(f"      diag_error      = {diag_error:.3e}")
        print(f"      R roundtrip     = {r_roundtrip_error:.3e}")
        print(f"      Python roundtrip= {roundtrip_error:.3e}")
        print(f"      seed z/Vincent  = {z_ratio:.6f}")
        print(f"      seed y/Vincent  = {y_ratio:.6f}")

    if REQUIRE_MUTABLE_VINCENT_SEED and not ok:
        raise RuntimeError(
            "R Reciprocal_CaDsd mutable Vincent seed could not be verified.\n"
            f"  valid           = {valid}\n"
            f"  diag_error      = {diag_error:.3e}  <= {VINCENT_SEED_DIAG_TOL:.3e}\n"
            f"  R roundtrip     = {r_roundtrip_error:.3e}\n"
            f"  Python roundtrip= {roundtrip_error:.3e}  <= {VINCENT_SEED_ROUNDTRIP_TOL:.3e}\n"
            f"  z_ratio         = {z_ratio:.6f}  >= {VINCENT_SEED_RATIO_FLOOR:.6f}\n"
            "The run is stopped so we do not silently initialize random nodes far below Vincent."
        )

    self.r_vincent_seed_ready = True
    return omega0, rho0


def _initialize_population_from_r_vincent(self, colony_size, verbose=True):
    if verbose:
        print(f" Initializing {colony_size} food sources around R Reciprocal_CaDsd Vincent seed...")

    _prepare_r_vincent_seed(self, verbose=verbose)
    population = []

    # 1) First food source = Vincent itself.
    self._current_candidate_source = "Vincent_R_mutable_seed"
    seed_food = self._food_from_arrays(self.vincent_seed_omega.copy(), self.vincent_seed_rho.copy(), trial=0)
    if seed_food is None:
        raise RuntimeError("The R Vincent mutable seed was invalid when evaluated.")
    population.append(seed_food)

    # 2) Remaining initial food sources = small perturbations around Vincent.
    max_attempts = max(colony_size * 50, 200)
    attempts = 0
    while len(population) < colony_size and attempts < max_attempts:
        omega, rho = _make_local_candidate_around_vincent(
            self,
            VINCENT_INIT_LOCAL_SCALE_OMEGA,
            VINCENT_INIT_LOCAL_SCALE_RHO,
        )
        self._current_candidate_source = "Vincent_local_initial"
        food = self._food_from_arrays(omega, rho, trial=0)
        if food is not None:
            population.append(food)
        attempts += 1

    self._current_candidate_source = "ABC_candidate"

    if len(population) < colony_size:
        raise RuntimeError(
            f"Only {len(population)}/{colony_size} local Vincent initial nodes were valid. "
            "Try slightly larger local scales or inspect the seed."
        )

    if verbose:
        print(f"\n    Initialized {len(population)} food sources (requested {colony_size})")
        print(
            f"    Best initial: eff_z={self.global_best_eff_z:.4f}, "
            f"eff_y={self.global_best_eff_y:.4f}, source={getattr(self, 'global_best_source', None)}"
        )
        print(
            f"    Best initial ratios: z/V={_ratio_to_ref(self.global_best_eff_z, self.eff_z_optimal):.6f}, "
            f"y/V={_ratio_to_ref(self.global_best_eff_y, self.eff_y_optimal):.6f}"
        )

    return population


def _random_search_step_local_vincent(self, n_evals=1, include_center_once=False):
    """Random-search baseline: local random perturbations around the same Vincent seed."""
    start_time = time.time()
    n_evals = int(max(0, n_evals))

    _prepare_r_vincent_seed(self, verbose=False)

    # Ensure its current best is at least the Vincent seed.
    self._current_candidate_source = "Vincent_R_mutable_seed"
    self._food_from_arrays(self.vincent_seed_omega.copy(), self.vincent_seed_rho.copy(), trial=0)

    for _ in range(n_evals):
        if RANDOM_SEARCH_LOCAL_AROUND_VINCENT:
            omega, rho = _make_local_candidate_around_vincent(
                self,
                RANDOM_SEARCH_LOCAL_SCALE_OMEGA,
                RANDOM_SEARCH_LOCAL_SCALE_RHO,
            )
            self._current_candidate_source = "Random_local_Vincent"
        else:
            omega, rho = self._random_candidate()
            self._current_candidate_source = "Random_global"
        self._food_from_arrays(omega, rho, trial=0)

    self._current_candidate_source = "Random_candidate"
    self.history.append(self.global_best_eff_z)
    record = {
        "step_evals": n_evals,
        "best_eff_z": self.global_best_eff_z,
        "best_eff_y": self.global_best_eff_y,
        "best_score": self.global_best_score,
        "eval_count": self.eval_count,
        "valid_count": self.valid_count,
        "strict_check_count": self.strict_check_count,
        "elapsed_step": time.time() - start_time,
    }
    if hasattr(self, "random_history_records"):
        self.random_history_records.append(record)
    return record


# Apply patches.
ABCAlgorithm._update_global_best = _update_global_best_with_source
ABCAlgorithm._food_from_arrays = _food_from_arrays_with_source
ABCAlgorithm.initialize_population = _initialize_population_from_r_vincent
RandomSearchAlgorithm.step = _random_search_step_local_vincent

# Extend result dictionary.
if not hasattr(ABCAlgorithm, "_prepare_results_before_r_reciprocal_seed"):
    ABCAlgorithm._prepare_results_before_r_reciprocal_seed = ABCAlgorithm._prepare_results


def _prepare_results_with_r_reciprocal_seed(self, total_time, colony_size, total_abandoned):
    res = ABCAlgorithm._prepare_results_before_r_reciprocal_seed(
        self,
        total_time,
        colony_size,
        total_abandoned,
    )
    res.update({
        "best_source": getattr(self, "global_best_source", None),
        "use_R_reciprocal_seed": USE_R_RECIPROCAL_VINCENT_SEED,
        "vincent_seed_diag_error": getattr(self, "vincent_seed_diag_error", np.nan),
        "vincent_seed_roundtrip_error": getattr(self, "vincent_seed_roundtrip_error", np.nan),
        "vincent_seed_R_roundtrip_error": getattr(self, "vincent_seed_R_roundtrip_error", np.nan),
        "vincent_seed_z_ratio": getattr(self, "vincent_seed_z_ratio", np.nan),
        "vincent_seed_y_ratio": getattr(self, "vincent_seed_y_ratio", np.nan),
    })
    return res

ABCAlgorithm._prepare_results = _prepare_results_with_r_reciprocal_seed

print("PATCH APPLIED v9: robust R matrix/vector conversion for Reciprocal_CaDsd Vincent seed.")
print("No fake/non-mutable floor is used. If the R seed is not verified, the run stops.")


PATCH APPLIED v9: robust R matrix/vector conversion for Reciprocal_CaDsd Vincent seed.
No fake/non-mutable floor is used. If the R seed is not verified, the run stops.


In [55]:
# ============================================================
# v11 OVERRIDE PATCH: drop-safe official R CaDsd + Reciprocal seed
# ============================================================
# v10 revealed the real R-side problem:
#   length(pi)=30, dim(omega)=5x30, dim(rho)=5x29, but CaDsd returned K=5x5.
# This can happen because the original compact R code uses matrix subsetting
# without drop=FALSE. When a subsetting step has one row/column, R silently
# drops the matrix dimension, and later matrix products can collapse the final
# kernel to M x M instead of N x N.
#
# This cell re-sources drop-safe versions of the official R functions and then
# uses them for the Vincent reciprocal seed and candidate evaluation.
# ============================================================

USE_R_CADSD_FOR_CANDIDATES = True
PRINT_R_SHAPE_DEBUG_ONCE = True
PRINT_R_DROP_SAFE_PATCH_REPORT = True

_R_SHAPE_DEBUG_PRINTED = False
_R_DROP_SAFE_BACKEND_READY = False
_ORIGINAL_ABC_EVALUATE_BEFORE_R_V11 = getattr(ABCAlgorithm, "evaluate", None)


def _r_dim_tuple(r_obj):
    """Return R dim(x) as a Python tuple, or None."""
    ro, numpy2ri, localconverter = _get_r_backend()
    try:
        dims = list(ro.r["dim"](r_obj))
        if len(dims) == 0:
            return None
        return tuple(int(v) for v in dims)
    except Exception:
        return None


def _r_matrix_to_np_strict(r_obj, dtype=np.complex128):
    """Convert an R matrix to NumPy, preserving R's column-major dimensions."""
    dims = _r_dim_tuple(r_obj)
    if dims is not None and len(dims) == 2:
        nrow, ncol = dims
        flat = np.array(list(r_obj), dtype=dtype)
        if flat.size != nrow * ncol:
            raise ValueError(
                f"Cannot convert R matrix: dim={dims}, but flat size={flat.size}."
            )
        return flat.reshape((nrow, ncol), order="F")

    arr = np.asarray(r_obj, dtype=dtype)
    return arr


def _patch_code_exact(code, replacements, label):
    """Apply exact textual replacements and print a small report."""
    applied = []
    missing = []
    for old, new in replacements:
        n = code.count(old)
        if n == 0:
            missing.append(old)
        else:
            code = code.replace(old, new)
            applied.append((old, n))
    if PRINT_R_DROP_SAFE_PATCH_REPORT:
        print(f"    R drop-safe patch report for {label}: applied {len(applied)}, missing {len(missing)}")
        if missing:
            print("      Missing patterns are not always fatal, but if K is still 5x5 we must inspect them.")
    return code


def _patch_cadsd_drop_safe(code: str) -> str:
    """Patch official CaDsd R code against R dimension dropping."""
    code = _strip_r_examples(code)
    replacements = [
        ("phi=round(sqrt(pi_down[1])*U[,1],7)",
         "phi=round(sqrt(pi_down[1])*U[,1,drop=FALSE],7)"),
        ("sigma1=diag(M)[E1_,]", "sigma1=diag(M)[E1_,,drop=FALSE]"),
        ("sigma2=diag(M)[E2_,]", "sigma2=diag(M)[E2_,,drop=FALSE]"),
        ("R=diag(r)[r:1,]%*%cbind(lambda2[E2],lambda1[E1])",
         "R=diag(r)[r:1,,drop=FALSE]%*%cbind(lambda2[E2],lambda1[E1])"),
    ]
    return _patch_code_exact(code, replacements, "CaDsd")


def _patch_reciprocal_drop_safe(code: str) -> str:
    """Patch official Reciprocal_CaDsd R code against singular solve and dimension dropping."""
    code = _patch_reciprocal_r_code(code)  # keeps MASS::ginv patch from v8/v9/v10
    replacements = [
        ("U=matrix(phi[,1]/Pi[1])", "U=matrix(phi[,1,drop=FALSE]/Pi[1], nrow=M, ncol=1)"),
        ("mat=phi[,1:(k-1)]%*%t(Conj(phi[,1:(k-1)]))",
         "mat=phi[,1:(k-1),drop=FALSE]%*%t(Conj(phi[,1:(k-1),drop=FALSE]))"),
        ("P=vp[,prev_pos:(pos-1)]", "P=vp[,prev_pos:(pos-1),drop=FALSE]"),
        ("X=cbind(U_chap[,0:(prev_pos-1)],U[,pos:M])",
         "X=cbind(U_chap[,0:(prev_pos-1),drop=FALSE],U[,pos:M,drop=FALSE])"),
        ("X=U_chap[,0:(prev_pos-1)]",
         "X=U_chap[,0:(prev_pos-1),drop=FALSE]"),
        ("X=cbind(U_chap[,(prev_pos+j-1)],X)",
         "X=cbind(U_chap[,(prev_pos+j-1),drop=FALSE],X)"),
        ("sigma1=diag(M)[E1_,]", "sigma1=diag(M)[E1_,,drop=FALSE]"),
        ("sigma2=diag(M)[E2_,]", "sigma2=diag(M)[E2_,,drop=FALSE]"),
        ("R=diag(r)[r:1,]%*%cbind(lambda2[E2],lambda1[E1])",
         "R=diag(r)[r:1,,drop=FALSE]%*%cbind(lambda2[E2],lambda1[E1])"),
    ]
    return _patch_code_exact(code, replacements, "Reciprocal_CaDsd")


def _source_drop_safe_R_functions():
    """Source drop-safe overrides for CaDsd and Reciprocal_CaDsd into R."""
    global _R_DROP_SAFE_BACKEND_READY
    if _R_DROP_SAFE_BACKEND_READY:
        return

    ro, numpy2ri, localconverter = _get_r_backend()
    paths = _ensure_r_dsd_files()

    # Source the drop-safe functions directly from text. This overrides the
    # already sourced functions in R's global environment.
    cadsd_code = Path(paths["CaDsd"]).read_text(encoding="utf-8")
    recip_code = Path(paths["Reciprocal_CaDsd"]).read_text(encoding="utf-8")
    cadsd_code = _patch_cadsd_drop_safe(cadsd_code)
    recip_code = _patch_reciprocal_drop_safe(recip_code)

    ro.r(cadsd_code)
    ro.r(recip_code)
    ro.r("suppressPackageStartupMessages(library(MASS))")
    ro.r("set.seed(12345)")

    _R_DROP_SAFE_BACKEND_READY = True
    print("R backend loaded: drop-safe CaDsd and robust drop-safe Reciprocal_CaDsd are active.")


def _assign_py_vector_to_R(name, x):
    """Assign a Python vector to R as a numeric vector, not a matrix."""
    ro, numpy2ri, localconverter = _get_r_backend()
    x = np.asarray(x, dtype=float).ravel()
    ro.globalenv[name] = ro.FloatVector([float(v) for v in x])


def _assign_py_matrix_to_R(name, x):
    """Assign a Python 2D array to R as an explicit matrix."""
    ro, numpy2ri, localconverter = _get_r_backend()
    x = np.asarray(x)
    if x.ndim != 2:
        raise ValueError(f"{name}: expected 2D array, got shape {x.shape}.")
    ro.globalenv[name] = _np_to_r_matrix(x)


def r_cadsd_kernel_Rmanaged(pi, M, omega, rho, return_debug=False):
    """Run drop-safe official R CaDsd using R-managed variables and shape checks."""
    _source_drop_safe_R_functions()
    ro, numpy2ri, localconverter = _get_r_backend()

    pi = np.asarray(pi, dtype=float).ravel()
    omega = np.asarray(omega, dtype=float)
    rho = np.asarray(rho, dtype=float)
    M = int(M)
    N = len(pi)

    if omega.shape != (M, N):
        raise ValueError(f"omega shape {omega.shape}, expected {(M, N)}")
    if rho.shape != (M, N - 1):
        raise ValueError(f"rho shape {rho.shape}, expected {(M, N - 1)}")

    _assign_py_vector_to_R("py_pi_v11", pi)
    _assign_py_matrix_to_R("py_omega_v11", omega)
    _assign_py_matrix_to_R("py_rho_v11", rho)
    ro.globalenv["py_M_v11"] = ro.IntVector([M])

    r_ans = ro.r(r'''
        if (length(py_pi_v11) != ncol(py_omega_v11)) {
            stop(sprintf("length(pi)=%d but ncol(omega)=%d", length(py_pi_v11), ncol(py_omega_v11)))
        }
        if (nrow(py_omega_v11) != py_M_v11[1]) {
            stop(sprintf("nrow(omega)=%d but M=%d", nrow(py_omega_v11), py_M_v11[1]))
        }
        if (nrow(py_rho_v11) != py_M_v11[1]) {
            stop(sprintf("nrow(rho)=%d but M=%d", nrow(py_rho_v11), py_M_v11[1]))
        }
        if (ncol(py_rho_v11) != (length(py_pi_v11) - 1)) {
            stop(sprintf("ncol(rho)=%d but length(pi)-1=%d", ncol(py_rho_v11), length(py_pi_v11)-1))
        }
        py_ans_v11 <- CaDsd(omega=py_omega_v11, rho=py_rho_v11, M=py_M_v11[1], pi=py_pi_v11)
        list(
            K = py_ans_v11$K,
            spectrum = py_ans_v11$spectrum,
            EigenBasis = py_ans_v11$EigenBasis,
            debug = list(
                len_pi = length(py_pi_v11),
                dim_omega = dim(py_omega_v11),
                dim_rho = dim(py_rho_v11),
                dim_K = dim(py_ans_v11$K),
                dim_spectrum = dim(py_ans_v11$spectrum),
                dim_EigenBasis = dim(py_ans_v11$EigenBasis),
                sum_pi = sum(py_pi_v11),
                M = py_M_v11[1]
            )
        )
    ''')

    K = _r_matrix_to_np_strict(r_ans.rx2("K"), dtype=np.complex128)
    if K.shape != (N, N):
        dbg = r_ans.rx2("debug")
        raise RuntimeError(
            f"Drop-safe R CaDsd still returned K shape {K.shape}, expected {(N, N)}. "
            f"R debug: len_pi={list(dbg.rx2('len_pi'))}, "
            f"dim_omega={list(dbg.rx2('dim_omega'))}, "
            f"dim_rho={list(dbg.rx2('dim_rho'))}, "
            f"dim_K={list(dbg.rx2('dim_K'))}, "
            f"dim_spectrum={list(dbg.rx2('dim_spectrum'))}, "
            f"dim_EigenBasis={list(dbg.rx2('dim_EigenBasis'))}, "
            f"sum_pi={list(dbg.rx2('sum_pi'))}, M={list(dbg.rx2('M'))}"
        )

    if return_debug:
        return K, r_ans.rx2("debug")
    return K


def r_reciprocal_cadsd_Rmanaged(K):
    """Run drop-safe robust R Reciprocal_CaDsd using an explicit R variable."""
    _source_drop_safe_R_functions()
    ro, numpy2ri, localconverter = _get_r_backend()
    K_np = np.asarray(K, dtype=np.complex128)
    if K_np.ndim != 2 or K_np.shape[0] != K_np.shape[1]:
        raise ValueError(f"K must be a square matrix, got shape {K_np.shape}")

    _assign_py_matrix_to_R("py_K_v11", K_np)

    last_error = None
    for attempt in range(R_RECIPROCAL_RETRIES):
        try:
            ro.r(f"set.seed({12345 + attempt})")
            r_res = ro.r(r'''
                py_rec_v11 <- Reciprocal_CaDsd(py_K_v11)
                list(
                    KSort = py_rec_v11$KSort,
                    omega = py_rec_v11$omega,
                    rho = py_rec_v11$rho,
                    spectre = py_rec_v11$spectre,
                    debug = list(
                        dim_input_K = dim(py_K_v11),
                        dim_KSort = dim(py_rec_v11$KSort),
                        dim_omega = dim(py_rec_v11$omega),
                        dim_rho = dim(py_rec_v11$rho),
                        dim_spectre = dim(py_rec_v11$spectre)
                    )
                )
            ''')

            Ksort = _r_matrix_to_np_strict(r_res.rx2("KSort"), dtype=np.complex128)
            omega = _r_matrix_to_np_strict(r_res.rx2("omega"), dtype=float)
            rho = _r_matrix_to_np_strict(r_res.rx2("rho"), dtype=float)
            spectre = _r_matrix_to_np_strict(r_res.rx2("spectre"), dtype=float)

            return {
                "KSort": Ksort,
                "omega": omega,
                "rho": rho,
                "spectre": spectre,
                "debug": r_res.rx2("debug"),
                "attempt": attempt + 1,
            }
        except Exception as e:
            last_error = e

    raise RuntimeError(
        f"Drop-safe R Reciprocal_CaDsd failed after {R_RECIPROCAL_RETRIES} attempts. Last error: {last_error}"
    )


def _evaluate_with_R_CaDsd_v11(self, omega, rho):
    """Evaluate candidate using drop-safe official R CaDsd kernel."""
    self.eval_count += 1
    try:
        if USE_R_CADSD_FOR_CANDIDATES:
            Kmat = r_cadsd_kernel_Rmanaged(self.pik_sorted, self.M, omega, rho)
        else:
            K_dict = CaDsd(pi=self.pik_sorted, M=self.M, omega=omega, rho=rho)
            Kmat = K_dict["K"].astype(np.complex128, copy=False)
    except Exception as e:
        # Uncomment for very deep debugging of invalid candidates:
        # print("candidate evaluation failed:", repr(e))
        return (0.0, 0.0, False)

    diag_K = np.real(np.diag(Kmat))
    if not self._kernel_passes_validation(Kmat, diag_K):
        return (0.0, 0.0, False)

    var_y, var_z = self._variance_pair_from_kernel(Kmat, diag_K=diag_K)
    if (not np.isfinite(var_y)) or (not np.isfinite(var_z)) or var_y <= 0 or var_z <= 0:
        return (0.0, 0.0, False)

    eff_y = self.var_srs_y / var_y
    eff_z = self.var_srs_z / var_z
    if not (np.isfinite(eff_y) and np.isfinite(eff_z)):
        return (0.0, 0.0, False)

    self.valid_count += 1
    return (eff_z, eff_y, True)


def _prepare_r_vincent_seed_v11(self, verbose=True):
    """Recover omega/rho from Vincent Pπ and verify by drop-safe R CaDsd roundtrip."""
    global _R_SHAPE_DEBUG_PRINTED

    if getattr(self, "r_vincent_seed_ready", False):
        return self.vincent_seed_omega, self.vincent_seed_rho

    if not USE_R_RECIPROCAL_VINCENT_SEED:
        raise RuntimeError("USE_R_RECIPROCAL_VINCENT_SEED=False.")

    K_vincent_internal = _vincent_kernel_in_internal_order(self)
    diag_target = self.pik_sorted

    rec = r_reciprocal_cadsd_Rmanaged(K_vincent_internal)
    Ksort = rec["KSort"]
    omega0 = np.asarray(rec["omega"], dtype=float)
    rho0 = np.asarray(rec["rho"], dtype=float)

    if omega0.shape != (self.M, self.N):
        raise RuntimeError(f"R omega has shape {omega0.shape}, expected {(self.M, self.N)}.")
    if rho0.shape != (self.M, self.N - 1):
        raise RuntimeError(f"R rho has shape {rho0.shape}, expected {(self.M, self.N - 1)}.")

    ksort_error = float(np.max(np.abs(Ksort - K_vincent_internal)))

    K_round_R, r_debug = r_cadsd_kernel_Rmanaged(
        self.pik_sorted, self.M, omega0, rho0, return_debug=True
    )

    if PRINT_R_SHAPE_DEBUG_ONCE and (not _R_SHAPE_DEBUG_PRINTED) and verbose:
        rec_dbg = rec["debug"]
        print("    R shape debug:")
        print(f"      input K to reciprocal = {list(rec_dbg.rx2('dim_input_K'))}")
        print(f"      reciprocal KSort      = {Ksort.shape}")
        print(f"      omega shape           = {omega0.shape}")
        print(f"      rho shape             = {rho0.shape}")
        print(f"      reconstructed K       = {K_round_R.shape}")
        print(f"      length(pi) in R       = {list(r_debug.rx2('len_pi'))}")
        print(f"      dim(omega) in R       = {list(r_debug.rx2('dim_omega'))}")
        print(f"      dim(rho) in R         = {list(r_debug.rx2('dim_rho'))}")
        print(f"      dim(K) in R           = {list(r_debug.rx2('dim_K'))}")
        print(f"      dim(spectrum) in R    = {list(r_debug.rx2('dim_spectrum'))}")
        print(f"      dim(EigenBasis) in R  = {list(r_debug.rx2('dim_EigenBasis'))}")
        _R_SHAPE_DEBUG_PRINTED = True

    diag_error = float(np.max(np.abs(np.real(np.diag(K_round_R)) - diag_target)))
    roundtrip_error = float(np.max(np.abs(K_round_R - K_vincent_internal)))

    eff_z, eff_y, valid = self.evaluate(omega0, rho0)
    z_ratio = _ratio_to_ref(eff_z, self.eff_z_optimal)
    y_ratio = _ratio_to_ref(eff_y, self.eff_y_optimal)

    self.vincent_seed_omega = omega0
    self.vincent_seed_rho = rho0
    self.vincent_seed_ksort_error = ksort_error
    self.vincent_seed_diag_error = diag_error
    self.vincent_seed_roundtrip_error = roundtrip_error
    self.vincent_seed_R_roundtrip_error = roundtrip_error
    self.vincent_seed_eff_z = eff_z
    self.vincent_seed_eff_y = eff_y
    self.vincent_seed_z_ratio = z_ratio
    self.vincent_seed_y_ratio = y_ratio

    ok = (
        valid
        and ksort_error <= VINCENT_SEED_ROUNDTRIP_TOL
        and diag_error <= VINCENT_SEED_DIAG_TOL
        and roundtrip_error <= VINCENT_SEED_ROUNDTRIP_TOL
        and z_ratio >= VINCENT_SEED_RATIO_FLOOR
    )

    if verbose:
        print("    Drop-safe R Reciprocal_CaDsd Vincent seed check:")
        print(f"      KSort input error = {ksort_error:.3e}")
        print(f"      diag_error        = {diag_error:.3e}")
        print(f"      R roundtrip       = {roundtrip_error:.3e}")
        print(f"      seed z/Vincent    = {z_ratio:.6f}")
        print(f"      seed y/Vincent    = {y_ratio:.6f}")

    if REQUIRE_MUTABLE_VINCENT_SEED and not ok:
        raise RuntimeError(
            "Drop-safe R mutable Vincent seed could not be verified.\n"
            f"  valid           = {valid}\n"
            f"  KSort error     = {ksort_error:.3e}  <= {VINCENT_SEED_ROUNDTRIP_TOL:.3e}\n"
            f"  diag_error      = {diag_error:.3e}  <= {VINCENT_SEED_DIAG_TOL:.3e}\n"
            f"  R roundtrip     = {roundtrip_error:.3e}  <= {VINCENT_SEED_ROUNDTRIP_TOL:.3e}\n"
            f"  z_ratio         = {z_ratio:.6f}  >= {VINCENT_SEED_RATIO_FLOOR:.6f}\n"
            "The run is stopped so we do not silently initialize random nodes far below Vincent."
        )

    self.r_vincent_seed_ready = True
    return omega0, rho0


# Apply v11 overrides. The initialization/random-search functions from v9/v10 call
# _prepare_r_vincent_seed by name, so we replace that global name here.
r_reciprocal_cadsd = r_reciprocal_cadsd_Rmanaged
_prepare_r_vincent_seed = _prepare_r_vincent_seed_v11
ABCAlgorithm.evaluate = _evaluate_with_R_CaDsd_v11

print("PATCH APPLIED v11: drop-safe R Reciprocal_CaDsd + drop-safe R CaDsd candidate evaluation.")
print("This specifically fixes the R dimension-dropping problem that produced K shape 5x5 for N=30.")


PATCH APPLIED v11: drop-safe R Reciprocal_CaDsd + drop-safe R CaDsd candidate evaluation.
This specifically fixes the R dimension-dropping problem that produced K shape 5x5 for N=30.


In [56]:
# ============================================================
# v12 OVERRIDE PATCH: STOP omega/rho reciprocal; use direct Vincent-K rotations
# ============================================================
# Why this patch exists:
#   The R Reciprocal_CaDsd path repeatedly returned a CaDsd reconstruction with
#   K shape M x M instead of N x N for the Vincent Pπ kernel. That means the
#   recovered omega/rho cannot be trusted as a mutable Vincent seed in this code.
#
# This patch implements the scientifically safer route:
#   Vincent ordered Pπ kernel K0
#        -> first food source is exactly K0
#        -> other food sources are small unitary pair rotations of K0
#
# Each pair rotation preserves:
#   1. the diagonal, hence first-order inclusion probabilities π,
#   2. the eigenvalues, hence fixed sample size/rank,
#   3. Hermitian projection structure, up to numerical tolerance.
#
# Therefore the first nodes should be very close to 1 relative to Vincent.
# ============================================================

USE_DIRECT_VINCENT_K_ROTATION = True

# Locality switches. Smaller values mean nodes are closer to Vincent.
DIRECT_INIT_THETA_SCALE = 0.006       # first 20 nodes: very close to Vincent
DIRECT_MUTATION_THETA_SCALE = 0.010   # ABC mutation scale near current food
DIRECT_RANDOM_THETA_SCALE = 0.012     # random baseline around Vincent/best
DIRECT_LOCAL_SEARCH_THETA_SCALE = 0.006
DIRECT_ROTATION_STEPS_INIT = 1
DIRECT_ROTATION_STEPS_MUTATION = 1
DIRECT_ROTATION_STEPS_SCOUT = 2
DIRECT_ROTATION_MAX_TRIES = 300
DIRECT_DIAG_TOL = 5e-8
DIRECT_EIG_TOL = 5e-7
PRINT_DIRECT_ROTATION_SETUP = True


def _symmetrize_K(K):
    K = np.asarray(K, dtype=np.complex128)
    return 0.5 * (K + K.conj().T)


def _vincent_kernel_input_order(self):
    """Vincent Pπ kernel in the input order, which is z/pi order in the main run."""
    if not hasattr(self, "vincent_K_input"):
        V = Ppi(self.input_pi)
        self.vincent_K_input = _symmetrize_K(V @ V.T)
    return self.vincent_K_input


def _vincent_kernel_internal_order(self):
    """Same Vincent Pπ kernel, permuted to ABC's internal descending-pi order."""
    if not hasattr(self, "vincent_K_internal"):
        K_in = _vincent_kernel_input_order(self)
        idx = np.asarray(self.order, dtype=int)
        self.vincent_K_internal = _symmetrize_K(K_in[np.ix_(idx, idx)])
    return self.vincent_K_internal


def _apply_diag_preserving_pair_rotation(K, i, j, theta, branch=0):
    """
    Apply a 2x2 unitary rotation on rows/columns i,j.

    The phase is chosen so that the two affected diagonal entries are preserved.
    K' = U K U^*, hence eigenvalues are also preserved.
    """
    K = np.asarray(K, dtype=np.complex128)
    a = float(np.real(K[i, i]))
    b = float(np.real(K[j, j]))
    x = K[i, j]
    absx = float(np.abs(x))
    diff = b - a

    # If there is no coupling and unequal diagonal entries, only theta=0 preserves the diagonal.
    if absx < 1e-14 and abs(diff) > 1e-14:
        return None

    # Feasibility: |tan(theta) * (b-a) / 2| <= |K_ij|.
    if abs(diff) > 1e-14 and absx > 0:
        max_theta = np.arctan(2.0 * absx / abs(diff)) * 0.95
        theta = float(np.clip(theta, -max_theta, max_theta))

    if abs(theta) < 1e-14:
        return None

    c = float(np.cos(theta))
    s = float(np.sin(theta))

    if absx < 1e-14:
        psi = 0.0
    else:
        target = np.tan(theta) * diff / 2.0
        val = float(np.clip(target / absx, -1.0, 1.0))
        alpha = float(np.angle(x))
        acos_val = float(np.arccos(val))
        # Either branch preserves the diagonal; branch adds diversity.
        psi = (acos_val - alpha) if (branch % 2 == 0) else (-acos_val - alpha)

    U = np.eye(K.shape[0], dtype=np.complex128)
    epos = np.exp(1j * psi)
    eneg = np.exp(-1j * psi)
    U[i, i] = c
    U[j, j] = c
    U[i, j] = -s * eneg
    U[j, i] = s * epos

    K_new = U @ K @ U.conj().T
    return _symmetrize_K(K_new)


def _local_rotate_K(self, K, theta_scale=0.006, n_steps=1, source="direct_rotation"):
    """Generate a local candidate by one or more diagonal-preserving pair rotations."""
    K_new = _symmetrize_K(K)
    N = K_new.shape[0]
    accepted = 0
    last_theta = np.nan
    last_pair = None

    for _ in range(int(max(1, n_steps))):
        done = False
        for _try in range(DIRECT_ROTATION_MAX_TRIES):
            i, j = self.rng.choice(N, size=2, replace=False)
            if i > j:
                i, j = j, i
            if np.abs(K_new[i, j]) < 1e-12:
                continue

            theta = float(self.rng.normal(0.0, theta_scale))
            if abs(theta) < 1e-10:
                continue

            cand = _apply_diag_preserving_pair_rotation(
                K_new, i, j, theta, branch=int(self.rng.integers(0, 2))
            )
            if cand is None:
                continue

            diag_err = float(np.max(np.abs(np.real(np.diag(cand)) - self.pik_sorted)))
            if np.isfinite(diag_err) and diag_err <= DIRECT_DIAG_TOL:
                K_new = cand
                accepted += 1
                last_theta = theta
                last_pair = (int(i), int(j))
                done = True
                break
        if not done:
            break

    return K_new, {
        "accepted_rotations": accepted,
        "last_theta": last_theta,
        "last_pair": last_pair,
        "source": source,
    }


def _record_direct_K_node(self, source, K, eff_z, eff_y, valid, meta=None):
    if not DETAILED_NODE_OUTPUT:
        return
    if not hasattr(self, "node_records"):
        self.node_records = []

    K0 = getattr(self, "vincent_seed_K", None)
    if K0 is not None and K is not None:
        d_K = float(np.linalg.norm(K - K0) / np.sqrt(K.size))
    else:
        d_K = np.nan

    diag_error = np.nan
    if K is not None:
        diag_error = float(np.max(np.abs(np.real(np.diag(K)) - self.pik_sorted)))

    meta = meta or {}
    rec = {
        "case_name": getattr(self, "case_name", None),
        "eval_count": getattr(self, "eval_count", np.nan),
        "source": source,
        "valid": bool(valid),
        "eff_z": eff_z,
        "eff_y": eff_y,
        "z_over_vincent": _ratio_to_ref(eff_z, getattr(self, "eff_z_optimal", np.nan)),
        "y_over_vincent": _ratio_to_ref(eff_y, getattr(self, "eff_y_optimal", np.nan)),
        "d_K_from_vincent": d_K,
        "diag_error": diag_error,
        "accepted_rotations": meta.get("accepted_rotations", np.nan),
        "last_theta": meta.get("last_theta", np.nan),
        "last_pair": str(meta.get("last_pair", None)),
        # keep old columns so your display code still works
        "d_omega_from_vincent": np.nan,
        "d_rho_from_vincent": np.nan,
    }
    self.node_records.append(rec)

    if globals().get("PRINT_EACH_NODE", False):
        print(
            f"    NODE {source:24s} valid={valid!s:5s} "
            f"z/V={rec['z_over_vincent']:.6f} y/V={rec['y_over_vincent']:.6f} "
            f"dK={d_K:.3e} diag={diag_error:.1e} theta={rec['last_theta']:.3e}"
        )


def _food_from_K_direct(self, K, source="direct_K_candidate", trial=0, meta=None):
    self.eval_count += 1
    K = _symmetrize_K(K)
    diag_K = np.real(np.diag(K))

    valid = True
    if not np.all(np.isfinite(K)):
        valid = False
    if not np.allclose(diag_K, self.pik_sorted, atol=DIRECT_DIAG_TOL, rtol=0):
        valid = False

    if valid and self._needs_strict_kernel_check():
        self.strict_check_count += 1
        try:
            eigvals = np.linalg.eigvalsh(K)
            if not (np.all(eigvals >= -DIRECT_EIG_TOL) and np.all(eigvals <= 1.0 + DIRECT_EIG_TOL)):
                valid = False
            if not np.isclose(eigvals.sum(), self.n, atol=1e-5):
                valid = False
        except Exception:
            valid = False

    if not valid:
        _record_direct_K_node(self, source, K, 0.0, 0.0, False, meta=meta)
        return None

    var_y, var_z = self._variance_pair_from_kernel(K, diag_K=diag_K)
    eff_y = self.var_srs_y / var_y if var_y > 0 else 0.0
    eff_z = self.var_srs_z / var_z if var_z > 0 else 0.0

    _record_direct_K_node(self, source, K, eff_z, eff_y, True, meta=meta)

    food = {
        "K": K,
        # dummy keys keep old result/display code from breaking; they are not used for mutation
        "omega": np.zeros((self.M, self.N)),
        "rho": np.zeros((self.M, self.N - 1)),
        "eff_z": eff_z,
        "eff_y": eff_y,
        "score": self._score(eff_z, eff_y),
        "trial": trial,
        "source": source,
    }
    self._update_global_best(food)
    return food


def _update_global_best_direct_K(self, food):
    if food["score"] > self.global_best_score:
        self.global_best_score = food["score"]
        self.global_best_eff_z = food["eff_z"]
        self.global_best_eff_y = food["eff_y"]
        self.global_best_K = food["K"].copy()
        # dummy flags so inherited local_search/scout logic knows a best exists
        self.global_best_omega = np.zeros((self.M, self.N))
        self.global_best_rho = np.zeros((self.M, self.N - 1))
        self.global_best_source = food.get("source", "direct_K_candidate")
        self.best_update_count += 1
        return True
    return False


def _food_from_arrays_direct_K(self, omega, rho, trial=0):
    # In v12, omega is actually a candidate K when rho is None.
    if rho is None:
        source = getattr(self, "_current_candidate_source", "direct_K_candidate")
        meta = getattr(self, "_current_candidate_meta", None)
        return _food_from_K_direct(self, omega, source=source, trial=trial, meta=meta)

    # Do not silently fall back to CaDsd in this notebook.
    raise RuntimeError(
        "v12 direct-K mode is active: candidate must be a kernel K with rho=None, "
        "not omega/rho arrays."
    )


def _initialize_population_direct_K(self, colony_size, verbose=True):
    if verbose:
        print(f" Initializing {colony_size} food sources by direct local rotations of Vincent K...")
        print("    No omega/rho reciprocal is used in v12.")

    self.vincent_seed_K = _vincent_kernel_internal_order(self).copy()
    population = []

    # 1) Exact Vincent seed: this guarantees best-of-bests starts at ratio 1.
    self._current_candidate_source = "Vincent_K_seed"
    self._current_candidate_meta = {"accepted_rotations": 0, "last_theta": 0.0, "last_pair": None}
    seed_food = _food_from_K_direct(
        self,
        self.vincent_seed_K,
        source="Vincent_K_seed",
        trial=0,
        meta=self._current_candidate_meta,
    )
    if seed_food is None:
        raise RuntimeError("Vincent K seed failed direct validation. This should not happen.")
    population.append(seed_food)

    # 2) Remaining initial food sources: small rotations around Vincent.
    attempts = 0
    while len(population) < colony_size and attempts < colony_size * 80:
        attempts += 1
        K_new, meta = _local_rotate_K(
            self,
            self.vincent_seed_K,
            theta_scale=DIRECT_INIT_THETA_SCALE,
            n_steps=DIRECT_ROTATION_STEPS_INIT,
            source="Vincent_K_local_init",
        )
        self._current_candidate_source = "Vincent_K_local_init"
        self._current_candidate_meta = meta
        food = _food_from_K_direct(self, K_new, source="Vincent_K_local_init", trial=0, meta=meta)
        if food is not None:
            population.append(food)

    if verbose:
        print(f"    Initialized {len(population)} food sources (requested {colony_size})")
        print(
            f"    Best initial: eff_z={self.global_best_eff_z:.4f}, "
            f"z/V={_ratio_to_ref(self.global_best_eff_z, self.eff_z_optimal):.6f}, "
            f"source={getattr(self, 'global_best_source', None)}"
        )

    if len(population) == 0:
        raise RuntimeError("Direct-K initialization produced no valid food sources.")
    return population


def _mutate_food_direct_K(self, food, partner, progress, mode="abc"):
    # Main mutation around the current food, with a weak pull toward global best.
    base = food.get("K", self.vincent_seed_K)
    scale = DIRECT_MUTATION_THETA_SCALE * max(0.20, 1.0 - 0.60 * float(progress))
    K_new, meta = _local_rotate_K(
        self,
        base,
        theta_scale=scale,
        n_steps=DIRECT_ROTATION_STEPS_MUTATION,
        source="ABC_K_candidate",
    )
    self._current_candidate_meta = meta
    return K_new, None


def _candidate_near_best_direct_K(self, scale=0.05):
    base = getattr(self, "global_best_K", None)
    if base is None:
        base = getattr(self, "vincent_seed_K", None)
    if base is None:
        base = _vincent_kernel_internal_order(self).copy()
        self.vincent_seed_K = base.copy()
    theta_scale = min(max(float(scale), 1e-4), DIRECT_RANDOM_THETA_SCALE)
    K_new, meta = _local_rotate_K(
        self,
        base,
        theta_scale=theta_scale,
        n_steps=DIRECT_ROTATION_STEPS_SCOUT,
        source="Direct_K_near_best",
    )
    self._current_candidate_meta = meta
    return K_new, None


def _random_candidate_direct_K(self):
    base = getattr(self, "vincent_seed_K", None)
    if base is None:
        base = _vincent_kernel_internal_order(self).copy()
        self.vincent_seed_K = base.copy()
    K_new, meta = _local_rotate_K(
        self,
        base,
        theta_scale=DIRECT_RANDOM_THETA_SCALE,
        n_steps=DIRECT_ROTATION_STEPS_SCOUT,
        source="Direct_K_random_local",
    )
    self._current_candidate_meta = meta
    return K_new, None


def _greedy_replace_direct_K(self, old_food, omega, rho):
    source = getattr(self, "_current_candidate_source", "ABC_K_candidate")
    meta = getattr(self, "_current_candidate_meta", None)
    new_food = _food_from_K_direct(self, omega, source=source, trial=0, meta=meta)
    if new_food is not None and new_food["score"] > old_food["score"]:
        return new_food
    old_food["trial"] += 1
    return old_food


def _inject_elite_direct_K(self, population):
    if len(population) == 0 or getattr(self, "global_best_K", None) is None:
        return population
    scores = np.array([f["score"] for f in population], dtype=float)
    if np.max(scores) + 1e-15 < self.global_best_score:
        worst = int(np.argmin(scores))
        population[worst] = {
            "K": self.global_best_K.copy(),
            "omega": np.zeros((self.M, self.N)),
            "rho": np.zeros((self.M, self.N - 1)),
            "eff_z": self.global_best_eff_z,
            "eff_y": self.global_best_eff_y,
            "score": self.global_best_score,
            "trial": 0,
            "source": getattr(self, "global_best_source", "elite_K"),
        }
    return population


def _random_step_direct_K(self, n_evals=1, include_center_once=False):
    # In direct-K mode, random search is a local random baseline around Vincent/best.
    if getattr(self, "vincent_seed_K", None) is None:
        self.vincent_seed_K = _vincent_kernel_internal_order(self).copy()
        # install Vincent floor for random search too
        _food_from_K_direct(self, self.vincent_seed_K, source="Vincent_K_seed_random", trial=0,
                            meta={"accepted_rotations": 0, "last_theta": 0.0, "last_pair": None})

    for _ in range(int(n_evals)):
        base = getattr(self, "global_best_K", self.vincent_seed_K)
        K_new, meta = _local_rotate_K(
            self,
            base,
            theta_scale=DIRECT_RANDOM_THETA_SCALE,
            n_steps=DIRECT_ROTATION_STEPS_SCOUT,
            source="Random_K_local",
        )
        self._current_candidate_source = "Random_K_local"
        self._current_candidate_meta = meta
        _food_from_K_direct(self, K_new, source="Random_K_local", trial=0, meta=meta)

    self.history.append(self.global_best_eff_z)
    return {
        "best_eff_z": self.global_best_eff_z,
        "best_eff_y": self.global_best_eff_y,
        "best_score": self.global_best_score,
        "eval_count": self.eval_count,
        "valid_count": self.valid_count,
    }


def _prepare_results_direct_K(self, total_time, colony_size, total_abandoned):
    # Start from the Vincent-reference patched result if available.
    base_fn = getattr(ABCAlgorithm, "_prepare_results_before_direct_K_patch", None)
    if base_fn is None:
        base_fn = ABCAlgorithm._prepare_results_before_vincent_patch
    results = base_fn(self, total_time, colony_size, total_abandoned)
    results.update({
        "best_K": getattr(self, "global_best_K", None),
        "best_source": getattr(self, "global_best_source", None),
        "direct_K_rotation_mode": True,
        "PpiVincent_eff_z": getattr(self, "eff_z_optimal", np.nan),
        "PpiVincent_eff_y": getattr(self, "eff_y_optimal", np.nan),
        "PpiInternal_eff_z": getattr(self, "eff_z_internal_ppi", np.nan),
        "PpiInternal_eff_y": getattr(self, "eff_y_internal_ppi", np.nan),
        "vincent_ppi_diag_error": getattr(self, "ppi_vincent_diag_error", np.nan),
        "internal_ppi_diag_error": getattr(self, "ppi_internal_diag_error", np.nan),
        "same_pi_multiset_check": getattr(self, "same_pi_multiset_check", None),
        "ABC_z_over_PpiVincent_z": _ratio_to_ref(self.global_best_eff_z, self.eff_z_optimal),
        "ABC_y_over_PpiVincent_y": _ratio_to_ref(self.global_best_eff_y, self.eff_y_optimal),
        "ABC_z_over_PpiInternal_z": _ratio_to_ref(
            self.global_best_eff_z, getattr(self, "eff_z_internal_ppi", np.nan)
        ),
        "ABC_y_over_PpiInternal_y": _ratio_to_ref(
            self.global_best_eff_y, getattr(self, "eff_y_internal_ppi", np.nan)
        ),
    })
    return results


# Install v12 overrides.
if not hasattr(ABCAlgorithm, "_prepare_results_before_direct_K_patch"):
    ABCAlgorithm._prepare_results_before_direct_K_patch = ABCAlgorithm._prepare_results

ABCAlgorithm._update_global_best = _update_global_best_direct_K
ABCAlgorithm._food_from_arrays = _food_from_arrays_direct_K
ABCAlgorithm.initialize_population = _initialize_population_direct_K
ABCAlgorithm._mutate_food = _mutate_food_direct_K
ABCAlgorithm._candidate_near_best = _candidate_near_best_direct_K
ABCAlgorithm._random_candidate = _random_candidate_direct_K
ABCAlgorithm._greedy_replace = _greedy_replace_direct_K
ABCAlgorithm._inject_elite = _inject_elite_direct_K
ABCAlgorithm._prepare_results = _prepare_results_direct_K
RandomSearchAlgorithm.step = _random_step_direct_K

if PRINT_DIRECT_ROTATION_SETUP:
    print("v12 direct-K rotation mode installed.")
    print("  Reciprocal omega/rho is NOT used.")
    print("  First food source = exact Vincent K.")
    print("  Other food sources = small diagonal-preserving unitary rotations of Vincent K.")


v12 direct-K rotation mode installed.
  Reciprocal omega/rho is NOT used.
  First food source = exact Vincent K.
  Other food sources = small diagonal-preserving unitary rotations of Vincent K.


In [57]:
# ============================================================
# v13 EXPLORATION PATCH: direct Vincent-K rotations with real search pressure
# ============================================================
# v12 was intentionally very conservative. It proved the algorithm starts from
# Vincent, but the rotations were so tiny that the run finished fast and only
# found microscopic improvements.
#
# v13 keeps the correct logic:
#   first food source = exact Vincent K
#   all candidates = diagonal-preserving local rotations of K
#   best-of-bests can never fall below Vincent
#
# but makes the search more useful:
#   1. multi-scale rotation steps,
#   2. more rotations per candidate,
#   3. valid counters fixed,
#   4. progress printed with 8 decimals,
#   5. node trace records d_K, theta, pair, accepted rotations.
# ============================================================

DIRECT_STEP_PROFILE = "explore"   # options: "near", "balanced", "explore", "aggressive"
PRINT_DIRECT_BATCH_SUMMARY = True
DIRECT_BATCH_LAST_N = 20


def _set_direct_profile(profile="explore"):
    global DIRECT_INIT_THETA_SCALE, DIRECT_MUTATION_THETA_SCALE, DIRECT_RANDOM_THETA_SCALE
    global DIRECT_LOCAL_SEARCH_THETA_SCALE
    global DIRECT_ROTATION_STEPS_INIT, DIRECT_ROTATION_STEPS_MUTATION, DIRECT_ROTATION_STEPS_SCOUT

    profile = str(profile).lower()

    if profile == "near":
        DIRECT_INIT_THETA_SCALE = 0.006
        DIRECT_MUTATION_THETA_SCALE = 0.010
        DIRECT_RANDOM_THETA_SCALE = 0.012
        DIRECT_LOCAL_SEARCH_THETA_SCALE = 0.008
        DIRECT_ROTATION_STEPS_INIT = 1
        DIRECT_ROTATION_STEPS_MUTATION = 1
        DIRECT_ROTATION_STEPS_SCOUT = 2

    elif profile == "balanced":
        DIRECT_INIT_THETA_SCALE = 0.015
        DIRECT_MUTATION_THETA_SCALE = 0.035
        DIRECT_RANDOM_THETA_SCALE = 0.045
        DIRECT_LOCAL_SEARCH_THETA_SCALE = 0.025
        DIRECT_ROTATION_STEPS_INIT = 2
        DIRECT_ROTATION_STEPS_MUTATION = 2
        DIRECT_ROTATION_STEPS_SCOUT = 3

    elif profile == "explore":
        DIRECT_INIT_THETA_SCALE = 0.030
        DIRECT_MUTATION_THETA_SCALE = 0.080
        DIRECT_RANDOM_THETA_SCALE = 0.100
        DIRECT_LOCAL_SEARCH_THETA_SCALE = 0.050
        DIRECT_ROTATION_STEPS_INIT = 3
        DIRECT_ROTATION_STEPS_MUTATION = 4
        DIRECT_ROTATION_STEPS_SCOUT = 5

    elif profile == "aggressive":
        DIRECT_INIT_THETA_SCALE = 0.060
        DIRECT_MUTATION_THETA_SCALE = 0.150
        DIRECT_RANDOM_THETA_SCALE = 0.180
        DIRECT_LOCAL_SEARCH_THETA_SCALE = 0.090
        DIRECT_ROTATION_STEPS_INIT = 5
        DIRECT_ROTATION_STEPS_MUTATION = 7
        DIRECT_ROTATION_STEPS_SCOUT = 8

    else:
        raise ValueError("Unknown DIRECT_STEP_PROFILE. Use near, balanced, explore, or aggressive.")


_set_direct_profile(DIRECT_STEP_PROFILE)


def _food_from_K_direct_v13(self, K, source="direct_K_candidate", trial=0, meta=None):
    self.eval_count += 1
    K = _symmetrize_K(K)
    diag_K = np.real(np.diag(K))

    valid = True
    if not np.all(np.isfinite(K)):
        valid = False
    if not np.allclose(diag_K, self.pik_sorted, atol=DIRECT_DIAG_TOL, rtol=0):
        valid = False

    if valid and self._needs_strict_kernel_check():
        self.strict_check_count += 1
        try:
            eigvals = np.linalg.eigvalsh(K)
            if not (np.all(eigvals >= -DIRECT_EIG_TOL) and np.all(eigvals <= 1.0 + DIRECT_EIG_TOL)):
                valid = False
            if not np.isclose(eigvals.sum(), self.n, atol=1e-5):
                valid = False
        except Exception:
            valid = False

    if not valid:
        _record_direct_K_node(self, source, K, 0.0, 0.0, False, meta=meta)
        return None

    self.valid_count += 1

    var_y, var_z = self._variance_pair_from_kernel(K, diag_K=diag_K)
    eff_y = self.var_srs_y / var_y if var_y > 0 else 0.0
    eff_z = self.var_srs_z / var_z if var_z > 0 else 0.0

    _record_direct_K_node(self, source, K, eff_z, eff_y, True, meta=meta)

    food = {
        "K": K,
        "omega": np.zeros((self.M, self.N)),
        "rho": np.zeros((self.M, self.N - 1)),
        "eff_z": eff_z,
        "eff_y": eff_y,
        "score": self._score(eff_z, eff_y),
        "trial": trial,
        "source": source,
    }
    self._update_global_best(food)
    return food


# Important: replace the global function name used inside the v12 helpers.
_food_from_K_direct = _food_from_K_direct_v13


def _initialize_population_direct_K_v13(self, colony_size, verbose=True):
    if verbose:
        print(f" Initializing {colony_size} food sources by direct local rotations of Vincent K...")
        print("    No omega/rho reciprocal is used.")
        print(f"    v13 profile={DIRECT_STEP_PROFILE}: init_theta={DIRECT_INIT_THETA_SCALE}, "
              f"mutation_theta={DIRECT_MUTATION_THETA_SCALE}, random_theta={DIRECT_RANDOM_THETA_SCALE}")
        print(f"    rotations per candidate: init={DIRECT_ROTATION_STEPS_INIT}, "
              f"mutation={DIRECT_ROTATION_STEPS_MUTATION}, scout/random={DIRECT_ROTATION_STEPS_SCOUT}")

    self.vincent_seed_K = _vincent_kernel_internal_order(self).copy()
    population = []

    self._current_candidate_source = "Vincent_K_seed"
    self._current_candidate_meta = {"accepted_rotations": 0, "last_theta": 0.0, "last_pair": None}
    seed_food = _food_from_K_direct(
        self,
        self.vincent_seed_K,
        source="Vincent_K_seed",
        trial=0,
        meta=self._current_candidate_meta,
    )
    if seed_food is None:
        raise RuntimeError("Vincent K seed failed direct validation. This should not happen.")
    population.append(seed_food)

    attempts = 0
    while len(population) < colony_size and attempts < colony_size * 120:
        attempts += 1
        K_new, meta = _local_rotate_K(
            self,
            self.vincent_seed_K,
            theta_scale=DIRECT_INIT_THETA_SCALE,
            n_steps=DIRECT_ROTATION_STEPS_INIT,
            source="Vincent_K_local_init",
        )
        self._current_candidate_source = "Vincent_K_local_init"
        self._current_candidate_meta = meta
        food = _food_from_K_direct(self, K_new, source="Vincent_K_local_init", trial=0, meta=meta)
        if food is not None:
            population.append(food)

    if verbose:
        print(f"    Initialized {len(population)} food sources (requested {colony_size})")
        print(
            f"    Best initial: eff_z={self.global_best_eff_z:.8f}, "
            f"z/V={_ratio_to_ref(self.global_best_eff_z, self.eff_z_optimal):.8f}, "
            f"source={getattr(self, 'global_best_source', None)}"
        )

    if len(population) == 0:
        raise RuntimeError("Direct-K initialization produced no valid food sources.")
    return population


def _mutate_food_direct_K_v13(self, food, partner, progress, mode="abc"):
    # Search around the current food. As progress grows, reduce scale but do not freeze.
    base = food.get("K", self.vincent_seed_K)
    scale = DIRECT_MUTATION_THETA_SCALE * max(0.35, 1.0 - 0.50 * float(progress))
    K_new, meta = _local_rotate_K(
        self,
        base,
        theta_scale=scale,
        n_steps=DIRECT_ROTATION_STEPS_MUTATION,
        source="ABC_K_candidate",
    )
    meta["theta_scale_used"] = scale
    self._current_candidate_source = "ABC_K_candidate"
    self._current_candidate_meta = meta
    return K_new, None


def _candidate_near_best_direct_K_v13(self, scale=0.05):
    base = getattr(self, "global_best_K", None)
    if base is None:
        base = getattr(self, "vincent_seed_K", None)
    if base is None:
        base = _vincent_kernel_internal_order(self).copy()
        self.vincent_seed_K = base.copy()

    theta_scale = min(max(float(scale), 1e-4), DIRECT_LOCAL_SEARCH_THETA_SCALE)
    K_new, meta = _local_rotate_K(
        self,
        base,
        theta_scale=theta_scale,
        n_steps=max(DIRECT_ROTATION_STEPS_MUTATION, DIRECT_ROTATION_STEPS_SCOUT),
        source="ABC_K_local_search",
    )
    meta["theta_scale_used"] = theta_scale
    self._current_candidate_source = "ABC_K_local_search"
    self._current_candidate_meta = meta
    return K_new, None


def _random_candidate_direct_K_v13(self):
    base = getattr(self, "vincent_seed_K", None)
    if base is None:
        base = _vincent_kernel_internal_order(self).copy()
        self.vincent_seed_K = base.copy()
    K_new, meta = _local_rotate_K(
        self,
        base,
        theta_scale=DIRECT_RANDOM_THETA_SCALE,
        n_steps=DIRECT_ROTATION_STEPS_SCOUT,
        source="Direct_K_random_local",
    )
    meta["theta_scale_used"] = DIRECT_RANDOM_THETA_SCALE
    self._current_candidate_source = "Direct_K_random_local"
    self._current_candidate_meta = meta
    return K_new, None


def _random_step_direct_K_v13(self, n_evals=1, include_center_once=False):
    if getattr(self, "vincent_seed_K", None) is None:
        self.vincent_seed_K = _vincent_kernel_internal_order(self).copy()
        _food_from_K_direct(
            self,
            self.vincent_seed_K,
            source="Vincent_K_seed_random",
            trial=0,
            meta={"accepted_rotations": 0, "last_theta": 0.0, "last_pair": None},
        )

    for _ in range(int(n_evals)):
        base = getattr(self, "global_best_K", self.vincent_seed_K)
        K_new, meta = _local_rotate_K(
            self,
            base,
            theta_scale=DIRECT_RANDOM_THETA_SCALE,
            n_steps=DIRECT_ROTATION_STEPS_SCOUT,
            source="Random_K_local",
        )
        meta["theta_scale_used"] = DIRECT_RANDOM_THETA_SCALE
        self._current_candidate_source = "Random_K_local"
        self._current_candidate_meta = meta
        _food_from_K_direct(self, K_new, source="Random_K_local", trial=0, meta=meta)

    self.history.append(self.global_best_eff_z)
    return {
        "best_eff_z": self.global_best_eff_z,
        "best_eff_y": self.global_best_eff_y,
        "best_score": self.global_best_score,
        "eval_count": self.eval_count,
        "valid_count": self.valid_count,
    }


# Install v13 overrides.
ABCAlgorithm.initialize_population = _initialize_population_direct_K_v13
ABCAlgorithm._mutate_food = _mutate_food_direct_K_v13
ABCAlgorithm._candidate_near_best = _candidate_near_best_direct_K_v13
ABCAlgorithm._random_candidate = _random_candidate_direct_K_v13
RandomSearchAlgorithm.step = _random_step_direct_K_v13

print("v13 exploration patch installed.")
print(f"  DIRECT_STEP_PROFILE={DIRECT_STEP_PROFILE}")
print("  Exact Vincent K is still the first food source.")
print("  First/local candidates are now stronger multi-rotation perturbations, not microscopic moves.")


v13 exploration patch installed.
  DIRECT_STEP_PROFILE=explore
  Exact Vincent K is still the first food source.
  First/local candidates are now stronger multi-rotation perturbations, not microscopic moves.


In [58]:
# ============================================================
# v14 HYBRID PATCH: Vincent seed + global CaDsd exploration
# ============================================================
# Why this patch exists:
#   v13 is a local direct-K search. That is good for testing whether Vincent
#   can be improved locally, but it can be trapped near Vincent, especially
#   for constant inclusion probabilities.
#
# v14 keeps the exact Vincent K as the first food source, but adds global
# CaDsd jumps. Therefore:
#   - best-of-bests can never be worse than Vincent;
#   - CONST cases can still find far-away equal-pi designs, like the older
#     global omega/rho ABC did;
#   - node traces show which source produced the improvement.
# ============================================================

USE_HYBRID_GLOBAL_CADSD = True

# Global jump intensity. For CONST, use more global jumps because the local
# Vincent neighborhood is often flat/trapped. For unequal pi, keep fewer jumps.
HYBRID_CONST_INIT_GLOBAL_FRACTION = 0.35
HYBRID_UNEQUAL_INIT_GLOBAL_FRACTION = 0.00
HYBRID_CONST_MUTATION_GLOBAL_PROB = 0.12
HYBRID_UNEQUAL_MUTATION_GLOBAL_PROB = 0.00
HYBRID_CONST_RANDOM_GLOBAL_PROB = 0.18
HYBRID_UNEQUAL_RANDOM_GLOBAL_PROB = 0.00
HYBRID_CONST_SCOUT_GLOBAL_PROB = 0.25
HYBRID_UNEQUAL_SCOUT_GLOBAL_PROB = 0.00
HYBRID_GLOBAL_MAX_TRIES = 3
HYBRID_CONST_PI_TOL = 1e-12


def _is_const_pi_direct(self):
    pi = np.asarray(self.pik_sorted, dtype=float)
    return float(np.max(pi) - np.min(pi)) <= HYBRID_CONST_PI_TOL


def _hybrid_probs(self):
    if _is_const_pi_direct(self):
        return {
            "init_global_fraction": HYBRID_CONST_INIT_GLOBAL_FRACTION,
            "mutation_global_prob": HYBRID_CONST_MUTATION_GLOBAL_PROB,
            "random_global_prob": HYBRID_CONST_RANDOM_GLOBAL_PROB,
            "scout_global_prob": HYBRID_CONST_SCOUT_GLOBAL_PROB,
            "case_type": "CONST-pi",
        }
    return {
        "init_global_fraction": HYBRID_UNEQUAL_INIT_GLOBAL_FRACTION,
        "mutation_global_prob": HYBRID_UNEQUAL_MUTATION_GLOBAL_PROB,
        "random_global_prob": HYBRID_UNEQUAL_RANDOM_GLOBAL_PROB,
        "scout_global_prob": HYBRID_UNEQUAL_SCOUT_GLOBAL_PROB,
        "case_type": "unequal-pi",
    }


def _global_cadsd_K_candidate(self, source="Global_CaDsd_jump"):
    """Generate one global CaDsd kernel in the current/internal order."""
    omega = self.rng.random((self.M, self.N))
    rho = self.rng.random((self.M, self.N - 1))
    try:
        out = CaDsd(pi=self.pik_sorted, M=self.M, omega=omega, rho=rho)
        K = _symmetrize_K(out["K"])
        meta = {
            "accepted_rotations": np.nan,
            "last_theta": np.nan,
            "last_pair": None,
            "source": source,
            "jump_type": "global_CaDsd",
        }
        return K, meta
    except Exception:
        return None, {
            "accepted_rotations": np.nan,
            "last_theta": np.nan,
            "last_pair": None,
            "source": source,
            "jump_type": "global_CaDsd_failed",
        }


def _make_global_cadsd_food(self, source="Global_CaDsd_jump", trial=0):
    for _ in range(HYBRID_GLOBAL_MAX_TRIES):
        K, meta = _global_cadsd_K_candidate(self, source=source)
        if K is None:
            continue
        food = _food_from_K_direct(self, K, source=source, trial=trial, meta=meta)
        if food is not None:
            food["source"] = source
            food["jump_type"] = "global_CaDsd"
            return food
    return None


def _initialize_population_hybrid_v14(self, colony_size, verbose=True):
    probs = _hybrid_probs(self)
    if verbose:
        print(f" Initializing {colony_size} food sources by HYBRID search...")
        print("    first food source = exact Vincent K")
        print("    local source      = direct-K rotations around Vincent/current K")
        print("    global source     = random CaDsd omega/rho jumps")
        print(f"    case_type={probs['case_type']}; init_global_fraction={probs['init_global_fraction']}")

    self.vincent_seed_K = _vincent_kernel_internal_order(self).copy()
    population = []

    # 1) Exact Vincent seed: guaranteed floor.
    self._current_candidate_source = "Vincent_K_seed"
    self._current_candidate_meta = {"accepted_rotations": 0, "last_theta": 0.0, "last_pair": None, "jump_type": "vincent"}
    seed_food = _food_from_K_direct(
        self,
        self.vincent_seed_K,
        source="Vincent_K_seed",
        trial=0,
        meta=self._current_candidate_meta,
    )
    if seed_food is None:
        raise RuntimeError("Vincent K seed failed direct validation. This should not happen.")
    population.append(seed_food)

    target_global = int(round((colony_size - 1) * probs["init_global_fraction"]))
    target_global = max(0, min(colony_size - 1, target_global))
    target_local = (colony_size - 1) - target_global

    # 2) Local candidates around Vincent.
    attempts = 0
    made_local = 0
    while made_local < target_local and attempts < max(200, target_local * 120):
        attempts += 1
        K_new, meta = _local_rotate_K(
            self,
            self.vincent_seed_K,
            theta_scale=DIRECT_INIT_THETA_SCALE,
            n_steps=DIRECT_ROTATION_STEPS_INIT,
            source="Vincent_K_local_init",
        )
        meta["jump_type"] = "local_K_rotation"
        food = _food_from_K_direct(self, K_new, source="Vincent_K_local_init", trial=0, meta=meta)
        if food is not None:
            population.append(food)
            made_local += 1

    # 3) Global CaDsd candidates.
    made_global = 0
    attempts = 0
    while made_global < target_global and attempts < max(200, target_global * HYBRID_GLOBAL_MAX_TRIES):
        attempts += 1
        food = _make_global_cadsd_food(self, source="Global_CaDsd_init", trial=0)
        if food is not None:
            population.append(food)
            made_global += 1

    # 4) Fill any missing slots with either local or global, biased by case type.
    attempts = 0
    while len(population) < colony_size and attempts < colony_size * 200:
        attempts += 1
        if self.rng.random() < probs["init_global_fraction"]:
            food = _make_global_cadsd_food(self, source="Global_CaDsd_fill", trial=0)
            if food is not None:
                population.append(food)
        else:
            K_new, meta = _local_rotate_K(
                self,
                self.vincent_seed_K,
                theta_scale=DIRECT_INIT_THETA_SCALE,
                n_steps=DIRECT_ROTATION_STEPS_INIT,
                source="Vincent_K_local_fill",
            )
            meta["jump_type"] = "local_K_rotation"
            food = _food_from_K_direct(self, K_new, source="Vincent_K_local_fill", trial=0, meta=meta)
            if food is not None:
                population.append(food)

    if verbose:
        n_global = sum(str(f.get("source", "")).startswith("Global_CaDsd") for f in population)
        n_local = len(population) - n_global - 1
        print(f"    Initialized {len(population)} food sources (requested {colony_size})")
        print(f"    composition: Vincent=1, local={n_local}, global={n_global}")
        print(
            f"    Best initial: eff_z={self.global_best_eff_z:.8f}, "
            f"z/V={_ratio_to_ref(self.global_best_eff_z, self.eff_z_optimal):.8f}, "
            f"source={getattr(self, 'global_best_source', None)}"
        )

    if len(population) == 0:
        raise RuntimeError("Hybrid initialization produced no valid food sources.")
    return population


def _mutate_food_hybrid_v14(self, food, partner, progress, mode="abc"):
    probs = _hybrid_probs(self)

    # For CONST we need occasional far jumps. Otherwise the greedy local search
    # can be trapped near Vincent forever.
    if self.rng.random() < probs["mutation_global_prob"]:
        K, meta = _global_cadsd_K_candidate(self, source="ABC_Global_CaDsd_jump")
        if K is not None:
            self._current_candidate_source = "ABC_Global_CaDsd_jump"
            self._current_candidate_meta = meta
            return K, None

    # Otherwise use the v13 local direct-K mutation.
    base = food.get("K", self.vincent_seed_K)
    scale = DIRECT_MUTATION_THETA_SCALE * max(0.35, 1.0 - 0.50 * float(progress))
    K_new, meta = _local_rotate_K(
        self,
        base,
        theta_scale=scale,
        n_steps=DIRECT_ROTATION_STEPS_MUTATION,
        source="ABC_K_local_candidate",
    )
    meta["theta_scale_used"] = scale
    meta["jump_type"] = "local_K_rotation"
    self._current_candidate_source = "ABC_K_local_candidate"
    self._current_candidate_meta = meta
    return K_new, None


def _candidate_near_best_hybrid_v14(self, scale=0.05):
    probs = _hybrid_probs(self)
    if self.rng.random() < probs["scout_global_prob"]:
        K, meta = _global_cadsd_K_candidate(self, source="ABC_Global_CaDsd_scout")
        if K is not None:
            self._current_candidate_source = "ABC_Global_CaDsd_scout"
            self._current_candidate_meta = meta
            return K, None

    base = getattr(self, "global_best_K", None)
    if base is None:
        base = getattr(self, "vincent_seed_K", None)
    if base is None:
        base = _vincent_kernel_internal_order(self).copy()
        self.vincent_seed_K = base.copy()

    theta_scale = min(max(float(scale), 1e-4), DIRECT_LOCAL_SEARCH_THETA_SCALE)
    K_new, meta = _local_rotate_K(
        self,
        base,
        theta_scale=theta_scale,
        n_steps=max(DIRECT_ROTATION_STEPS_MUTATION, DIRECT_ROTATION_STEPS_SCOUT),
        source="ABC_K_local_search",
    )
    meta["theta_scale_used"] = theta_scale
    meta["jump_type"] = "local_K_rotation"
    self._current_candidate_source = "ABC_K_local_search"
    self._current_candidate_meta = meta
    return K_new, None


def _random_candidate_hybrid_v14(self):
    probs = _hybrid_probs(self)
    if self.rng.random() < probs["random_global_prob"]:
        K, meta = _global_cadsd_K_candidate(self, source="Direct_Global_CaDsd_random")
        if K is not None:
            self._current_candidate_source = "Direct_Global_CaDsd_random"
            self._current_candidate_meta = meta
            return K, None

    base = getattr(self, "vincent_seed_K", None)
    if base is None:
        base = _vincent_kernel_internal_order(self).copy()
        self.vincent_seed_K = base.copy()
    K_new, meta = _local_rotate_K(
        self,
        base,
        theta_scale=DIRECT_RANDOM_THETA_SCALE,
        n_steps=DIRECT_ROTATION_STEPS_SCOUT,
        source="Direct_K_random_local",
    )
    meta["theta_scale_used"] = DIRECT_RANDOM_THETA_SCALE
    meta["jump_type"] = "local_K_rotation"
    self._current_candidate_source = "Direct_K_random_local"
    self._current_candidate_meta = meta
    return K_new, None


def _random_step_hybrid_v14(self, n_evals=1, include_center_once=False):
    if getattr(self, "vincent_seed_K", None) is None:
        self.vincent_seed_K = _vincent_kernel_internal_order(self).copy()
        _food_from_K_direct(
            self,
            self.vincent_seed_K,
            source="Vincent_K_seed_random",
            trial=0,
            meta={"accepted_rotations": 0, "last_theta": 0.0, "last_pair": None, "jump_type": "vincent"},
        )

    probs = _hybrid_probs(self)
    for _ in range(int(n_evals)):
        if self.rng.random() < probs["random_global_prob"]:
            food = _make_global_cadsd_food(self, source="Random_Global_CaDsd", trial=0)
            # _make_global_cadsd_food already updates the global best via _food_from_K_direct.
            continue

        # Local random hill-climb around current best, as in v13.
        base = getattr(self, "global_best_K", self.vincent_seed_K)
        K_new, meta = _local_rotate_K(
            self,
            base,
            theta_scale=DIRECT_RANDOM_THETA_SCALE,
            n_steps=DIRECT_ROTATION_STEPS_SCOUT,
            source="Random_K_local",
        )
        meta["theta_scale_used"] = DIRECT_RANDOM_THETA_SCALE
        meta["jump_type"] = "local_K_rotation"
        self._current_candidate_source = "Random_K_local"
        self._current_candidate_meta = meta
        _food_from_K_direct(self, K_new, source="Random_K_local", trial=0, meta=meta)

    self.history.append(self.global_best_eff_z)
    return {
        "best_eff_z": self.global_best_eff_z,
        "best_eff_y": self.global_best_eff_y,
        "best_score": self.global_best_score,
        "eval_count": self.eval_count,
        "valid_count": self.valid_count,
    }


# Install v14 overrides.
ABCAlgorithm.initialize_population = _initialize_population_hybrid_v14
ABCAlgorithm._mutate_food = _mutate_food_hybrid_v14
ABCAlgorithm._candidate_near_best = _candidate_near_best_hybrid_v14
ABCAlgorithm._random_candidate = _random_candidate_hybrid_v14
RandomSearchAlgorithm.step = _random_step_hybrid_v14

print("v14 hybrid patch installed.")
print("  Exact Vincent K is still the first food source and best-of-bests floor.")
print("  CONST cases get global CaDsd jumps so the search is not trapped near Vincent.")
print("  Sources in trace/results tell you whether improvement came from local K rotation or global CaDsd jump.")


v14 hybrid patch installed.
  Exact Vincent K is still the first food source and best-of-bests floor.
  CONST cases get global CaDsd jumps so the search is not trapped near Vincent.
  Sources in trace/results tell you whether improvement came from local K rotation or global CaDsd jump.


In [59]:
# ============================================================
# v15 PATCH: no-silence hybrid initialization
# ============================================================
# The v14 initialization could look frozen after:
#     case_type=...; init_global_fraction=...
# because it used nested global-CaDsd attempt loops.
# v15 makes initialization explicit:
#   - prints progress during local/global initialization;
#   - removes nested global attempt loops in initialization;
#   - for unequal pi, starts with local Vincent rotations only by default;
#   - for CONST pi, keeps global CaDsd exploration but with controlled attempts.
# ============================================================

HYBRID_UNEQUAL_INIT_GLOBAL_FRACTION = 0.00   # unequal-pi already works well with local K search
HYBRID_CONST_INIT_GLOBAL_FRACTION = 0.35     # CONST needs far/global candidates
HYBRID_GLOBAL_MAX_TRIES = 3                 # used by later global random/scout calls; was too large
HYBRID_INIT_VERBOSE_EVERY = 5
HYBRID_INIT_GLOBAL_ATTEMPT_MULT = 40         # one CaDsd call per attempt; no nested 80x loop


def _make_one_global_cadsd_food_v15(self, source="Global_CaDsd_init", trial=0):
    """One global CaDsd attempt only. No nested retry loop."""
    K, meta = _global_cadsd_K_candidate(self, source=source)
    if K is None:
        return None
    food = _food_from_K_direct(self, K, source=source, trial=trial, meta=meta)
    if food is not None:
        food["source"] = source
        food["jump_type"] = "global_CaDsd"
    return food


def _initialize_population_hybrid_v15(self, colony_size, verbose=True):
    probs = _hybrid_probs(self)
    if verbose:
        print(f" Initializing {colony_size} food sources by HYBRID search [v15 no-silence]...", flush=True)
        print("    first food source = exact Vincent K", flush=True)
        print("    local source      = direct-K rotations around Vincent/current K", flush=True)
        print("    global source     = random CaDsd omega/rho jumps", flush=True)
        print(f"    case_type={probs['case_type']}; init_global_fraction={probs['init_global_fraction']}", flush=True)

    self.vincent_seed_K = _vincent_kernel_internal_order(self).copy()
    population = []

    # 1) Exact Vincent seed: guaranteed floor.
    self._current_candidate_source = "Vincent_K_seed"
    self._current_candidate_meta = {
        "accepted_rotations": 0,
        "last_theta": 0.0,
        "last_pair": None,
        "jump_type": "vincent",
    }
    seed_food = _food_from_K_direct(
        self,
        self.vincent_seed_K,
        source="Vincent_K_seed",
        trial=0,
        meta=self._current_candidate_meta,
    )
    if seed_food is None:
        raise RuntimeError("Vincent K seed failed direct validation. This should not happen.")
    population.append(seed_food)
    if verbose:
        print("    added Vincent seed: 1/{}".format(colony_size), flush=True)

    target_global = int(round((colony_size - 1) * probs["init_global_fraction"]))
    target_global = max(0, min(colony_size - 1, target_global))
    target_local = (colony_size - 1) - target_global

    # 2) Local candidates around Vincent.
    attempts = 0
    made_local = 0
    max_local_attempts = max(200, target_local * 120)
    while made_local < target_local and attempts < max_local_attempts:
        attempts += 1
        K_new, meta = _local_rotate_K(
            self,
            self.vincent_seed_K,
            theta_scale=DIRECT_INIT_THETA_SCALE,
            n_steps=DIRECT_ROTATION_STEPS_INIT,
            source="Vincent_K_local_init",
        )
        meta["jump_type"] = "local_K_rotation"
        food = _food_from_K_direct(self, K_new, source="Vincent_K_local_init", trial=0, meta=meta)
        if food is not None:
            population.append(food)
            made_local += 1
            if verbose and (made_local == target_local or made_local % HYBRID_INIT_VERBOSE_EVERY == 0):
                print(f"    local init:  {made_local}/{target_local} accepted; population={len(population)}/{colony_size}", flush=True)

    # 3) Global CaDsd candidates. One CaDsd attempt per loop; no hidden nested loop.
    made_global = 0
    attempts = 0
    max_global_attempts = max(20, target_global * HYBRID_INIT_GLOBAL_ATTEMPT_MULT)
    if verbose and target_global > 0:
        print(f"    starting global init: target={target_global}, max_attempts={max_global_attempts}", flush=True)

    while made_global < target_global and attempts < max_global_attempts:
        attempts += 1
        food = _make_one_global_cadsd_food_v15(self, source="Global_CaDsd_init", trial=0)
        if food is not None:
            population.append(food)
            made_global += 1
            if verbose and (made_global == target_global or made_global % HYBRID_INIT_VERBOSE_EVERY == 0):
                print(f"    global init: {made_global}/{target_global} accepted after {attempts} attempts; population={len(population)}/{colony_size}", flush=True)
        elif verbose and attempts % 50 == 0:
            print(f"    global init: {made_global}/{target_global} accepted after {attempts} attempts...", flush=True)

    # 4) Fill any missing slots. For unequal-pi, fill locally; for CONST, try global sometimes.
    attempts = 0
    max_fill_attempts = colony_size * 120
    while len(population) < colony_size and attempts < max_fill_attempts:
        attempts += 1
        use_global = (_is_const_pi_direct(self) and self.rng.random() < probs["init_global_fraction"])
        if use_global:
            food = _make_one_global_cadsd_food_v15(self, source="Global_CaDsd_fill", trial=0)
            if food is not None:
                population.append(food)
        else:
            K_new, meta = _local_rotate_K(
                self,
                self.vincent_seed_K,
                theta_scale=DIRECT_INIT_THETA_SCALE,
                n_steps=DIRECT_ROTATION_STEPS_INIT,
                source="Vincent_K_local_fill",
            )
            meta["jump_type"] = "local_K_rotation"
            food = _food_from_K_direct(self, K_new, source="Vincent_K_local_fill", trial=0, meta=meta)
            if food is not None:
                population.append(food)

        if verbose and (len(population) == colony_size or len(population) % HYBRID_INIT_VERBOSE_EVERY == 0):
            print(f"    fill init: population={len(population)}/{colony_size}", flush=True)

    if verbose:
        n_global = sum(str(f.get("source", "")).startswith("Global_CaDsd") for f in population)
        n_local = len(population) - n_global - 1
        print(f"    Initialized {len(population)} food sources (requested {colony_size})", flush=True)
        print(f"    composition: Vincent=1, local={n_local}, global={n_global}", flush=True)
        print(
            f"    Best initial: eff_z={self.global_best_eff_z:.8f}, "
            f"z/V={_ratio_to_ref(self.global_best_eff_z, self.eff_z_optimal):.8f}, "
            f"source={getattr(self, 'global_best_source', None)}",
            flush=True,
        )

    if len(population) == 0:
        raise RuntimeError("Hybrid initialization produced no valid food sources.")
    return population


ABCAlgorithm.initialize_population = _initialize_population_hybrid_v15

print("v15 no-silence hybrid initialization installed.")
print("  Unequal-pi init_global_fraction =", HYBRID_UNEQUAL_INIT_GLOBAL_FRACTION)
print("  CONST-pi init_global_fraction   =", HYBRID_CONST_INIT_GLOBAL_FRACTION)
print("  Global initialization now prints progress and has no nested retry loop.")


v15 no-silence hybrid initialization installed.
  Unequal-pi init_global_fraction = 0.0
  CONST-pi init_global_fraction   = 0.35
  Global initialization now prints progress and has no nested retry loop.


In [60]:
# ============================================================
# PRECHECK before main run:
# 1) Vincent Pπ is computed directly and through the ABC probe.
# 2) The same inclusion probabilities are used everywhere.
# 3) CaDsd uses the same π values, only in descending order internally.
# ============================================================

RUN_PRECHECK = True

# CaDsd rounds the constructed kernel internally, so its diagonal is
# only approximately equal to pi. This matches the tolerance used in
# ABCAlgorithm._kernel_passes_validation.
CADSD_PI_ATOL = 1e-3


def debug_vincent_reference_one_case(df, y_name, z_name, x_name, n):
    y_raw = df[y_name].to_numpy(dtype=float)
    z_raw = df[z_name].to_numpy(dtype=float)
    x_raw = df[x_name].to_numpy(dtype=float)

    N = len(df)
    pi_raw = inclusionprobabilities(x_raw, n)

    var_srs_y = N**2 * (1.0 - n / N) * np.var(y_raw, ddof=1) / n
    var_srs_z = N**2 * (1.0 - n / N) * np.var(z_raw, ddof=1) / n

    # Vincent order: rank by z/pi.
    idx_vincent = np.argsort(z_raw / pi_raw)

    y_v = y_raw[idx_vincent]
    z_v = z_raw[idx_vincent]
    pi_v = pi_raw[idx_vincent]

    # Direct Vincent Pπ calculation.
    vin_direct = _ppi_efficiency_for_order(
        y_v,
        z_v,
        pi_v,
        var_srs_y,
        var_srs_z,
    )

    # Probe ABC object. This does not run optimization.
    # It only computes the corrected references.
    probe = ABCAlgorithm(
        y_sorted=y_v,
        z_sorted=z_v,
        pik_sorted=pi_v,
        var_srs_y=var_srs_y,
        var_srs_z=var_srs_z,
        M=n,
        n=n,
        case_name=f"DEBUG_{z_name}_n{n}",
        objective="eff_z",
        enforce_cadsd_order=True,
        random_state=123,
        validation_mode="fast",
        eigen_check_interval=0,
        initial_strict_checks=2,
    )

    # Check one CaDsd center kernel: it should use the same π values,
    # but internally sorted decreasingly.
    center_omega = 0.5 * np.ones((n, N))
    center_rho = 0.5 * np.ones((n, N - 1))

    try:
        K_center = CaDsd(pi=pi_v, M=n, omega=center_omega, rho=center_rho)["K"]
        cadsd_diag = np.real(np.diag(K_center))
        cadsd_same_pi_multiset = np.allclose(
            np.sort(cadsd_diag),
            np.sort(pi_raw),
            atol=CADSD_PI_ATOL,
            rtol=CADSD_PI_ATOL,
        )
        cadsd_diag_sorted_error = float(np.max(np.abs(np.sort(cadsd_diag) - np.sort(pi_raw))))
    except Exception:
        cadsd_same_pi_multiset = False
        cadsd_diag_sorted_error = np.nan

    return {
        "z": z_name,
        "x_for_pi": x_name,
        "n": n,
        "sum_pi": pi_raw.sum(),

        "same_pi_raw_vs_vincent_order": np.allclose(
            np.sort(pi_raw),
            np.sort(pi_v),
            atol=1e-12,
            rtol=1e-12,
        ),

        "ABC_probe_same_pi_check": probe.same_pi_multiset_check,
        "CaDsd_center_same_pi_multiset": cadsd_same_pi_multiset,

        "PpiVincent_diag_error": vin_direct["diag_error"],
        "CaDsd_diag_sorted_error": cadsd_diag_sorted_error,

        "PpiVincent_eff_z_direct": vin_direct["eff_z"],
        "PpiVincent_eff_z_probe": probe.eff_z_optimal,
        "PpiInternal_eff_z_probe": probe.eff_z_internal_ppi,

        "Probe_matches_direct_Vincent": np.allclose(
            probe.eff_z_optimal,
            vin_direct["eff_z"],
            atol=1e-8,
            rtol=1e-8,
        ),

        "PpiVincent_over_Internal_z": probe.eff_z_optimal / probe.eff_z_internal_ppi,
    }


if RUN_PRECHECK:
    debug_rows = []
    for z_name, x_name, label in test_cases:
        for n in sample_sizes:
            debug_rows.append(
                debug_vincent_reference_one_case(
                    df=df,
                    y_name=y_var,
                    z_name=z_name,
                    x_name=x_name,
                    n=n,
                )
            )

    debug_vincent = pd.DataFrame(debug_rows)

    must_be_true = [
        "same_pi_raw_vs_vincent_order",
        "ABC_probe_same_pi_check",
        "CaDsd_center_same_pi_multiset",
        "Probe_matches_direct_Vincent",
    ]

    print("=" * 90)
    print("VINCENT REFERENCE / SAME-π PRECHECK")
    print("=" * 90)
    display_cols = [
        "z", "n", "sum_pi",
        "same_pi_raw_vs_vincent_order",
        "ABC_probe_same_pi_check",
        "CaDsd_center_same_pi_multiset",
        "Probe_matches_direct_Vincent",
        "PpiVincent_eff_z_direct",
        "PpiInternal_eff_z_probe",
        "PpiVincent_over_Internal_z",
        "PpiVincent_diag_error",
        "CaDsd_diag_sorted_error",
    ]

    debug_display = debug_vincent[display_cols].copy()
    num_cols = debug_display.select_dtypes(include=[np.number]).columns
    debug_display[num_cols] = debug_display[num_cols].round(6)
    display(debug_display)

    if not debug_vincent[must_be_true].all().all():
        bad = debug_vincent.loc[~debug_vincent[must_be_true].all(axis=1), ["z", "n"] + must_be_true]
        display(bad)
        raise RuntimeError("Precheck failed: at least one same-π or Vincent-reference check is False.")

    if debug_vincent["PpiVincent_diag_error"].max() > 1e-6:
        raise RuntimeError("Precheck failed: Vincent Pπ diagonal error is too large.")

    if debug_vincent["CaDsd_diag_sorted_error"].max() > CADSD_PI_ATOL:
        raise RuntimeError("Precheck failed: CaDsd diagonal differs from pi beyond CADSD_PI_ATOL.")

    print("PRECHECK PASSED: same π values are used; Vincent Pπ reference is correctly computed.")
else:
    print("RUN_PRECHECK=False: skipped precheck.")

VINCENT REFERENCE / SAME-π PRECHECK


,z,n,sum_pi,same_pi_raw_vs_vincent_order,ABC_probe_same_pi_check,CaDsd_center_same_pi_multiset,Probe_matches_direct_Vincent,PpiVincent_eff_z_direct,PpiInternal_eff_z_probe,PpiVincent_over_Internal_z,PpiVincent_diag_error,CaDsd_diag_sorted_error
0,ME84,5,5.0,True,True,True,True,6.484103,6.484103,1.0,0.0,0.000013
1,S82,5,5.0,True,True,True,True,4.941245,4.941244,1.0,0.0,0.000013
2,CS82,5,5.0,True,True,True,True,11.053318,11.053318,1.0,0.0,0.000013


PRECHECK PASSED: same π values are used; Vincent Pπ reference is correctly computed.


In [61]:
# ============================================================
# UNIT ORDER DEBUG:
# shows that π is the same, but Vincent and CaDsd internal orders differ.
# ============================================================

SHOW_UNIT_ORDER_DEBUG = True
UNIT_DEBUG_Z = "ME84"
UNIT_DEBUG_X = "P75"
UNIT_DEBUG_N = 5

if SHOW_UNIT_ORDER_DEBUG:
    z_name = UNIT_DEBUG_Z
    x_name = UNIT_DEBUG_X
    n = UNIT_DEBUG_N

    y_raw = df[y_var].to_numpy(dtype=float)
    z_raw = df[z_name].to_numpy(dtype=float)
    x_raw = df[x_name].to_numpy(dtype=float)

    pi_raw = inclusionprobabilities(x_raw, n)
    z_over_pi = z_raw / pi_raw

    idx_vincent = np.argsort(z_over_pi)

    # Internal order after CaDsd/ABC sorting.
    idx_internal = idx_vincent[np.argsort(pi_raw[idx_vincent])[::-1]]

    order_example = pd.DataFrame({
        "unit_original_index": np.arange(len(df)),
        "x_for_pi": x_raw,
        "pi": pi_raw,
        f"z_{z_name}": z_raw,
        "z_over_pi": z_over_pi,
    })

    order_example["rank_Vincent_z_over_pi"] = np.empty(len(df), dtype=int)
    order_example.loc[idx_vincent, "rank_Vincent_z_over_pi"] = np.arange(1, len(df) + 1)

    order_example["rank_internal_pi_desc"] = np.empty(len(df), dtype=int)
    order_example.loc[idx_internal, "rank_internal_pi_desc"] = np.arange(1, len(df) + 1)

    print("Same π values in raw and Vincent order:",
          np.allclose(np.sort(pi_raw), np.sort(pi_raw[idx_vincent])))

    print("Same π values in raw and internal order:",
          np.allclose(np.sort(pi_raw), np.sort(pi_raw[idx_internal])))

    print(f"\nFirst 15 units in Vincent z/pi order for z={z_name}, n={n}:")
    display(order_example.sort_values("rank_Vincent_z_over_pi").head(15))

    print(f"\nFirst 15 units in internal descending-pi order for z={z_name}, n={n}:")
    display(order_example.sort_values("rank_internal_pi_desc").head(15))
else:
    print("SHOW_UNIT_ORDER_DEBUG=False: skipped unit-order debug.")

Same π values in raw and Vincent order: True
Same π values in raw and internal order: True

First 15 units in Vincent z/pi order for z=ME84, n=5:


,unit_original_index,x_for_pi,pi,z_ME84,z_over_pi,rank_Vincent_z_over_pi,rank_internal_pi_desc
10,10,11.0,0.097345,442.0,4540.545455,1,21
25,25,10.0,0.088496,418.0,4723.400000,2,25
7,7,13.0,0.115044,564.0,4902.461538,3,18
28,28,20.0,0.176991,868.0,4904.200000,4,13
13,13,5.0,0.044248,235.0,5311.000000,5,30
21,21,14.0,0.123894,664.0,5359.428571,6,17
17,17,19.0,0.168142,917.0,5453.736842,7,14
29,29,33.0,0.292035,1597.0,5468.515152,8,4
1,1,7.0,0.061947,339.0,5472.428571,9,28
24,24,28.0,0.247788,1398.0,5641.928571,10,7



First 15 units in internal descending-pi order for z=ME84, n=5:


,unit_original_index,x_for_pi,pi,z_ME84,z_over_pi,rank_Vincent_z_over_pi,rank_internal_pi_desc
11,11,47.0,0.415929,2771.0,6662.191489,30,1
15,15,39.0,0.345133,2294.0,6646.717949,29,2
12,12,33.0,0.292035,1741.0,5961.606061,17,3
29,29,33.0,0.292035,1597.0,5468.515152,8,4
8,8,30.0,0.265487,1537.0,5789.366667,14,5
19,19,28.0,0.247788,1616.0,6521.714286,27,6
24,24,28.0,0.247788,1398.0,5641.928571,10,7
0,0,27.0,0.238938,1377.0,5763.000000,13,8
3,3,27.0,0.238938,1441.0,6030.851852,20,9
9,9,25.0,0.221239,1311.0,5925.720000,16,10


### main run

In [62]:
# ============================================================
# v16 FAST-AUTO GUARD
# ============================================================
# This cell is intentionally placed immediately before MAIN RUN.
# It prevents the slow v14 behavior from coming back.
#
# Unequal-pi cases, e.g. REV84:
#     use fast direct-K local search only.
#     no global CaDsd jumps, because they are slow and unnecessary here.
#
# Constant-pi cases, i.e. CONST:
#     allow controlled global CaDsd jumps, because local Vincent rotations
#     can be flat/trapped for equal inclusion probabilities.
# ============================================================

HYBRID_UNEQUAL_INIT_GLOBAL_FRACTION = 0.00
HYBRID_UNEQUAL_MUTATION_GLOBAL_PROB = 0.00
HYBRID_UNEQUAL_RANDOM_GLOBAL_PROB = 0.00
HYBRID_UNEQUAL_SCOUT_GLOBAL_PROB = 0.00

HYBRID_CONST_INIT_GLOBAL_FRACTION = 0.35
HYBRID_CONST_MUTATION_GLOBAL_PROB = 0.12
HYBRID_CONST_RANDOM_GLOBAL_PROB = 0.18
HYBRID_CONST_SCOUT_GLOBAL_PROB = 0.25

HYBRID_GLOBAL_MAX_TRIES = 3
HYBRID_INIT_GLOBAL_ATTEMPT_MULT = 12

print("v16 fast-auto guard active.")
print("  Unequal pi: global CaDsd disabled -> fast direct-K Vincent search.")
print("  CONST pi: controlled global CaDsd enabled -> can escape Vincent local trap.")
print(f"  Unequal global probs: init={HYBRID_UNEQUAL_INIT_GLOBAL_FRACTION}, mut={HYBRID_UNEQUAL_MUTATION_GLOBAL_PROB}, rand={HYBRID_UNEQUAL_RANDOM_GLOBAL_PROB}, scout={HYBRID_UNEQUAL_SCOUT_GLOBAL_PROB}")
print(f"  CONST global probs: init={HYBRID_CONST_INIT_GLOBAL_FRACTION}, mut={HYBRID_CONST_MUTATION_GLOBAL_PROB}, rand={HYBRID_CONST_RANDOM_GLOBAL_PROB}, scout={HYBRID_CONST_SCOUT_GLOBAL_PROB}")
print(f"  HYBRID_GLOBAL_MAX_TRIES={HYBRID_GLOBAL_MAX_TRIES}")


v16 fast-auto guard active.
  Unequal pi: global CaDsd disabled -> fast direct-K Vincent search.
  CONST pi: controlled global CaDsd enabled -> can escape Vincent local trap.
  Unequal global probs: init=0.0, mut=0.0, rand=0.0, scout=0.0
  CONST global probs: init=0.35, mut=0.12, rand=0.18, scout=0.25
  HYBRID_GLOBAL_MAX_TRIES=3


In [63]:
# ============================================================
# MAIN RUN: ABC + Random Search
# Reference in ratios = Vincent ordered Pπ
# ============================================================

RUN_MAIN = True

if RUN_MAIN:
    print("=" * 80)
    print("ABC + RANDOM SEARCH ON CURRENT MU284 DATAFRAME")
    print("Corrected reference: Vincent z/pi ordered Pπ")
    print("=" * 80)

    N = len(df)
    all_results = []
    abc_run_details = {}

    for z_name, x_name, corr_label in test_cases:

        print(f"\n{'=' * 80}")
        print(f"{corr_label}: z = {z_name}; π from {x_name}")
        print(f"{'=' * 80}")

        y_raw = df[y_var].to_numpy(dtype=float)
        z_raw = df[z_name].to_numpy(dtype=float)
        x_raw = df[x_name].to_numpy(dtype=float)

        actual_corr = np.corrcoef(y_raw, z_raw)[0, 1]

        print(f"Corr({y_var}, {z_name}) = {actual_corr:.3f}")
        print(f"N = {N} (current df used in this run)")

        for n in sample_sizes:

            print(f"\n  n = {n}:")

            # Same inclusion probabilities for Vincent, ABC, Random, and internal reference.
            pik_raw = inclusionprobabilities(x_raw, n)

            # Vincent empirical order: rank by z/pi.
            z_over_pi = z_raw / pik_raw
            sort_idx = np.argsort(z_over_pi)

            y_sorted = y_raw[sort_idx]
            z_sorted = z_raw[sort_idx]
            pik_sorted = pik_raw[sort_idx]

            # SRS variance with the same sample size:
            # Var_SRS(total) = N^2 * (1 - n/N) * s^2 / n
            var_srs_y = N**2 * (1.0 - n / N) * np.var(y_raw, ddof=1) / n
            var_srs_z = N**2 * (1.0 - n / N) * np.var(z_raw, ddof=1) / n

            print(
                f"    ABC params: colony={SIM_PARAMS['colony_size']}, "
                f"iter={SIM_PARAMS['max_iterations']}, "
                f"limit={SIM_PARAMS['limit']}, "
                f"onlookers={SIM_PARAMS['onlooker_factor']}, "
                f"progress_every={SIM_PARAMS['progress_interval']}, "
                f"time_limit={SIM_PARAMS['time_limit_seconds']}"
            )

            print(
                f"    Random Search: start_evals={SIM_PARAMS['random_start_evals']}, "
                f"evals_per_iter={SIM_PARAMS['random_evals_per_iteration']}"
            )

            run_key = f"{z_name}_N{N}_n{n}"
            seed_shift = 1000 * n + len(all_results)

            abc = ABCAlgorithm(
                y_sorted=y_sorted,
                z_sorted=z_sorted,
                pik_sorted=pik_sorted,
                var_srs_y=var_srs_y,
                var_srs_z=var_srs_z,
                M=n,
                n=n,
                case_name=run_key,
                objective=OBJECTIVE,
                enforce_cadsd_order=True,
                random_state=ABC_RANDOM_SEED + seed_shift,
                validation_mode=SIM_PARAMS["validation_mode"],
                eigen_check_interval=0,
                initial_strict_checks=2,
            )

            random_search = RandomSearchAlgorithm(
                y_sorted=y_sorted,
                z_sorted=z_sorted,
                pik_sorted=pik_sorted,
                var_srs_y=var_srs_y,
                var_srs_z=var_srs_z,
                M=n,
                n=n,
                case_name=run_key + "_random",
                objective=OBJECTIVE,
                enforce_cadsd_order=True,
                random_state=RANDOM_SEARCH_SEED + seed_shift,
                validation_mode=SIM_PARAMS["validation_mode"],
                eigen_check_interval=0,
                initial_strict_checks=2,
            )

            if not abc.same_pi_multiset_check:
                raise RuntimeError(f"Same-π check failed for {run_key}.")

            res = abc.optimize(
                colony_size=SIM_PARAMS["colony_size"],
                max_iterations=SIM_PARAMS["max_iterations"],
                limit=SIM_PARAMS["limit"],
                verbose=True,
                progress_interval=SIM_PARAMS["progress_interval"],
                local_search_interval=SIM_PARAMS["local_search_interval"],
                local_search_attempts=SIM_PARAMS["local_search_attempts"],
                onlooker_factor=SIM_PARAMS["onlooker_factor"],
                early_stopping=SIM_PARAMS["early_stopping"],
                patience=SIM_PARAMS["patience"],
                min_iterations=SIM_PARAMS["min_iterations"],
                rel_tol=SIM_PARAMS["rel_tol"],
                time_limit_seconds=SIM_PARAMS["time_limit_seconds"],
                random_searcher=random_search,
                random_start_evals=SIM_PARAMS["random_start_evals"],
                random_evals_per_iteration=SIM_PARAMS["random_evals_per_iteration"],
            )

            abc_z_vin = _ratio_to_ref(res["best_eff_z"], res["PpiVincent_eff_z"])
            abc_y_vin = _ratio_to_ref(res["best_eff_y"], res["PpiVincent_eff_y"])
            rnd_z_vin = _ratio_to_ref(res["random_best_eff_z"], res["PpiVincent_eff_z"])
            rnd_y_vin = _ratio_to_ref(res["random_best_eff_y"], res["PpiVincent_eff_y"])

            abc_z_int = _ratio_to_ref(res["best_eff_z"], res["PpiInternal_eff_z"])
            abc_y_int = _ratio_to_ref(res["best_eff_y"], res["PpiInternal_eff_y"])
            rnd_z_int = _ratio_to_ref(res["random_best_eff_z"], res["PpiInternal_eff_z"])
            rnd_y_int = _ratio_to_ref(res["random_best_eff_y"], res["PpiInternal_eff_y"])

            abc_valid_pct = _valid_percent(res["valid_count"], res["eval_count"])
            rnd_valid_pct = _valid_percent(res["random_valid_count"], res["random_eval_count"])

            print("\n    Results:")
            print(f"      PπVincent_z efficiency vs SRS = {res['PpiVincent_eff_z']:.2f}")
            print(f"      PπVincent_y efficiency vs SRS = {res['PpiVincent_eff_y']:.2f}")
            print(f"      PπInternal_z efficiency vs SRS = {res['PpiInternal_eff_z']:.2f}")
            print(f"      PπInternal_y efficiency vs SRS = {res['PpiInternal_eff_y']:.2f}")
            print(f"      ABC_z / PπVincent_z           = {abc_z_vin:.8f}  ({(abc_z_vin - 1.0) * 100:.6f}% over Vincent)")
            print(f"      ABC_y / PπVincent_y           = {abc_y_vin:.8f}  ({(abc_y_vin - 1.0) * 100:.6f}% over Vincent)")
            print(f"      Random_z / PπVincent_z        = {rnd_z_vin:.8f}  ({(rnd_z_vin - 1.0) * 100:.6f}% over Vincent)")
            print(f"      Random_y / PπVincent_y        = {rnd_y_vin:.8f}  ({(rnd_y_vin - 1.0) * 100:.6f}% over Vincent)")
            print(f"      ABC_z / PπInternal_z          = {abc_z_int:.8f}")
            print(f"      Random_z / PπInternal_z       = {rnd_z_int:.8f}")
            print(f"      same π check                  = {res['same_pi_multiset_check']}")
            print(f"      Vincent Pπ diag error          = {res['vincent_ppi_diag_error']:.3e}")
            print(f"      ABC valid/try                  = {abc_valid_pct:.2f}%")
            print(f"      Random valid/try               = {rnd_valid_pct:.2f}%")
            print(f"      Best ABC source                = {res.get('best_source')}")
            print(f"      Best Random source             = {getattr(random_search, 'global_best_source', None)}")
            print(f"      R seed z/Vincent               = {res.get('vincent_seed_z_ratio', np.nan):.6f}")
            print(f"      R seed y/Vincent               = {res.get('vincent_seed_y_ratio', np.nan):.6f}")
            print(f"      R seed roundtrip error         = {res.get('vincent_seed_roundtrip_error', np.nan):.3e}")
            print(f"      Time                           = {res['total_time']:.2f}s")

            all_results.append({
                "z": z_name,
                "x_for_pi": x_name,
                "corr": actual_corr,
                "n": n,
                "N": N,

                "PpiVincent_z_eff_vs_SRS": res["PpiVincent_eff_z"],
                "PpiVincent_y_eff_vs_SRS": res["PpiVincent_eff_y"],
                "PpiInternal_z_eff_vs_SRS": res["PpiInternal_eff_z"],
                "PpiInternal_y_eff_vs_SRS": res["PpiInternal_eff_y"],

                "ABC_z": res["best_eff_z"],
                "ABC_y": res["best_eff_y"],
                "Random_z": res["random_best_eff_z"],
                "Random_y": res["random_best_eff_y"],
                "ABC_best_source": res.get("best_source"),
                "Random_best_source": getattr(random_search, "global_best_source", None),
                "use_R_reciprocal_seed": res.get("use_R_reciprocal_seed"),
                "vincent_seed_z_ratio": res.get("vincent_seed_z_ratio"),
                "vincent_seed_y_ratio": res.get("vincent_seed_y_ratio"),
                "vincent_seed_roundtrip_error": res.get("vincent_seed_roundtrip_error"),
                "vincent_seed_diag_error": res.get("vincent_seed_diag_error"),

                "ABC_z_over_PpiVincent_z": abc_z_vin,
                "ABC_y_over_PpiVincent_y": abc_y_vin,
                "Random_z_over_PpiVincent_z": rnd_z_vin,
                "Random_y_over_PpiVincent_y": rnd_y_vin,

                "ABC_z_over_PpiInternal_z": abc_z_int,
                "ABC_y_over_PpiInternal_y": abc_y_int,
                "Random_z_over_PpiInternal_z": rnd_z_int,
                "Random_y_over_PpiInternal_y": rnd_y_int,

                "same_pi_multiset_check": res["same_pi_multiset_check"],
                "vincent_ppi_diag_error": res["vincent_ppi_diag_error"],
                "internal_ppi_diag_error": res["internal_ppi_diag_error"],
                "ABC_valid_percent": abc_valid_pct,
                "Random_valid_percent": rnd_valid_pct,
                "time_seconds": res["total_time"],
                "completed_iterations": res.get("completed_iterations", SIM_PARAMS["max_iterations"]),
                "stopped_early": res.get("stopped_early", False),
                "stopped_by_time": res.get("stopped_by_time", False),
                "abc_eval_count": res["eval_count"],
                "random_eval_count": res["random_eval_count"],
            })

            abc_run_details[run_key] = {
                "abc": abc,
                "random_search": random_search,
                "result": res,
                "sort_idx": sort_idx,
                "z_name": z_name,
                "x_name": x_name,
                "n": n,
                "N": N,
            }

            # Save after every finished case, so results survive interruption.
            pd.DataFrame(all_results).to_csv(OUTPUT_CSV, index=False)
            print(f"    Partial results saved to: {OUTPUT_CSV}")

    print(f"\n\n{'=' * 80}")
    print("FINAL SUMMARY TABLE")
    print(f"{'=' * 80}\n")

    df_res = pd.DataFrame(all_results)

    summary_cols = [
        "z", "x_for_pi", "corr", "n", "N",
        "PpiVincent_z_eff_vs_SRS", "PpiInternal_z_eff_vs_SRS",
        "ABC_z_over_PpiVincent_z", "Random_z_over_PpiVincent_z",
        "ABC_z_over_PpiInternal_z", "Random_z_over_PpiInternal_z",
        "ABC_best_source", "Random_best_source",
        "vincent_seed_z_ratio", "vincent_seed_roundtrip_error",
        "ABC_valid_percent", "Random_valid_percent",
        "time_seconds", "completed_iterations", "same_pi_multiset_check",
    ]

    df_print = df_res[summary_cols].copy()
    num_cols = df_print.select_dtypes(include=[np.number]).columns
    df_print[num_cols] = df_print[num_cols].round(8)

    display(df_print)
    print(df_print.to_string(index=False))

    df_res.to_csv(OUTPUT_CSV, index=False)
    print(f"\nSaved: {OUTPUT_CSV}")
    print("Complete run objects are available in: abc_run_details")
else:
    print("RUN_MAIN=False: skipped main ABC run.")

ABC + RANDOM SEARCH ON CURRENT MU284 DATAFRAME
Corrected reference: Vincent z/pi ordered Pπ

High Corr: z = ME84; π from CONST
Corr(P85, ME84) = 0.986
N = 30 (current df used in this run)

  n = 5:
    ABC params: colony=30, iter=1000, limit=25, onlookers=1.0, progress_every=50, time_limit=None
    Random Search: start_evals=10, evals_per_iter=20
 ABC ALGORITHM - ME84_N30_n5
 Configuration:
   colony_size          = 30
   max_iterations       = 1000
   abandonment limit    = 25
   objective            = eff_z
   validation mode      = fast
   onlooker factor      = 1.0
   early stopping       = False, patience=50
   time limit           = None
   random search        = ON
   random start evals   = 10
   random evals/iter    = 20
   progress interval    = every 50 iterations
   local search interval= every 10 iterations
 Reference Pπ efficiency versus SRS:
   Pπ_z = 6.48
   Pπ_y = 5.45

 Initializing 30 food sources by HYBRID search [v15 no-silence]...
    first food source = exact Vinc

,z,x_for_pi,corr,n,N,PpiVincent_z_eff_vs_SRS,PpiInternal_z_eff_vs_SRS,ABC_z_over_PpiVincent_z,Random_z_over_PpiVincent_z,ABC_z_over_PpiInternal_z,Random_z_over_PpiInternal_z,ABC_best_source,Random_best_source,vincent_seed_z_ratio,vincent_seed_roundtrip_error,ABC_valid_percent,Random_valid_percent,time_seconds,completed_iterations,same_pi_multiset_check
0,ME84,CONST,0.985981,5,30,6.484103,6.484103,1.0,1.0,1.0,1.0,ABC_K_local_search,Random_K_local,NaN,NaN,83.598360,60.472563,196.677283,1000,True
1,S82,CONST,0.806532,5,30,4.941245,4.941244,1.0,1.0,1.0,1.0,ABC_K_local_search,Random_K_local,NaN,NaN,83.564304,59.673353,195.801739,1000,True
2,CS82,CONST,0.664956,5,30,11.053318,11.053318,1.0,1.0,1.0,1.0,ABC_K_local_search,Random_K_local,NaN,NaN,83.484674,60.188686,192.980926,1000,True


   z x_for_pi     corr  n  N  PpiVincent_z_eff_vs_SRS  PpiInternal_z_eff_vs_SRS  ABC_z_over_PpiVincent_z  Random_z_over_PpiVincent_z  ABC_z_over_PpiInternal_z  Random_z_over_PpiInternal_z    ABC_best_source Random_best_source  vincent_seed_z_ratio  vincent_seed_roundtrip_error  ABC_valid_percent  Random_valid_percent  time_seconds  completed_iterations  same_pi_multiset_check
ME84    CONST 0.985981  5 30                 6.484103                  6.484103                      1.0                         1.0                       1.0                          1.0 ABC_K_local_search     Random_K_local                   NaN                           NaN          83.598360             60.472563    196.677283                  1000                    True
 S82    CONST 0.806532  5 30                 4.941245                  4.941244                      1.0                         1.0                       1.0                          1.0 ABC_K_local_search     Random_K_local                 

In [64]:
# ============================================================
# VIEW NODE TRACE AFTER A RUN - v13 direct-K diagnostics
# ============================================================
# Use this after the main run. It shows the actual efficiencies of nodes
# created by local diagonal-preserving rotations around Vincent K.

if "abc_run_details" not in globals() or len(abc_run_details) == 0:
    print("No completed run in abc_run_details yet.")
else:
    last_key = list(abc_run_details.keys())[-1]
    abc_obj = abc_run_details[last_key]["abc"]
    trace = pd.DataFrame(getattr(abc_obj, "node_records", []))

    print(f"Showing node trace for: {last_key}")
    if trace.empty:
        print("No node trace was recorded. Set DETAILED_NODE_OUTPUT=True and rerun.")
    else:
        cols = [
            "eval_count", "source", "valid",
            "z_over_vincent", "y_over_vincent",
            "d_K_from_vincent", "accepted_rotations", "last_theta", "last_pair",
            "diag_error", "eff_z", "eff_y",
        ]
        cols = [c for c in cols if c in trace.columns]

        show = trace[cols].copy()
        for c in ["z_over_vincent", "y_over_vincent", "d_K_from_vincent", "diag_error", "eff_z", "eff_y", "last_theta"]:
            if c in show.columns:
                show[c] = pd.to_numeric(show[c], errors="coerce")

        print("\nFirst 30 evaluated nodes:")
        display(show.head(30))

        print("\nBest 30 nodes by z/Vincent:")
        display(show.sort_values("z_over_vincent", ascending=False).head(30))

        print("\nSummary by source:")
        summary = trace.groupby("source", dropna=False).agg(
            n=("source", "size"),
            valid_percent=("valid", lambda x: 100.0 * np.mean(x.astype(bool))),
            best_z_over_vincent=("z_over_vincent", "max"),
            mean_z_over_vincent=("z_over_vincent", "mean"),
            best_y_over_vincent=("y_over_vincent", "max"),
            mean_d_K=("d_K_from_vincent", "mean"),
        ).reset_index()
        display(summary.sort_values("best_z_over_vincent", ascending=False))

        out_trace = f"node_trace_{last_key}_v13.csv"
        trace.to_csv(out_trace, index=False)
        print(f"Saved node trace: {out_trace}")


Showing node trace for: CS82_N30_n5

First 30 evaluated nodes:


,eval_count,source,valid,z_over_vincent,y_over_vincent,d_K_from_vincent,accepted_rotations,last_theta,last_pair,diag_error,eff_z,eff_y
0,1,Vincent_K_seed,True,1.000000,1.000000,0.000000,0.0,0.000000,None,1.505731e-09,11.053318,1.429432
1,2,Vincent_K_local_init,True,0.998715,0.999771,0.001936,3.0,0.059014,"(2, 10)",1.505731e-09,11.039119,1.429105
2,3,Vincent_K_local_init,True,1.000000,1.000000,0.000979,3.0,-0.016393,"(6, 9)",1.505731e-09,11.053314,1.429432
3,4,Vincent_K_local_init,True,0.999693,0.999925,0.001325,3.0,0.028203,"(2, 11)",1.505731e-09,11.049922,1.429324
4,5,Vincent_K_local_init,True,0.999464,0.999874,0.001430,3.0,-0.038364,"(2, 7)",1.505731e-09,11.047393,1.429252
5,6,Vincent_K_local_init,True,0.999697,0.999934,0.000981,3.0,-0.011367,"(3, 11)",1.505731e-09,11.049974,1.429337
6,7,Vincent_K_local_init,True,1.000000,1.000000,0.000450,3.0,0.001592,"(6, 7)",1.505731e-09,11.053318,1.429432
7,8,Vincent_K_local_init,True,0.998310,0.999562,0.002348,3.0,-0.022020,"(0, 1)",1.505731e-09,11.034640,1.428806
8,9,Vincent_K_local_init,True,0.999997,1.000000,0.001420,3.0,0.063826,"(9, 10)",1.505731e-09,11.053284,1.429432
9,10,Vincent_K_local_init,True,0.999837,1.000000,0.000757,3.0,0.000358,"(4, 5)",1.505731e-09,11.051512,1.429431



Best 30 nodes by z/Vincent:


,eval_count,source,valid,z_over_vincent,y_over_vincent,d_K_from_vincent,accepted_rotations,last_theta,last_pair,diag_error,eff_z,eff_y
102064,102065,ABC_K_local_search,True,1.0,1.0,0.040253,5.0,0.039407,"(12, 15)",1.505731e-09,11.053318,1.429432
101448,101449,ABC_K_local_search,True,1.0,1.0,0.040176,5.0,-0.000113,"(8, 9)",1.505731e-09,11.053318,1.429432
101265,101266,ABC_K_local_search,True,1.0,1.0,0.040913,5.0,-0.041594,"(0, 5)",1.505731e-09,11.053318,1.429432
101445,101446,ABC_K_local_search,True,1.0,1.0,0.040815,5.0,-0.002437,"(1, 5)",1.505731e-09,11.053318,1.429432
101685,101686,ABC_K_local_candidate,True,1.0,1.0,0.040257,4.0,-0.020441,"(19, 23)",1.505731e-09,11.053318,1.429432
101601,101602,ABC_K_local_search,True,1.0,1.0,0.040037,5.0,-0.055095,"(7, 11)",1.505731e-09,11.053318,1.429432
102517,102518,ABC_K_local_search,True,1.0,1.0,0.039970,5.0,-0.019876,"(3, 4)",1.505731e-09,11.053318,1.429432
102292,102293,ABC_K_local_search,True,1.0,1.0,0.039813,5.0,0.040969,"(24, 26)",1.505731e-09,11.053318,1.429432
102582,102583,ABC_K_local_candidate,True,1.0,1.0,0.039972,4.0,-0.041115,"(28, 29)",1.505731e-09,11.053318,1.429432
100859,100860,ABC_K_local_search,True,1.0,1.0,0.040246,5.0,0.032066,"(7, 9)",1.505731e-09,11.053318,1.429432



Summary by source:


,source,n,valid_percent,best_z_over_vincent,mean_z_over_vincent,best_y_over_vincent,mean_d_K
3,ABC_K_local_search,19108,100.0,1.0,0.998365,1.005318,0.031038
2,ABC_K_local_candidate,52683,100.0,1.0,0.997720,1.015456,0.024295
5,Direct_K_random_local,13866,100.0,1.0,0.993000,1.031961,0.004979
10,Vincent_K_seed,1,100.0,1.0,1.000000,1.000000,0.000000
8,Vincent_K_local_fill,10,100.0,1.0,0.999890,1.000203,0.000904
9,Vincent_K_local_init,19,100.0,1.0,0.999631,1.000099,0.001203
0,ABC_Global_CaDsd_jump,7317,0.0,0.0,0.000000,0.000000,0.096207
1,ABC_Global_CaDsd_scout,6546,0.0,0.0,0.000000,0.000000,0.096200
4,Direct_Global_CaDsd_random,2955,0.0,0.0,0.000000,0.000000,0.096200
6,Global_CaDsd_fill,13,0.0,0.0,0.000000,0.000000,0.095470


Saved node trace: node_trace_CS82_N30_n5_v13.csv


# Sensitivity Analysis (moved to the end)


In [65]:
import numpy as np
import pandas as pd


def _abc_variance_from_omega_rho(abc: ABCAlgorithm, omega: np.ndarray, rho: np.ndarray):
    """Return kernel, variances, efficiencies, and validation details for a candidate."""
    try:
        K_dict = CaDsd(pi=abc.pik_sorted, M=abc.M, omega=omega, rho=rho)
        Kmat = K_dict["K"].astype(np.complex128)
    except Exception as e:
        return None, {
            "valid": False,
            "reason": f"CaDsd error: {str(e)[:80]}",
        }

    diag_K = np.real(np.diag(Kmat))
    max_diag_diff = float(np.max(np.abs(diag_K - abc.pik_sorted_desc)))

    if not np.allclose(diag_K, abc.pik_sorted_desc, atol=1e-3):
        return Kmat, {
            "valid": False,
            "reason": f"diagonal mismatch, max diff={max_diag_diff:.3e}",
        }

    try:
        evals = np.linalg.eigvalsh(Kmat)
    except Exception as e:
        return Kmat, {
            "valid": False,
            "reason": f"eigvalsh error: {str(e)[:80]}",
        }

    eig_min = float(evals.min())
    eig_max = float(evals.max())
    trace = float(evals.sum())

    if not (np.all(evals >= -1e-3) and np.all(evals <= 1 + 1e-3)):
        return Kmat, {
            "valid": False,
            "reason": f"eigenvalues outside [0,1], min={eig_min:.3e}, max={eig_max:.3e}",
            "eig_min": eig_min,
            "eig_max": eig_max,
            "trace": trace,
            "max_diag_diff": max_diag_diff,
        }

    if not np.isclose(trace, abc.n, atol=1e-3):
        return Kmat, {
            "valid": False,
            "reason": f"trace mismatch, trace={trace:.6f}, n={abc.n}",
            "eig_min": eig_min,
            "eig_max": eig_max,
            "trace": trace,
            "max_diag_diff": max_diag_diff,
        }

    A = Kmat * (abc.I_N - Kmat.conj())
    var_y = float(np.real(abc.y_sorted.conj().T @ abc.Dpi_inv @ A @ abc.Dpi_inv @ abc.y_sorted))
    var_z = float(np.real(abc.z_sorted.conj().T @ abc.Dpi_inv @ A @ abc.Dpi_inv @ abc.z_sorted))

    eff_y = abc.var_srs_y / var_y if var_y > 0 else np.nan
    eff_z = abc.var_srs_z / var_z if var_z > 0 else np.nan

    return Kmat, {
        "valid": True,
        "reason": "OK",
        "var_y": var_y,
        "var_z": var_z,
        "eff_y": eff_y,
        "eff_z": eff_z,
        "eig_min": eig_min,
        "eig_max": eig_max,
        "trace": trace,
        "max_diag_diff": max_diag_diff,
    }


def sensitivity_analysis_uniform(
    abc: ABCAlgorithm,
    result: dict,
    epsilons=(0.01, 0.02),
    case_name: Optional[str] = None,
    print_table: bool = True,
) -> pd.DataFrame:
    """
    Concise uniform sensitivity analysis around the best ABC design.

    It tests eight perturbation directions for each epsilon:
    omega/rho +/-, omega only +/-, and rho only +/-.
    """
    if result.get("best_omega") is None or result.get("best_rho") is None:
        raise ValueError("The result does not contain best_omega/best_rho. Run ABC first.")

    base_omega = result["best_omega"]
    base_rho = result["best_rho"]
    if case_name is None:
        case_name = result.get("case", abc.case_name)

    _, base_info = _abc_variance_from_omega_rho(abc, base_omega, base_rho)
    if not base_info["valid"]:
        raise ValueError(f"Base design is not valid: {base_info['reason']}")

    perturbation_cases = [
        ("omega+, rho+", +1, +1),
        ("omega+, rho-", +1, -1),
        ("omega-, rho+", -1, +1),
        ("omega-, rho-", -1, -1),
        ("omega+, rho=0", +1, 0),
        ("omega-, rho=0", -1, 0),
        ("omega=0, rho+", 0, +1),
        ("omega=0, rho-", 0, -1),
    ]

    rows = []
    print("\n" + "=" * 90)
    print(f"SENSITIVITY ANALYSIS: {case_name}")
    print("=" * 90)
    print(
        f"Base: var_y={base_info['var_y']:.6g}, var_z={base_info['var_z']:.6g}, "
        f"eff_y={base_info['eff_y']:.4f}, eff_z={base_info['eff_z']:.4f}"
    )

    for eps in epsilons:
        print(f"\nEpsilon = {eps}")
        print("-" * 90)
        print(f"{'case':<18} {'valid':>6} {'d_var_y%':>10} {'d_var_z%':>10} {'d_eff_z%':>10}  reason")
        print("-" * 90)

        for label, s_omega, s_rho in perturbation_cases:
            omega_new = np.clip(base_omega + s_omega * eps, 0.0, 1.0)
            rho_new = np.clip(base_rho + s_rho * eps, 0.0, 1.0)

            _, info = _abc_variance_from_omega_rho(abc, omega_new, rho_new)

            row = {
                "case": case_name,
                "epsilon": eps,
                "perturbation_type": label,
                "valid": info["valid"],
                "reason": info["reason"],
                "base_var_y": base_info["var_y"],
                "base_var_z": base_info["var_z"],
                "base_eff_y": base_info["eff_y"],
                "base_eff_z": base_info["eff_z"],
            }

            if info["valid"]:
                row.update({
                    "var_y": info["var_y"],
                    "var_z": info["var_z"],
                    "eff_y": info["eff_y"],
                    "eff_z": info["eff_z"],
                    "delta_var_y_percent": 100.0 * (info["var_y"] / base_info["var_y"] - 1.0),
                    "delta_var_z_percent": 100.0 * (info["var_z"] / base_info["var_z"] - 1.0),
                    "delta_eff_z_percent": 100.0 * (info["eff_z"] / base_info["eff_z"] - 1.0),
                    "eig_min": info["eig_min"],
                    "eig_max": info["eig_max"],
                    "trace": info["trace"],
                    "max_diag_diff": info["max_diag_diff"],
                })
                print(
                    f"{label:<18} {str(True):>6} "
                    f"{row['delta_var_y_percent']:>+10.3f} "
                    f"{row['delta_var_z_percent']:>+10.3f} "
                    f"{row['delta_eff_z_percent']:>+10.3f}  OK"
                )
            else:
                row.update({
                    "var_y": np.nan,
                    "var_z": np.nan,
                    "eff_y": np.nan,
                    "eff_z": np.nan,
                    "delta_var_y_percent": np.nan,
                    "delta_var_z_percent": np.nan,
                    "delta_eff_z_percent": np.nan,
                })
                print(f"{label:<18} {str(False):>6} {'nan':>10} {'nan':>10} {'nan':>10}  {info['reason']}")

            rows.append(row)

    df_sens = pd.DataFrame(rows)

    if print_table:
        valid_df = df_sens[df_sens["valid"] == True]
        print("\n" + "=" * 90)
        print("SENSITIVITY SUMMARY")
        print("=" * 90)
        print(f"Valid perturbations: {len(valid_df)}/{len(df_sens)}")
        if len(valid_df) > 0:
            print(
                f"Mean delta var_z: {valid_df['delta_var_z_percent'].mean():+.3f}% "
                f"(min {valid_df['delta_var_z_percent'].min():+.3f}%, "
                f"max {valid_df['delta_var_z_percent'].max():+.3f}%)"
            )
            print(
                f"Mean delta eff_z: {valid_df['delta_eff_z_percent'].mean():+.3f}% "
                f"(min {valid_df['delta_eff_z_percent'].min():+.3f}%, "
                f"max {valid_df['delta_eff_z_percent'].max():+.3f}%)"
            )

    return df_sens


print("Sensitivity functions loaded.")


Sensitivity functions loaded.


In [66]:
# Run sensitivity after all ABC runs.
# By default this analyzes only the last completed ABC run, so the output stays concise.
# To analyze another run, set SENSITIVITY_KEY to one key from abc_run_details.keys().

RUN_SENSITIVITY = True
SENSITIVITY_KEY = None      # None -> use the last run
SENSITIVITY_EPSILONS = (0.01, 0.02)

if RUN_SENSITIVITY:
    if "abc_run_details" not in globals() or len(abc_run_details) == 0:
        print("No ABC runs found yet. Run the ABC section first, then run this cell.")
    else:
        if SENSITIVITY_KEY is None:
            SENSITIVITY_KEY = list(abc_run_details.keys())[-1]

        print(f"Available run keys: {list(abc_run_details.keys())}")
        print(f"Running sensitivity for: {SENSITIVITY_KEY}")

        selected = abc_run_details[SENSITIVITY_KEY]
        df_sensitivity = sensitivity_analysis_uniform(
            abc=selected["abc"],
            result=selected["result"],
            epsilons=SENSITIVITY_EPSILONS,
            case_name=SENSITIVITY_KEY,
            print_table=True,
        )

        df_sensitivity.to_csv(f"sensitivity_{SENSITIVITY_KEY}.csv", index=False)
        print(f"\nSaved: sensitivity_{SENSITIVITY_KEY}.csv")
else:
    print("Sensitivity is skipped. Set RUN_SENSITIVITY = True when you want to run it.")


Available run keys: ['ME84_N30_n5', 'S82_N30_n5', 'CS82_N30_n5']
Running sensitivity for: CS82_N30_n5

SENSITIVITY ANALYSIS: CS82_N30_n5
Base: var_y=13058, var_z=119.009, eff_y=1.4294, eff_z=11.0525

Epsilon = 0.01
------------------------------------------------------------------------------------------
case                valid   d_var_y%   d_var_z%   d_eff_z%  reason
------------------------------------------------------------------------------------------
omega+, rho+         True     +0.012     +7.663     -7.118  OK
omega+, rho-         True     +0.012     +7.663     -7.118  OK
omega-, rho+         True     -0.000     -0.000     +0.000  OK
omega-, rho-         True     +0.000     +0.000     +0.000  OK
omega+, rho=0        True     +0.012     +7.663     -7.118  OK
omega-, rho=0        True     +0.000     +0.000     +0.000  OK
omega=0, rho+        True     -0.000     -0.000     +0.000  OK
omega=0, rho-        True     +0.000     +0.000     +0.000  OK

Epsilon = 0.02
----------------